# B6 · Explicabilité, équité et éthique — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/b6_explicabilite/05_solutions.ipynb)

Les réponses des exercices, exécutées. Les démarches détaillées (le *pourquoi*, les erreurs fréquentes, les variantes) sont dans `05_solutions.md`. Les cellules marquées `answer` enregistrent les réponses vérifiées par `wb.check` (`tools/build_answers.py`).

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | B6.Q1 à B6.Q12, B6.R1 à B6.R3, B6.1, B6.3 à B6.5 | vérifier tes réponses courtes | 🧠 🔁 ✏️ | ★ à ★★ | |
| A | B6.12 | Explications globales sur California : coefficients, permutation, dépendance partielle et ICE | 📦 | ★★ | 30 |
|  | B6.13 | Prédire la forme des courbes ICE du revenu médian et de la latitude | 🔮 | ★ | 10 |
|  | B6.14 | shapley_values : Shapley exact par énumération des coalitions | 🔨 | ★★★ | 45 |
|  | B6.15 | SHAP sur un gradient boosting California : TreeExplainer, waterfall, beeswarm, dépendance | 📦 | ★★ | 30 |
|  | B6.16 | Le fond change tout : trois fonds différents, puis une feature dupliquée | 🔬 | ★★★ | 40 |
|  | B6.17 | Penguins : SHAP par classe pour une régression logistique et une forêt aléatoire | 📦 | ★★ | 25 |
| B | B6.18 | integrated_gradients et grad_cam | 🔨 | ★★ | 30 |
|  | B6.19 | Saillance, gradient × entrée et gradients intégrés sur un CNN CIFAR-10 | 📦 | ★★ | 30 |
|  | B6.20 | Grad-CAM par hooks sur le ResNet fine-tuné du B1 : bonnes prédictions et erreurs | 📦 | ★★★ | 45 |
|  | B6.21 | Grad-CAM à l'envers : mauvaise couche, mauvaise classe, ReLU oubliée, carte mal alignée | 🐛 | ★★ | 25 |
|  | B6.22 | Test de santé d'Adebayo : randomiser les poids couche par couche, la carte change-t-elle ? | 🔬 | ★★★ | 40 |
| C | B6.23 | group_rates, demographic_parity_difference et equalized_odds_difference | 🔨 | ★★ | 25 |
|  | B6.24 | Audit d'équité sur Adult : indicateurs par sexe et par âge, intervalles de confiance bootstrap | 📦 | ★★★ | 45 |
|  | B6.25 | Lire un tableau d'audit : où est l'écart, est-il significatif, que conclure ? | 📈 | ★★ | 15 |
|  | B6.26 | Prédire : retirer la variable « sexe » rend-il le modèle équitable ? | 🔮 | ★ | 10 |
|  | B6.27 | Atténuer : retrait des variables de substitution, repondération, seuils par groupe, et ce que cela coûte | 🔬 | ★★★ | 50 |
|  | B6.28 | Fiche modèle avec sections explicabilité et équité : indicateurs par groupe, limites, usage prévu | 🛠️ | ★★ | 25 |
|  | B6.29 | Défi : écart d'equalized odds sous 0,07 sur Adult, en perdant moins de 2 points d'accuracy | 🏆 | ★★★★ | 90 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("ref")  # reference implementation

## Objectifs et rappel express

- Calculer à la main des valeurs de Shapley exactes, des gradients intégrés et une carte Grad-CAM, et les relier à ta librairie (`explain.py`) et à shap.
- Expliquer un modèle tabulaire globalement (permutation, dépendance partielle, ICE) et localement (Shapley, waterfall), en disant quel fond a servi.
- Produire des cartes d'attribution sur un CNN et les soumettre à un test de santé.
- Auditer un classifieur groupe par groupe, avec des intervalles, essayer des corrections, chiffrer leur coût et documenter le tout dans une fiche modèle.

**Rappel express.** Shapley : $v(S) = \frac{1}{B}\sum_k f(\mathbf{x}_S, \mathbf{b}^{(k)}_{\bar S})$, $\phi_i = \sum_{S \not\ni i} \frac{|S|!\,(M - |S| - 1)!}{M!}\,[v(S \cup \{i\}) - v(S)]$, $\sum_i \phi_i = f(\mathbf{x}) - v(\varnothing)$ ; modèle linéaire : $\phi_i = w_i(x_i - \bar b_i)$. Gradients intégrés : $\operatorname{IG}_i = (x_i - x'_i)\int_0^1 \partial_i f(\mathbf{x}' + \alpha(\mathbf{x} - \mathbf{x}'))\,d\alpha$, de somme $f(\mathbf{x}) - f(\mathbf{x}')$. Grad-CAM : $\alpha_c$ = moyenne spatiale de $\partial y^k / \partial A^c$, carte $\operatorname{ReLU}(\sum_c \alpha_c A^c)$. Équité : $\Delta_{\text{DP}}$ = écart des taux de sélection, $\Delta_{\text{EO}}$ = le plus grand des écarts de TPR et de FPR ; $\operatorname{FPR} = \frac{p}{1-p}\cdot\frac{1 - \operatorname{PPV}}{\operatorname{PPV}}\cdot(1 - \operatorname{FNR})$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ B6.1, B6.3, B6.4 et B6.5)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; un vecteur : une liste de nombres (`[0.5, -1.25]`). Un taux s'écrit comme une proportion (`0.25`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », les dérivations ∂ B6.2 et B6.6, l'estimation 🧮 B6.7, la réflexion (🗣️ B6.8, ⚖️ B6.9 et B6.10, 📄 B6.11) et l'entretien se corrigent avec `05_solutions.md`.

In [ ]:
# The short answers only use the numbers of their statements (02_exercices.md). Features are numbered 1..M in
# the formulas, columns 0..M-1 in the code (feature 1 is column 0).
import itertools
import math

import numpy as np


def coalition_values(f, x, background):
    """v(S) for every mask S (a tuple of 0/1): the mean of f over the background rows where S takes x's values."""
    x, background = np.asarray(x, float), np.asarray(background, float)
    values = {}
    for mask in itertools.product((0, 1), repeat=len(x)):
        rows = background.copy()
        keep = np.array(mask, bool)
        rows[:, keep] = x[keep]
        values[mask] = float(np.mean(f(rows)))
    return values


def shapley_from_values(values, M, weight=None):
    """phi_i = sum over S without i of weight(|S|) (v(S + i) - v(S)); weight defaults to |S|! (M-|S|-1)! / M!."""
    weight = weight or (lambda k: math.factorial(k) * math.factorial(M - k - 1) / math.factorial(M))
    phi = []
    for i in range(M):
        total = 0.0
        for mask, value in values.items():
            if not mask[i]:
                with_i = tuple(1 if j == i else mask[j] for j in range(M))
                total += weight(sum(mask)) * (values[with_i] - value)
        phi.append(total)
    return phi


# ✏️ B6.1: f(x) = 2 x1 + x1 x3 + x2, x = (2, 1, 3), background (0, 2, 0) and (2, 2, 2)
F_B61 = lambda X: 2 * X[:, 0] + X[:, 0] * X[:, 2] + X[:, 1]
X_B61, BG_B61 = np.array([2.0, 1.0, 3.0]), np.array([[0.0, 2.0, 0.0], [2.0, 2.0, 2.0]])
V_B61 = coalition_values(F_B61, X_B61, BG_B61)
V0_B61, V1_B61, V13_B61 = V_B61[(0, 0, 0)], V_B61[(1, 0, 0)], V_B61[(1, 0, 1)]
W3_B61 = math.factorial(1) * math.factorial(1) / math.factorial(3)        # |S| = 1, M = 3
PHI_B61 = shapley_from_values(V_B61, 3)
FIRST_B61 = [V_B61[tuple(int(j == i) for j in range(3))] - V0_B61 for i in range(3)]   # contribution to the empty set
LAST_B61 = [V_B61[(1, 1, 1)] - V_B61[tuple(int(j != i) for j in range(3))] for i in range(3)]
EVALS_B61 = 2 ** 3 * len(BG_B61)
MEAN_ROW_B61 = BG_B61.mean(axis=0, keepdims=True)                          # (1, 2, 1)
PHI_MEAN_B61 = shapley_from_values(coalition_values(F_B61, X_B61, MEAN_ROW_B61), 3)
INTER_SHARE_B61 = [PHI_B61[0] - 2 * (X_B61[0] - BG_B61[:, 0].mean()), PHI_B61[2]]   # the share of x1 x3 in phi_1, phi_3

# ✏️ B6.3: f(x) = x1**2 + 3 x2 at x = (2, -1), zero baseline unless stated otherwise
X_B63 = np.array([2.0, -1.0])
F_B63 = lambda P: P[..., 0] ** 2 + 3 * P[..., 1]
GRAD_B63 = lambda P: np.stack([2 * P[..., 0], np.full(np.shape(P)[:-1], 3.0)], axis=-1)


def ig_rule_b63(n, method, x=X_B63, ref=(0.0, 0.0)):
    """Integrated gradients with n intervals (the conventions of the docstring of integrated_gradients)."""
    x, ref = np.asarray(x, float), np.asarray(ref, float)
    k = np.arange(n + 1) / n
    alphas = {"left": k[:-1], "right": k[1:], "midpoint": (np.arange(n) + 0.5) / n, "trapezoid": k}[method]
    weights = np.full(len(alphas), 1 / n)
    if method == "trapezoid":
        weights[[0, -1]] /= 2
    grads = GRAD_B63(ref + alphas[:, None] * (x - ref))
    return (x - ref) * (weights[:, None] * grads).sum(axis=0)


GRAD_AT_X_B63 = [float(g) for g in GRAD_B63(X_B63)]
GXI_B63 = [float(g) for g in GRAD_B63(X_B63) * X_B63]
GAP_B63 = float(F_B63(X_B63) - F_B63(np.zeros(2)))
IG_EXACT_B63 = [float(X_B63[0] ** 2), float(3 * X_B63[1])]                 # x1 * integral of 2 a x1 = x1**2 ; 3 x2
IG1_LEFT_B63, IG1_RIGHT_B63, IG1_TRAP_B63 = (float(ig_rule_b63(5, m)[0]) for m in ("left", "right", "trapezoid"))
N_LEFT_B63 = math.ceil(round(X_B63[0] ** 2 / 0.03, 9))                     # the left error of IG_1 is x1**2 / n
REF2_B63 = (1.0, 0.0)
IG1_REF2_B63 = float(X_B63[0] ** 2 - REF2_B63[0] ** 2)                     # (x1 - x1') (x1 + x1')
SUM_REF2_B63 = float(F_B63(X_B63) - F_B63(np.array(REF2_B63)))
CAPTUM_B63 = IG1_TRAP_B63 * 5 / 6                                          # 6 points, weights 1/6 halved at both ends

# ✏️ B6.4: two 3 x 3 channels (index 0 = channel 1)
A_B64 = np.array([[[1, 0, 0], [2, 1, 0], [3, 2, 1]], [[0, 1, 2], [0, 0, 1], [0, 0, 0]]], dtype=float)
G_B64 = np.array([[[0, 1, 2], [1, 1, 1], [2, 1, 0]], [[-1, -1, -1], [0, 0, 0], [-1, -0.5, 0]]], dtype=float)
ALPHA_B64 = G_B64.mean(axis=(1, 2))
RAW_B64 = np.tensordot(ALPHA_B64, A_B64, axes=1)                           # before the ReLU
CAM_B64 = np.maximum(RAW_B64, 0)
NORM_B64 = CAM_B64 / np.abs(CAM_B64).max()
TOPRIGHT_B64 = float(RAW_B64[0, 2])
ZEROS_B64 = int((CAM_B64 == 0).sum())
CENTER_B64 = float(NORM_B64[1, 1])
NORELU_TOPRIGHT_B64 = float(RAW_B64[0, 2] / np.abs(RAW_B64).max())
OTHER_B64 = np.tensordot(np.array([-0.5, 1.0]), A_B64, axes=1)             # class k': alpha' = (-0.5, 1)
CORNERS_B64 = {(0, 0): "A", (2, 0): "B", (0, 2): "C"}
MAX_B64 = CORNERS_B64[tuple(int(i) for i in np.unravel_index(np.argmax(CAM_B64), CAM_B64.shape))]
MAX_OTHER_B64 = CORNERS_B64[tuple(int(i) for i in np.unravel_index(np.argmax(OTHER_B64), OTHER_B64.shape))]
SHIFT_B64 = 72 / 3 / 2                                                     # half a layer pixel, in image pixels

# ✏️ B6.5: two groups (TP, FN, FP, TN)
COUNTS_B65 = {"F": (72, 48, 19, 361), "H": (256, 64, 40, 280)}


def rates_b65(tp, fn, fp, tn):
    n = tp + fn + fp + tn
    return {"base": (tp + fn) / n, "sel": (tp + fp) / n, "tpr": tp / (tp + fn), "fpr": fp / (fp + tn),
            "ppv": tp / (tp + fp), "acc": (tp + tn) / n}


R_B65 = {g: rates_b65(*c) for g, c in COUNTS_B65.items()}
DP_B65 = abs(R_B65["H"]["sel"] - R_B65["F"]["sel"])
TPR_GAP_B65 = abs(R_B65["H"]["tpr"] - R_B65["F"]["tpr"])
FPR_GAP_B65 = abs(R_B65["H"]["fpr"] - R_B65["F"]["fpr"])
EO_B65 = max(TPR_GAP_B65, FPR_GAP_B65)
RATIO_B65 = R_B65["F"]["sel"] / R_B65["H"]["sel"]
ACC_GAP_B65 = abs(R_B65["F"]["acc"] - R_B65["H"]["acc"])
POS_F_B65 = COUNTS_B65["F"][0] + COUNTS_B65["F"][1]
EXTRA_B65 = round(R_B65["H"]["tpr"] * POS_F_B65 - COUNTS_B65["F"][0])

# quizzes and recalls with a computation
Q1E = 2 ** 4
Q2E = 0.83 - float(np.mean([0.66, 0.62, 0.65, 0.63, 0.64]))
Q3C = 3 * 2 + float(np.mean(np.array([0, 1, 2]) ** 2))
Q4A = 0.72 - 0.40 - 0.15 - (-0.05)
Q4C = 0.3 + (-0.1)
Q4E = 2 ** (5 - 1)
Q5C, Q5D = 2 * (4 - 3), 2 * (4 - 5)
Q6A, Q6B = 2 * (1.5 - 0.5), 0 * (1.5 - 0.5)
Q7A = 0                                                     # min(x, 1) is flat beyond 1
Q7B = 3 * (1 / 3)                                           # (x - 0) * integral of f'(3 a): f' = 1 for a < 1/3
Q7C = 0                                                     # the path from 2 to 3 stays where f is flat
Q8A, Q8C = 160 // 32, 224 // 4
Q9E = 1 / 10
Q11A = 0.6 - 0.25                                           # a perfect classifier selects exactly the positives
Q11D = 0.5 / (1 - 0.5) * (1 - 0.8) / 0.8 * (1 - 0.2)        # Chouldechova
EPS_R1, X_R1, G_R1 = 0.03, np.array([0.40, 0.98, 0.10]), np.array([-0.7, 0.2, 0.0])
R1A = [float(v) for v in np.clip(X_R1 + EPS_R1 * np.sign(G_R1), 0, 1)]
R1D = 0.01 * 3072 * 0.01
R2E = round((0.88 - 0.86) * 400)
CM_R3 = np.array([[88, 12], [10, 40]])                      # rows: truth 0, 1 ; columns: prediction 0, 1
(TN_R3, FP_R3), (FN_R3, TP_R3) = CM_R3
R3A, R3B = TP_R3 / (TP_R3 + FN_R3), FP_R3 / (FP_R3 + TN_R3)
R3C, R3D = TP_R3 / (TP_R3 + FP_R3), (TP_R3 + FP_R3) / CM_R3.sum()

# The same answers from the libraries (no download): mylearn (reference), shap, scikit-learn and torchvision
E = mylearn.explain

# ✏️ B6.1: shapley_values, then shap's exact explainer with every background row
phi, base = E.shapley_values(F_B61, X_B61, BG_B61)
assert np.allclose(phi, PHI_B61) and math.isclose(base, V0_B61)
assert math.isclose(phi.sum() + base, float(F_B61(X_B61[None])[0]))       # efficiency
assert np.allclose(E.shapley_values(F_B61, X_B61, MEAN_ROW_B61)[0], PHI_MEAN_B61)
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")   # tqdm, imported by shap
try:
    import shap
except ImportError:
    print("ℹ️ shap is not installed: the comparison with shap.explainers.Exact is skipped")
else:
    exact = shap.explainers.Exact(F_B61, shap.maskers.Independent(BG_B61, max_samples=len(BG_B61)))(X_B61[None])
    assert np.allclose(exact.values[0], PHI_B61) and math.isclose(float(exact.base_values[0]), V0_B61)

# quiz Q4 to Q6: the linear models of the statements (phi_i = w_i (x_i - mean_i))
lin = lambda w: (lambda X: X @ np.asarray(w, float))
assert np.allclose(E.shapley_values(lin([2, -3]), [4.0, 1.0], [[3.0, 1.0]])[0], [Q5C, -3 * 0])
assert math.isclose(E.shapley_values(lin([2, -3]), [4.0, 1.0], [[4.0, 1.0], [6.0, 3.0]])[0][0], Q5D)
phi_a = E.shapley_values(lin([2, 2]), [1.5, 1.5], [[0.0, 0.0], [1.0, 1.0]])[0]
phi_b = E.shapley_values(lin([4, 0]), [1.5, 1.5], [[0.0, 0.0], [1.0, 1.0]])[0]
assert math.isclose(phi_a[0], Q6A) and math.isclose(phi_b[1], Q6B) and math.isclose(phi_a.sum(), phi_b.sum())
# ∂ B6.2, question 5 (an application, not checked by wb.check)
phi_d2, base_d2 = E.shapley_values(lambda X: 1 + X @ np.array([0.5, -2.0, 0.0]), [4.0, 1.0, 7.0],
                                   [[2.0, 0.0, 1.0], [4.0, 1.0, 5.0], [0.0, 2.0, 3.0]])
assert np.allclose(phi_d2, [1.0, 0.0, 0.0]) and math.isclose(base_d2, 0.0, abs_tol=1e-12)

# ✏️ B6.3: integrated_gradients with the rules of its docstring
ig = lambda **kw: E.integrated_gradients(GRAD_B63, X_B63, **kw)
for method, answer in (("left", IG1_LEFT_B63), ("right", IG1_RIGHT_B63), ("trapezoid", IG1_TRAP_B63)):
    assert math.isclose(ig(n_steps=5, method=method)[0], answer)
assert np.allclose(ig(n_steps=5, method="trapezoid"), IG_EXACT_B63)       # exact: the gradient is linear on the path
assert abs(GAP_B63 - ig(n_steps=N_LEFT_B63, method="left").sum()) <= 0.03 - 1e-4      # far from the edge (4/134)
assert abs(GAP_B63 - ig(n_steps=N_LEFT_B63 - 1, method="left").sum()) > 0.03 + 1e-5
assert np.allclose(ig(baseline=list(REF2_B63), n_steps=3), [IG1_REF2_B63, IG_EXACT_B63[1]])
assert math.isclose(ig(baseline=list(REF2_B63), n_steps=3).sum(), SUM_REF2_B63, abs_tol=1e-12)
alphas = np.linspace(0, 1, 6)                                             # captum's riemann_trapezoid, 6 points
weights = np.full(6, 1 / 6)
weights[[0, -1]] /= 2
assert math.isclose(X_B63[0] * float(weights @ GRAD_B63(alphas[:, None] * X_B63)[:, 0]), CAPTUM_B63)

# quiz Q7: min(x, 1) at x = 3 (its gradient: 1 below 1, 0 beyond)
grad_min = lambda P: (P < 1).astype(float)
assert math.isclose(float(E.integrated_gradients(grad_min, [3.0], n_steps=3000, method="midpoint")[0]), Q7B,
                    rel_tol=1e-3)
assert float(E.integrated_gradients(grad_min, [3.0], baseline=[2.0], n_steps=10)[0]) == Q7C

# ✏️ B6.4: grad_cam, with and without the ReLU
assert np.allclose(E.grad_cam(A_B64[None], G_B64[None])[0], NORM_B64)
assert math.isclose(float(E.grad_cam(A_B64[None], G_B64[None], relu=False)[0, 0, 2]), NORELU_TOPRIGHT_B64)
assert np.allclose(E.grad_cam(10 * A_B64[None], G_B64[None])[0], NORM_B64)
assert np.allclose(E.grad_cam(A_B64[None], 10 * G_B64[None])[0], NORM_B64)
assert np.allclose(E.grad_cam(A_B64[None], G_B64[None], relu=False, normalize=False)[0], RAW_B64)

# ✏️ B6.5: group_rates and the two gaps, from the labels rebuilt out of the counts
def labels_from_counts(counts):
    y_true, y_pred, groups = [], [], []
    for group, (tp, fn, fp, tn) in counts.items():
        for t, p, k in ((1, 1, tp), (1, 0, fn), (0, 1, fp), (0, 0, tn)):
            y_true += [t] * k
            y_pred += [p] * k
            groups += [group] * k
    return np.array(y_true), np.array(y_pred), np.array(groups)


y_true, y_pred, groups = labels_from_counts(COUNTS_B65)
rates = E.group_rates(y_true, y_pred, groups)
for g in ("F", "H"):
    for ours, theirs in (("base", "base_rate"), ("sel", "selection_rate"), ("tpr", "tpr"), ("fpr", "fpr"),
                         ("ppv", "precision"), ("acc", "accuracy")):
        assert math.isclose(rates[g][theirs], R_B65[g][ours])
assert math.isclose(E.demographic_parity_difference(y_pred, groups), DP_B65)
assert math.isclose(E.equalized_odds_difference(y_true, y_pred, groups), EO_B65)
from sklearn.metrics import confusion_matrix as sk_confusion_matrix

for g, (tp, fn, fp, tn) in COUNTS_B65.items():
    assert (sk_confusion_matrix(y_true[groups == g], y_pred[groups == g]) == [[tn, fp], [fn, tp]]).all()
for p_, ppv, fnr, fpr in ((R_B65[g]["base"], R_B65[g]["ppv"], 1 - R_B65[g]["tpr"], R_B65[g]["fpr"]) for g in "FH"):
    assert math.isclose(p_ / (1 - p_) * (1 - ppv) / ppv * (1 - fnr), fpr)    # ∂ B6.6, question 5

# recall R3: the confusion matrix of ch. 3 (the reference metrics.py) and scikit-learn
y3 = np.repeat([0, 0, 1, 1], [TN_R3, FP_R3, FN_R3, TP_R3])
p3 = np.repeat([0, 1, 0, 1], [TN_R3, FP_R3, FN_R3, TP_R3])
assert (np.asarray(mylearn.metrics.confusion_matrix(y3, p3)) == CM_R3).all()
assert (sk_confusion_matrix(y3, p3) == CM_R3).all()
r3 = E.group_rates(y3, p3, np.zeros(len(y3), int))[0]
assert math.isclose(r3["tpr"], R3A) and math.isclose(r3["fpr"], R3B) and math.isclose(r3["precision"], R3C)
assert math.isclose(r3["selection_rate"], R3D)

# quiz Q8: the sizes of the maps of an untrained ResNet-18 (no download)
try:
    import torch
    from torchvision.models import resnet18
except ImportError:
    print("ℹ️ torchvision is not installed: the sizes of quiz Q8 are not checked")
else:
    net, sizes = resnet18(weights=None).eval(), {}
    hooks = [getattr(net, name).register_forward_hook(lambda m, i, o, name=name: sizes.__setitem__(name, o.shape[-1]))
             for name in ("layer1", "layer4")]
    with torch.no_grad():
        net(torch.zeros(1, 3, 160, 160))
        assert sizes["layer4"] == Q8A
        net(torch.zeros(1, 3, 224, 224))
        assert sizes["layer1"] == Q8C
    for hook in hooks:
        hook.remove()
print("✅ the paper answers agree with mylearn (reference: explain, metrics), shap, scikit-learn and torchvision")

**B6.Q1 — Interprétable par construction ou expliqué après coup ?**

In [ ]:
wb.record("B6.Q1a", "B", mistakes={"relis « Interprétable par construction, ou expliqué après coup » (§206.1) : la forêt se lit-elle elle-même, ou faut-il TreeSHAP ?": "A", "une carte de saillance est-elle le modèle, ou une explication ajoutée ? Relis 206.1": "C"})
wb.record("B6.Q1b", False, mistakes={"relis la définition de l'explication post hoc (§206.1) : est-elle la règle du modèle ?": True})
wb.record("B6.Q1c", "A", mistakes={"Grad-CAM se contente-t-il des prédictions ? Relis la phrase sur les méthodes spécifiques (§206.1)": "B", "Grad-CAM change-t-il le modèle ? Relis « Interprétable par construction, ou expliqué après coup »": "C"})
wb.record("B6.Q1d", True, mistakes={"relis 206.1 : de quoi l'importance par permutation a-t-elle besoin ?": False})
wb.record("B6.Q1e", Q1E, mistakes={"c'est la profondeur : on demande le nombre de feuilles": 4,
          "c'est le nombre de nœuds internes : on demande les feuilles": 15,
          "c'est le nombre de feuilles d'un arbre de profondeur 3": 8,
          "c'est le nombre total de nœuds : on demande les feuilles": 31})

**B6.Q2 — Explication globale ou locale : quel outil répond à quelle question ?**

In [ ]:
wb.record("B6.Q2a", "C", mistakes={"la question porte-t-elle sur tout le modèle ou sur un dossier ? Relis « Global ou local » (§206.1)": "A", "la question compare-t-elle des groupes ? Relis le tableau de « Global ou local »": "B"})
wb.record("B6.Q2b", "A", mistakes={"un waterfall explique combien de prédictions ? Relis le tableau de « Global ou local »": "B", "une courbe ICE d'un seul client dit-elle ce que le modèle utilise en tout ? Relis 206.2": "C"})
wb.record("B6.Q2c", False, mistakes={"relis la fin de « Global ou local » (§206.1) : une importance moyenne dit-elle tout d'un cas particulier ?": True})
wb.record("B6.Q2d", "B", mistakes={"une moyenne sur 1 000 clients explique-t-elle encore un client ? Relis la puce sur le beeswarm (§206.4)": "A", "la moyenne des |φ_i| mélange-t-elle une colonne ? Relis la puce sur le beeswarm (§206.4)": "C"})
wb.record("B6.Q2e", Q2E, decimals=2, mistakes={"c'est le score après mélange : l'importance est une baisse": 0.64,
          "fais la moyenne des cinq mélanges avant de soustraire": 0.83 - 0.66,
          "vérifie le sens de la soustraction": -Q2E})

**B6.Q3 — Dépendance partielle et courbes ICE : ce que montre la moyenne, ce qu'elle cache**

In [ ]:
wb.record("B6.Q3a", "C", mistakes={"fais la moyenne de v · x₂ sur tous les exemples : que vaut la moyenne de x₂ ?": "A", "relis la formule de PD_j(v) : on moyenne v · x₂ sur tous les exemples ; que vaut la moyenne de x₂ ?": "B"})
wb.record("B6.Q3b", 2, mistakes={"chaque exemple a sa courbe ICE : quelles pentes prennent-elles selon x₂ ?": 1,
          "une courbe ICE de ce modèle est une droite : combien de valeurs de x₂ donnent-elles des pentes différentes ?": 4})
wb.record("B6.Q3c", Q3C, decimals=2, mistakes={"tu as mis la moyenne de x₂ au carré : la dépendance partielle fait la moyenne des prédictions": 6 + 1.0,
          "relis la question : on impose v = 2 à x₁ ; n'oublie pas le terme x₂²": 6.0,
          "fais la moyenne de x₂² sur les trois valeurs, pas leur somme": 6 + 5.0})
wb.record("B6.Q3d", True, mistakes={"relis le paragraphe qui suit le code de 206.2 : que fait centered=True à chaque courbe ?": False})
wb.record("B6.Q3e", "A", mistakes={"relis la formule de PD_j(v) : sur quels exemples fait-on la moyenne ?": "B", "relis le piège classique « une dépendance partielle hors des données » (§206.2)": "C"})

**B6.Q4 — Les quatre propriétés de Shapley : efficacité, symétrie, joueur nul, additivité**

In [ ]:
wb.record("B6.Q4a", Q4A, decimals=2, mistakes={"l'efficacité répartit f(x) moins la valeur de base, pas f(x)": 0.72 - 0.15 + 0.05,
          "vérifie le signe de φ₂ : il est négatif": 0.72 - 0.40 - 0.15 - 0.05,
          "c'est l'écart à répartir entre les trois features : retire les deux valeurs connues": 0.72 - 0.40})
wb.record("B6.Q4b", "A", mistakes={"l'efficacité porte sur la somme des parts : relis les quatre propriétés (§206.3)": "B", "la symétrie compare deux joueurs : relis les quatre propriétés (§206.3)": "C"})
wb.record("B6.Q4c", Q4C, decimals=1, mistakes={"relis l'additivité : les parts s'ajoutent, elles ne se moyennent pas": 0.1,
          "vérifie le signe de la part de h": 0.4})
wb.record("B6.Q4d", False, mistakes={"la symétrie exige que les deux joueurs apportent la même chose à toute coalition : avec x₁ = 1 et x₂ = 5, est-ce le cas ? Essaie f = x₁ + x₂": True})
wb.record("B6.Q4e", Q4E, mistakes={"c'est le nombre de toutes les coalitions : retire celles qui contiennent i": 32,
          "c'est le nombre d'ordres d'arrivée des cinq features": 120,
          "relis l'encadré 🧮 de 206.3 : un ensemble de m éléments a 2^m sous-ensembles": 5})

**B6.Q5 — Changer le fond (background) change les valeurs SHAP : vrai ou faux**

In [ ]:
wb.record("B6.Q5a", False, mistakes={"relis l'efficacité avec un fond (§206.3) : de quoi dépend la somme des φ_i ?": True})
wb.record("B6.Q5b", "C", mistakes={"relis la définition de la valeur de base (§206.3) : sur quelles lignes la moyenne est-elle prise ?": "A", "relis la définition de la valeur de base (§206.3) : v(∅) est-il nul ?": "B"})
wb.record("B6.Q5c", Q5C, mistakes={"vérifie le sens (x₁ moins la moyenne du fond) et le fond utilisé : le fond A": -Q5C,
          "n'oublie pas le coefficient w₁ = 2": 1})
wb.record("B6.Q5d", Q5D, mistakes={"vérifie le sens (x₁ moins la moyenne du fond) et le fond utilisé : le fond B": -Q5D,
          "n'oublie pas le coefficient w₁ = 2": -1})
wb.record("B6.Q5e", False, mistakes={"relis les deux modes de TreeExplainer (§206.4) : répondent-ils à la même question ?": True})
wb.record("B6.Q5f", 100, mistakes={"relis le paragraphe qui suit le code de shap (§206.3) : que fait max_samples par défaut ?": 300})

**B6.Q6 — Features corrélées : ce que SHAP répartit et ce qu'il ne dit pas**

In [ ]:
wb.record("B6.Q6a", round(Q6A), mistakes={"relis le cas linéaire : φ₁ = w₁ (x₁ − moyenne), avec w₁ = 2": 4,
          "n'oublie pas de retirer la moyenne du fond": 3})
wb.record("B6.Q6b", round(Q6B), mistakes={"le modèle B lit-il x₂ ? Relis le cas linéaire": 2,
          "le modèle B lit-il x₂ ? Relis « Features corrélées : ce que SHAP répartit »": 4})
wb.record("B6.Q6c", True, mistakes={"calcule φ₁ + φ₂ pour les deux modèles, avec la formule du cas linéaire": False})
wb.record("B6.Q6d", "C", mistakes={"relis « Features corrélées : ce que SHAP répartit » : le Shapley observationnel est-il fidèle au modèle ou aux données ?": "A", "relis « Features corrélées : ce que SHAP répartit » : pour qui le Shapley observationnel a-t-il été conçu ?": "B"})
wb.record("B6.Q6e", "A", mistakes={"la plus grande des deux dépend du modèle choisi : relis la fin de « Features corrélées »": "B", "la répartition entre deux copies est-elle stable ? Relis la fin de « Features corrélées »": "C"})

**B6.Q7 — Gradient, gradient × entrée, gradients intégrés : pourquoi une ligne de référence ?**

In [ ]:
wb.record("B6.Q7a", Q7A, mistakes={"au-delà de 1, f vaut toujours 1 : quelle est sa pente ?": 1,
          "relis la question : on demande la dérivée, pas la valeur de f": 3})
wb.record("B6.Q7b", round(Q7B), mistakes={"le gradient est nul en x, mais pas tout le long du chemin depuis 0 : relis la formule des gradients intégrés": 0,
          "c'est x − x′ : multiplie par la moyenne de la pente le long du chemin": 3})
wb.record("B6.Q7c", Q7C, mistakes={"le chemin va de 2 à 3 : quelle est la pente de f sur ce segment ?": 1,
          "relis la question : la référence a changé": 3})
wb.record("B6.Q7d", True, mistakes={"pour f = wᵀx, le gradient est le même en tout point : calcule les deux attributions": False})
wb.record("B6.Q7e", "A", mistakes={"un pixel égal à sa référence a-t-il un x_i − x′_i non nul ? Relis la formule des gradients intégrés": "B", "relis la formule des gradients intégrés : que vaut le facteur x_i − x′_i pour un pixel du fond ?": "C"})

**B6.Q8 — Grad-CAM : pourquoi la dernière couche convolutive ?**

In [ ]:
wb.record("B6.Q8a", Q8A, mistakes={"c'est la taille de layer3, qui divise par 16 : relis les quatre détails de « Avec PyTorch : deux hooks »": 160 // 16,
          "c'est une division par 8 : de combien layer4 divise-t-il la taille de l'image ?": 160 // 8,
          "c'est la taille de layer1 : relis la question": 160 // 4})
wb.record("B6.Q8b", "B", mistakes={"relis « Pourquoi la dernière couche convolutive ? » (§206.6) : que vient-il après elle ?": "A", "une softmax rend un vecteur de probabilités, sans position : relis « Pourquoi la dernière couche convolutive ? »": "C"})
wb.record("B6.Q8c", Q8C, mistakes={"relis l'énoncé : layer1 divise par 4": 224 // 8,
          "c'est la taille de layer4 à 224 pixels : relis la question": 7,
          "relis l'énoncé : layer1 divise par 4, pas par 2": 112})
wb.record("B6.Q8d", True, mistakes={"un poids α_c peut-il être négatif ? Relis l'exemple à la main de 206.6": False})
wb.record("B6.Q8e", "C", mistakes={"relis la phrase qui suit la formule de Grad-CAM (§206.6) : de quel score prend-on le gradient ?": "A", "relis la phrase qui suit la formule de Grad-CAM (§206.6) : avant ou après la softmax ?": "B"})

**B6.Q9 — Une carte de saillance qui ne dépend pas du modèle : que prouve le test de randomisation ?**

In [ ]:
wb.record("B6.Q9a", "B", mistakes={"relis « Les tests de santé » (§206.7) : qu'exige-t-on d'une explication quand le modèle change ?": "A", "le test de randomisation mesure-t-il la robustesse aux attaques ? Relis 206.7": "C"})
wb.record("B6.Q9b", False, mistakes={"relis le paragraphe « Ce que prouve un test de santé réussi » (§206.7)": True})
wb.record("B6.Q9c", "C", mistakes={"c'est la randomisation indépendante : relis les deux variantes (§206.7)": "A", "c'est la randomisation des données : relis les deux tests (§206.7)": "B"})
wb.record("B6.Q9d", False, mistakes={"une corrélation de rangs compare l'ordre des pixels : deux cartes peuvent-elles avoir le même ordre et des valeurs différentes ?": True})
wb.record("B6.Q9e", Q9E, decimals=1, mistakes={"relis l'énoncé : CIFAR-10 a dix classes, et les labels du test ne sont pas mélangés": 0.5,
          "un réseau qui répond au hasard a-t-il une accuracy nulle ?": 0.0,
          "c'est l'accuracy d'entraînement d'un réseau qui apprend par cœur : relis la question, on demande le test": 1.0})

**B6.Q10 — Sources de biais : données, labels, variables de substitution, boucles de rétroaction**

In [ ]:
wb.record("B6.Q10a", "A", mistakes={"les décisions du modèle fabriquent-elles ses données ? Relis les puces de « D'où vient un biais » (§206.8)": "B", "les transcriptions sont-elles fausses ? Relis les puces de « D'où vient un biais » (§206.8)": "C"})
wb.record("B6.Q10b", "B", mistakes={"la note des managers est-elle une feature ou la cible ? Relis 206.8": "A", "le modèle sert-il une autre population que celle de son évaluation ? Relis 206.8": "C"})
wb.record("B6.Q10c", "C", mistakes={"qui choisit les dossiers contrôlés ? Relis les puces de « D'où vient un biais »": "A", "une colonne porte-t-elle un attribut sensible à sa place ? Relis la question : qui choisit les contrôles ?": "B"})
wb.record("B6.Q10d", True, mistakes={"une colonne qui n'est pas le sexe peut-elle en porter une part de l'information ? Relis la puce sur les variables de substitution (§206.8)": False})
wb.record("B6.Q10e", "A", mistakes={"le modèle sert-il une autre population ? Relis les puces de 206.8 : qu'est-ce qui est mal mesuré ?": "B", "les décisions du modèle changent-elles le capteur ? Relis les puces de 206.8": "C"})

**B6.Q11 — Parité démographique, égalité des chances, calibration : peut-on tout avoir ?**

In [ ]:
wb.record("B6.Q11a", Q11A, decimals=2, mistakes={"un classifieur parfait sélectionne-t-il tout le monde ? Son taux de sélection vaut le taux de base": 0.0,
          "relis la définition de Δ_DP : la différence des taux de sélection, pas leur rapport": 0.6 / 0.25})
wb.record("B6.Q11b", True, mistakes={"calcule le TPR et le FPR d'un classifieur parfait dans chaque groupe": False})
wb.record("B6.Q11c", True, mistakes={"calcule la precision d'un classifieur parfait dans chaque groupe": False})
wb.record("B6.Q11d", Q11D, decimals=2, mistakes={"relis la relation de Chouldechova : le dernier facteur est 1 − FNR": 0.5 / 0.5 * 0.25 * 0.2,
          "relis la relation : (1 − PPV) / PPV, pas PPV / (1 − PPV)": 0.5 / 0.5 * 4 * 0.8,
          "relis la relation : le premier facteur est p / (1 − p)": 0.5 * 0.25 * 0.8})
wb.record("B6.Q11e", "A", mistakes={"la parité démographique tient-elle compte de qui rembourserait ? Relis « Les critères » (§206.9)": "B", "un faux positif est ici un prêt accordé à qui ne rembourserait pas : est-ce l'erreur la plus coûteuse pour les demandeurs ?": "C"})

**B6.Q12 — AI Act : dans quelle catégorie de risque tombe un scoring de crédit ?**

In [ ]:
wb.record("B6.Q12a", "B", mistakes={"relis la puce « haut risque » de « L'AI Act » (§206.11) : que vise le point 5 b) ?": "A", "le scoring de crédit figure-t-il parmi les pratiques interdites ? Relis « L'AI Act » (§206.11)": "C"})
wb.record("B6.Q12b", "A", mistakes={"relis le point 5 b) cité en 206.11 : quelle exception contient-il ?": "B", "la détection de fraude figure-t-elle à l'article 5 ? Relis « L'AI Act »": "C"})
wb.record("B6.Q12c", "C", mistakes={"relis la puce « risque inacceptable » de « L'AI Act » (§206.11)": "A", "l'article 50 parle de transparence envers l'utilisateur : relis la puce « risque inacceptable »": "B"})
wb.record("B6.Q12d", False, mistakes={"relis l'encadré 🕰️ sur la régulation (§206.11) : quelles dates le règlement 2026/1744 a-t-il fixées ?": True})
wb.record("B6.Q12e", "B", mistakes={"relis le dernier paragraphe de « L'AI Act » : qui, du fournisseur ou du déployeur, réalise l'analyse d'impact ?": "A", "relis le dernier paragraphe de « L'AI Act » : l'article 27 s'adresse à qui ?": "C"})
wb.record("B6.Q12f", False, mistakes={"relis le texte du point 5 b) cité en 206.11 : de quelles personnes évalue-t-il la solvabilité ?": True})

**B6.R1 — Visualiser ce qu'a appris un CNN, et l'attaque FGSM**

In [ ]:
wb.record("B6.R1a", R1A, decimals=2, mistakes={"ramène chaque pixel dans [0, 1] après le pas": [0.37, 1.01, 0.1],
          "vérifie le signe : FGSM monte la loss, x + ε sign(∇L)": [0.43, 0.95, 0.1],
          "FGSM ajoute ε fois le signe du gradient, pas ε fois le gradient": [0.38, 0.99, 0.1],
          "le signe de 0 vaut 0 : le troisième pixel ne bouge pas": [0.37, 1.0, 0.13]})
wb.record("B6.R1b", True, mistakes={"relis le début de 206.5 et la section FGSM du ch. 21 : les deux calculs dérivent-ils par rapport à la même chose, avec la même passe arrière ?": False})
wb.record("B6.R1c", "B", mistakes={"relis le rappel maths « le gradient par rapport à l'entrée » du ch. 21 : les poids bougent-ils ?": "A", "la maximisation d'activation a-t-elle besoin de labels ? Relis le ch. 21": "C"})
wb.record("B6.R1d", R1D, decimals=2, mistakes={"relis la formule ε Σ|w_j| du ch. 21 : il manque un facteur": 0.01 * 0.01,
          "c'est ε fois le nombre de pixels : n'oublie pas |w_j|": 0.01 * 3072,
          "relis l'énoncé : ε = 0,01 et |w_j| = 0,01": 3.072})
wb.record("B6.R1e", False, mistakes={"relis le rappel maths du ch. 21 : la montée de gradient cherche-t-elle un minimum ?": True})

**B6.R2 — feature_importances_ contre importance par permutation**

In [ ]:
wb.record("B6.R2a", "A", mistakes={"relis l'importance des features au ch. 14 : qu'arrive-t-il aux features à forte cardinalité ?": "B", "une baisse d'impureté peut-elle être négative ? Relis le ch. 14": "C"})
wb.record("B6.R2b", True, mistakes={"relis le ch. 14 : sur quelles données l'importance MDI est-elle calculée ?": False})
wb.record("B6.R2c", 1, mistakes={"relis le ch. 14 : les importances MDI sont normalisées": 100})
wb.record("B6.R2d", "B", mistakes={"mélanger un identifiant fait-il perdre quelque chose sur des exemples jamais vus ? Relis le ch. 14": "A", "une importance par permutation fortement négative voudrait dire que le mélange aide beaucoup : est-ce plausible ?": "C"})
wb.record("B6.R2e", R2E, mistakes={"c'est la baisse d'accuracy : multiplie-la par le nombre d'exemples": 0,
          "vérifie l'ordre de grandeur : 2 points de 400 exemples": 80})

**B6.R3 — TPR, FPR et precision à partir d'une matrice de confusion**

In [ ]:
wb.record("B6.R3a", R3A, decimals=2, mistakes={"le TPR divise TP par les positifs réels (la ligne 1), pas par les sélectionnés": R3C,
          "le TPR divise par les positifs réels, pas par tous les exemples": 40 / 150})
wb.record("B6.R3b", R3B, decimals=2, mistakes={"c'est la part de faux positifs parmi les sélectionnés : le FPR divise par les négatifs réels": 12 / 52,
          "relis la disposition : la case [0, 1] contient quoi ?": 10 / 50,
          "le FPR divise par les négatifs réels, pas par tous les exemples": 12 / 150})
wb.record("B6.R3c", R3C, decimals=3, mistakes={"la precision divise TP par les sélectionnés (la colonne 1), pas par les positifs réels": R3A,
          "c'est la part de faux positifs parmi les sélectionnés : on demande la part de vrais positifs": 12 / 52})
wb.record("B6.R3d", R3D, decimals=3, mistakes={"c'est la part de positifs réels (la ligne 1) : on demande la part de prédictions positives": 50 / 150,
          "le taux de sélection divise par tous les exemples, pas par les négatifs réels": 52 / 100})
wb.record("B6.R3e", "A", mistakes={"relis la disposition : la ligne 0 est la vérité 0, la colonne 1 la prédiction 1": "B", "relis la disposition : la ligne 0 est la vérité 0": "C"})

**Ex B6.1 — Valeurs de Shapley exactes à la main : trois features, huit coalitions**

In [ ]:
wb.record("B6.1a", round(V0_B61), mistakes={"tu as calculé f sur la ligne moyenne du fond : v(∅) est la moyenne de f sur les deux lignes": 5,
          "relis la question : v(∅) n'a aucune feature du client": 11,
          "tu as additionné les deux lignes : v(S) est une moyenne": 12})
wb.record("B6.1b", round(V1_B61), mistakes={"dans v({1}), seule la feature 1 vient du client : as-tu inversé le masque ?": round(V_B61[(0, 1, 1)]),
          "tu as additionné les deux lignes : v(S) est une moyenne": 16,
          "relis la question : v({1}) garde x₂ et x₃ du fond": 11})
wb.record("B6.1c", round(V13_B61), mistakes={"dans v({1, 3}), les features 1 et 3 viennent du client : as-tu inversé le masque ?": round(V_B61[(0, 1, 0)]),
          "tu as additionné les deux lignes : v(S) est une moyenne": 24})
wb.record("B6.1d", W3_B61, decimals=2, mistakes={"c'est le poids des coalitions de taille 0 ou 2 : {3} a un membre": 1 / 3,
          "les poids ne sont pas uniformes sur les quatre coalitions sans la feature 1 : relis la formule": 1 / 4,
          "les poids ne sont pas uniformes sur les huit coalitions : relis la formule": 1 / 8})
wb.record("B6.1e", PHI_B61[0], decimals=1, mistakes={"tu n'as gardé que l'apport de la feature 1 à la coalition vide (ou le seul terme 2x₁) : il y a quatre coalitions sans elle": FIRST_B61[0],
          "tu n'as gardé que son apport quand elle arrive la dernière : il y a quatre coalitions sans elle": LAST_B61[0]})
wb.record("B6.1f", round(PHI_B61[1]), mistakes={"vérifie le signe : x₂ est plus petit que ses valeurs du fond": 1,
          "x₂ entre dans f : un joueur nul n'apporte jamais rien, est-ce le cas ?": 0})
wb.record("B6.1g", PHI_B61[2], decimals=1, mistakes={"tu n'as gardé que l'apport de la feature 3 à la coalition vide": FIRST_B61[2],
          "tu n'as gardé que son apport quand elle arrive la dernière": LAST_B61[2],
          "as-tu pris pour valeur de base f sur la ligne moyenne du fond ? Relis a)": PHI_B61[2] + 1})
wb.record("B6.1h", False, mistakes={"calcule ce que l'interaction apporte à φ₁ (retire le terme 2(x₁ − moyenne)) et compare à φ₃": True})
wb.record("B6.1i", EVALS_B61, mistakes={"chaque v(S) demande une évaluation par ligne du fond : il y en a deux": 8,
          "chaque v(S) se calcule une fois et sert aux trois features": 3 * EVALS_B61,
          "compte toutes les coalitions, la vide et la pleine comprises": 6 * 2,
          "la coalition pleine se calcule aussi sur les deux lignes du fond, comme les autres": EVALS_B61 - 1})
wb.record("B6.1j", PHI_MEAN_B61[0], decimals=1, mistakes={"relis la question : le fond n'a plus qu'une ligne, recalcule les v(S)": 3.5,
          "tu n'as gardé que le terme 2x₁ : la feature 1 entre aussi dans l'interaction": 2.0})
wb.record("B6.1k", "B", mistakes={"relis l'encadré 📐 de la fiche : la feature 1 est la colonne 0": "A", "relis la signature de shapley_values : que rend-elle à part phi ?": "C"})

**Ex B6.3 — Gradients intégrés à la main sur f(x) = x₁² + 3x₂ et vérification de la complétude**

In [ ]:
wb.record("B6.3a", GRAD_AT_X_B63, decimals=1, mistakes={"la dérivée de x₁² est 2x₁": [2.0, 3.0],
          "la dérivée de 3x₂ par rapport à x₂ ne dépend pas de x₂": [4.0, -3.0]})
wb.record("B6.3b", GXI_B63, decimals=1, mistakes={"multiplie chaque dérivée par la valeur de sa feature": GRAD_AT_X_B63,
          "vérifie le signe : x₂ = −1": [8.0, 3.0]})
wb.record("B6.3c", round(GAP_B63), mistakes={"c'est la somme de gradient × entrée : on demande l'écart de prédiction f(x) − f(x′)": 5,
          "vérifie le signe de 3x₂ : x₂ = −1": 7,
          "vérifie le sens de la soustraction : f(x) − f(x′)": -1})
wb.record("B6.3d", IG_EXACT_B63, decimals=1, mistakes={"c'est gradient × entrée : intègre la dérivée le long du chemin": GXI_B63,
          "multiplie l'intégrale par x_i − x′_i": [2.0, 3.0],
          "vérifie le signe de la seconde attribution : x₂ = −1": [4.0, 3.0]})
wb.record("B6.3e", IG1_LEFT_B63, decimals=1, mistakes={"c'est la règle 'right' : 'left' prend les points k/n pour k = 0 à n − 1": IG1_RIGHT_B63,
          "multiplie la moyenne des gradients par x₁ − x′₁": IG1_LEFT_B63 / 2,
          "n = 5 intervalles : les points 0 ; 0,2 ; 0,4 ; 0,6 ; 0,8": float(ig_rule_b63(4, "left")[0]),
          "relis la question : la règle de gauche, avec n = 5": 4.0})
wb.record("B6.3f", IG1_RIGHT_B63, decimals=1, mistakes={"c'est la règle 'left' : 'right' prend les points k/n pour k = 1 à n": IG1_LEFT_B63,
          "multiplie la moyenne des gradients par x₁ − x′₁": IG1_RIGHT_B63 / 2,
          "relis la question : la règle de droite, avec n = 5": 4.0})
wb.record("B6.3g", IG1_TRAP_B63, decimals=1, mistakes={"relis la question : la règle des trapèzes, dont les deux extrémités comptent pour moitié": 4.8,
          "multiplie la somme pondérée par x₁ − x′₁": 2.0,
          "les poids valent 1/n, divisés par deux aux extrémités : relis l'énoncé": 2 * (0 + 0.8 + 1.6 + 2.4 + 3.2 + 2) / 6})
wb.record("B6.3h", N_LEFT_B63, mistakes={"avec ce n, l'erreur dépasse encore un peu 0,03 : prends le plus petit entier qui convient": N_LEFT_B63 - 1,
          "l'erreur de la règle de gauche sur IG₁ dépend de x₁² : recalcule-la": 34,
          "recalcule l'erreur de la règle de gauche en fonction de n : où intervient x₁ ?": 67},
          fractional="le nombre d'intervalles est entier : prends le plus petit n qui convient")
wb.record("B6.3i", round(IG1_REF2_B63), mistakes={"relis la question : la référence a changé ; un gradient pris en x seul ne suffit pas non plus": 4,
          "le chemin part de x′ : son point est x′ + α(x − x′)": 2})
wb.record("B6.3j", round(SUM_REF2_B63), mistakes={"relis la question : avec cette référence, la somme vaut f(x) − f(x′)": 1,
          "relis la question : on demande la somme des deux attributions": 3,
          "c'est la seconde attribution seule : ajoute IG₁": -3})
wb.record("B6.3k", CAPTUM_B63, decimals=2, mistakes={"c'est le trapèze de ta docstring : relis l'encadré 🕰️ de 206.6 sur captum": 4.0,
          "le facteur est n/(n + 1), pas (n + 1)/n : relis l'encadré 🕰️ de 206.6": 4.8,
          "6 points font 5 intervalles, pas 6 : relis l'énoncé": 4 * 6 / 7})
wb.record("B6.3l", "C", mistakes={"n_steps compte les intervalles dans ta librairie, pas les points": "A", "relis la question : e) utilise la règle 'left'": "B"})

**Ex B6.4 — Grad-CAM à la main : deux cartes 3×3 et leurs gradients**

In [ ]:
wb.record("B6.4a", float(ALPHA_B64[0]), decimals=1, mistakes={"α_c est la moyenne des gradients, pas leur somme": float(G_B64[0].sum()),
          "α_c est la moyenne des gradients seuls, sans les multiplier par les activations": float((G_B64[0] * A_B64[0]).mean())})
wb.record("B6.4b", float(ALPHA_B64[1]), decimals=1, mistakes={"α_c est la moyenne des gradients, pas leur somme": float(G_B64[1].sum()),
          "vérifie le signe : les gradients du canal 2 sont négatifs ou nuls": -float(ALPHA_B64[1]),
          "α_c est la moyenne des gradients seuls, sans les multiplier par les activations": float((G_B64[1] * A_B64[1]).mean())})
wb.record("B6.4c", TOPRIGHT_B64, decimals=1, mistakes={"relis la question : avant le ReLU": 0.0,
          "vérifie le signe de α₂": -TOPRIGHT_B64,
          "relis la case : en haut à droite, A¹ vaut 0 et A² vaut 2": -0.5})
wb.record("B6.4d", ZEROS_B64, mistakes={"compte aussi les cases dont la somme pondérée est négative mais proche de 0": 2,
          "une case positive n'est pas mise à 0 : compte les cases négatives de la somme pondérée": 4})
wb.record("B6.4e", CENTER_B64, decimals=2, mistakes={"n'oublie pas de diviser par la plus grande valeur de la carte": 1.0,
          "on divise par la plus grande valeur absolue, pas par la somme de la carte": float(CAM_B64[1, 1] / CAM_B64.sum())})
wb.record("B6.4f", "B", mistakes={"cherche la plus grande valeur de α₁A¹ + α₂A² après le ReLU": "A", "en haut à droite, quel est le signe de la somme pondérée ?": "C"})
wb.record("B6.4g", NORELU_TOPRIGHT_B64, decimals=2, mistakes={"sans le ReLU, la case garde son signe": -NORELU_TOPRIGHT_B64,
          "n'oublie pas de diviser par la plus grande valeur absolue": TOPRIGHT_B64,
          "relis la question : sans le ReLU, la case n'est pas mise à 0": 0.0})
wb.record("B6.4h", "C", mistakes={"calcule −0,5 A¹ + A² case par case : que vaut-elle en haut à gauche ?": "A", "calcule −0,5 A¹ + A² case par case : que vaut-elle en bas à gauche ?": "B"})
wb.record("B6.4i", False, mistakes={"multiplier les activations par 10 multiplie toute la somme pondérée : que fait la division par le maximum ?": True})
wb.record("B6.4j", round(SHIFT_B64), mistakes={"c'est la taille d'un pixel de la couche : on demande un demi-pixel": 24,
          "relis l'énoncé : la carte fait 3 × 3 et l'image 72 × 72": 36})

**Ex B6.5 — Audit à la main : deux groupes, deux matrices de confusion, cinq indicateurs**

In [ ]:
wb.record("B6.5a", R_B65["F"]["base"], decimals=2, mistakes={"relis la question : on demande la part de positifs réels (TP + FN)": R_B65["F"]["sel"],
          "le taux de base divise par tout le groupe, pas par les négatifs": 120 / 380})
wb.record("B6.5b", R_B65["F"]["sel"], decimals=3, mistakes={"le taux de sélection compte tous les sélectionnés, TP et FP": 72 / 500,
          "relis la question : on demande la part sélectionnée (TP + FP)": R_B65["F"]["base"]})
wb.record("B6.5c", R_B65["F"]["tpr"], decimals=2, mistakes={"c'est le FNR : le TPR divise TP par les positifs réels": 0.4,
          "c'est la precision : le TPR divise par les positifs réels, pas par les sélectionnés": R_B65["F"]["ppv"],
          "le TPR divise par les positifs réels, pas par tout le groupe": 72 / 500})
wb.record("B6.5d", R_B65["H"]["fpr"], decimals=3, mistakes={"le FPR divise par les négatifs réels, pas par tout le groupe": 40 / 640,
          "c'est la part de vrais négatifs : le FPR compte les faux positifs": 280 / 320,
          "le FPR divise par les négatifs réels (FP + TN), pas par les sélectionnés": 40 / 296})
wb.record("B6.5e", R_B65["F"]["ppv"], decimals=3, mistakes={"la precision divise TP par les sélectionnés (TP + FP)": R_B65["F"]["tpr"],
          "c'est la part de faux positifs parmi les sélectionnés : on demande la precision": 19 / 91})
wb.record("B6.5f", DP_B65, decimals=2, mistakes={"c'est l'écart des taux de base : la parité démographique compare les taux de sélection": 0.5 - 0.24,
          "relis la question : la parité démographique compare les taux de sélection": TPR_GAP_B65,
          "les taux de sélection divisent TP + FP par l'effectif du groupe": 296 / 640 - 72 / 500})
wb.record("B6.5g", EO_B65, decimals=2, mistakes={"l'equalized odds prend le plus grand des deux écarts, TPR et FPR": FPR_GAP_B65,
          "l'equalized odds prend le plus grand des deux écarts, pas leur somme": TPR_GAP_B65 + FPR_GAP_B65,
          "l'equalized odds prend le plus grand des deux écarts, pas leur moyenne": (TPR_GAP_B65 + FPR_GAP_B65) / 2})
wb.record("B6.5h", RATIO_B65, decimals=2, mistakes={"relis la question : le taux des femmes divisé par celui des hommes": 1 / RATIO_B65,
          "relis la question : on demande le rapport, pas son complément": 1 - RATIO_B65})
wb.record("B6.5i", False, mistakes={"compare ton rapport h) à 4/5 = 0,8": True})
wb.record("B6.5j", True, mistakes={"calcule l'accuracy (TP + TN) / n de chaque groupe": False})
wb.record("B6.5k", EXTRA_B65, mistakes={"c'est le nombre total de qualifiées sélectionnées : on demande combien de plus": 96,
          "le TPR des hommes s'applique aux 120 candidates qualifiées, pas à toutes": round(0.8 * 500) - 72,
          "relis l'énoncé : le TPR à atteindre est celui des hommes": 120 - 72})

## Partie A · Modèles tabulaires : explications globales, valeurs de Shapley et SHAP

Tu expliques d'abord un gradient boosting de California **globalement** : coefficients d'une régression, importance par permutation, dépendance partielle et courbes ICE (B6.12), dont tu prédis la forme pour deux features (B6.13). Puis **localement** : tu écris le Shapley exact de ta librairie (B6.14), tu le compares à TreeSHAP sur un gradient boosting et tu lis les trois graphiques de shap (B6.15), tu mesures ce que changent le fond et une feature dupliquée (B6.16), et tu expliques un classifieur à trois classes de Penguins (B6.17). Au ch. 14, tu as entraîné des gradient boostings sur California et comparé `feature_importances_` à la permutation ; au ch. 15, tu as construit des pipelines : ici, tu ouvres la boîte.

La cellule suivante charge les outils de la partie : `wbx` (`wb.explain_data`, l'infrastructure du chapitre), le découpage figé de California (`X_TRAIN`, `Y_TRAIN` : 12 384 districts ; `X_VAL`, `Y_VAL` : 4 128 ; les mêmes à chaque exécution), `FEATURES` (les 8 noms de colonnes), `GBR`, le gradient boosting de référence (`HistGradientBoostingRegressor(random_state=0)` ajusté sur `X_TRAIN`), dont se servent les vérifications et les expériences données, `TEST_EXPLAIN` (le fichier des tests de `explain.py`) et `shap`. Les features de California y sont arrondies en float32 (B6.15 dit pourquoi).

In [ ]:
# Tools of part A: California and Penguins (frozen splits), the reference model of the checks, your library explain.py
import math
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestClassifier
from sklearn.inspection import partial_dependence, permutation_importance
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from wb import explain_data as wbx
from wb.nbtools import (StopCheck, as_numpy, checked, filled, fr, pair, refuses, returned, run_pytest_group, same,
                        verdict, written)

warnings.filterwarnings("ignore", message=".*IProgress not found.*")              # tqdm, imported by shap
warnings.filterwarnings("ignore", message="X does not have valid feature names")   # a model fitted on a DataFrame,
                                                                                   # called on an array (shapley_values)
try:
    import shap
except ImportError:                       # shap is on Colab and in requirements.txt; wb.ensure("shap") installs it
    shap = None
    print("⏳ shap n'est pas installé : les cellules qui s'en servent le diront (wb.ensure('shap') l'installe).")

TEST_EXPLAIN = "tests/test_chb6_explain.py"            # the tests of your library explain.py
CAL = wbx.california()                                 # the frozen split of California (features rounded to float32)
FEATURES = CAL.features
X_TRAIN, Y_TRAIN, X_VAL, Y_VAL = CAL.X_train, CAL.y_train, CAL.X_val, CAL.y_val
GBR = HistGradientBoostingRegressor(random_state=0).fit(X_TRAIN, Y_TRAIN)   # the reference gradient boosting


def pct(x, digits=0):
    """A share written the French way: 0.734 -> '73 %'."""
    return f"{100 * x:.{digits}f} %".replace(".", ",")


def shapley_ready(ex_id):
    """True when your shapley_values (Ex B6.14) runs on a tiny case where f only accepts 2-D arrays (one feature,
    a background of one row); otherwise one ⏳ or ❌ line that sends you back to B6.14."""
    tiny = lambda: mylearn.explain.shapley_values(lambda X: np.asarray(X, dtype=float)[:, 0], [1.0], [[0.0]])
    if not written(tiny):
        print(f"⏳ Ex {ex_id} : cette vérification attend ta shapley_values (Ex B6.14).")
        return False
    try:
        tiny()
    except Exception as exc:  # noqa: BLE001 - the learner's code
        print(f"❌ Ex {ex_id} : ta shapley_values (Ex B6.14) lève {type(exc).__name__} sur un cas minuscule (une "
              "feature, un fond d'une ligne, une f qui n'accepte que des tableaux 2-D) : fais d'abord passer ses tests.")
        return False
    return True


print(f"California : {len(X_TRAIN)} districts d'entraînement, {len(X_VAL)} de validation, {len(CAL.X_test)} de test, "
      f"{len(FEATURES)} features ({', '.join(FEATURES)}) ; cible MedHouseVal (centaines de milliers de dollars). "
      f"Gradient boosting de référence : R² de validation {fr(GBR.score(X_VAL, Y_VAL))}.")

### Ex B6.12 — Explications globales sur California : coefficients, permutation, dépendance partielle et ICE 📦 ★★ ⏱️ 30 min
**Objectif :** comparer trois explications globales d'un même problème et savoir ce que chacune mesure.  
**Prérequis :** ch. 14 et 15 · fiche §206.2 (« Coefficients et arbres », « L'importance par permutation », « La dépendance partielle », « Les courbes ICE ») · 🔁 R2 · **Fil rouge :** California · **Parcours :** R, C

**Contrat** des cinq fonctions de la cellule suivante :
- `linear_12(X, y)` rend un `Pipeline` `StandardScaler` → `LinearRegression`, ajusté sur `(X, y)` ;
- `coefs_12(pipeline)` rend une `pd.Series` des 8 coefficients de la `LinearRegression` du pipeline, indexée par les features, dans l'ordre de `FEATURES` : l'effet d'**un écart-type** de chaque feature, les autres fixées ;
- `boosting_12(X, y)` rend un `HistGradientBoostingRegressor(random_state=0)`, tous les autres réglages par défaut, ajusté sur `(X, y)` ;
- `permutation_12(model, X, y)` rend une `pd.Series` : la baisse moyenne du R² de `permutation_importance(model, X, y, n_repeats=5, random_state=0)`, indexée par les colonnes de `X`, triée de la plus grande à la plus petite ;
- `pd_ice_12(model, X, feature)` rend le dictionnaire `{"grid": (50,), "average": (50,), "individual": (len(X), 50)}` de `partial_dependence(model, X, [feature], kind="both", grid_resolution=50)` (la grille va du 5ᵉ au 95ᵉ centile de la feature).

La cellule de vérification appelle `linear_12` et `boosting_12` sur `X_TRAIN`, `permutation_12` sur le gradient boosting de référence `GBR` et la **validation**, `pd_ice_12` sur `GBR`, 300 districts de validation et `AveOccup`, et compare chaque résultat à scikit-learn. Une fois les trois résultats justes, elle trace les coefficients en valeur absolue, les importances, puis les courbes ICE de `AveOccup` et de `HouseAge` avec leur moyenne. Rien à taper.

Dans tes notes : range les features par |coefficient| et par importance : lesquelles changent de rang, et pourquoi une régression linéaire et un gradient boosting peuvent-ils ne pas s'appuyer sur les mêmes features ? Pourquoi la permutation se calcule-t-elle sur la validation, et que perdrait-on à la calculer sur l'entraînement (🔁 R2) ? Sur les courbes ICE de `AveOccup`, où la dépendance partielle résume-t-elle bien les courbes, et où les cache-t-elle ? Que deviennent ces courbes au-delà de la grille, là où il y a peu de districts (piège « une dépendance partielle hors des données ») ?

In [ ]:
def linear_12(X, y):
    """A Pipeline StandardScaler -> LinearRegression, fitted on (X, y), returned."""
    return make_pipeline(StandardScaler(), LinearRegression()).fit(X, y)


def coefs_12(pipeline):
    """pd.Series of the 8 coefficients of the LinearRegression of the pipeline, indexed by the features."""
    return pd.Series(pipeline[-1].coef_, index=pipeline.feature_names_in_)


def boosting_12(X, y):
    """HistGradientBoostingRegressor(random_state=0), every other setting by default, fitted on (X, y), returned."""
    return HistGradientBoostingRegressor(random_state=0).fit(X, y)


def permutation_12(model, X, y):
    """pd.Series: the mean drop of R² of permutation_importance(model, X, y, n_repeats=5, random_state=0), indexed by
    the columns of X, sorted from the largest to the smallest."""
    result = permutation_importance(model, X, y, n_repeats=5, random_state=0)
    return pd.Series(result.importances_mean, index=X.columns).sort_values(ascending=False)


def pd_ice_12(model, X, feature):
    """dict {"grid": (50,), "average": (50,), "individual": (len(X), 50)} of
    partial_dependence(model, X, [feature], kind="both", grid_resolution=50)."""
    result = partial_dependence(model, X, [feature], kind="both", grid_resolution=50)
    return {"grid": result["grid_values"][0], "average": result["average"][0], "individual": result["individual"][0]}


ROWS_12 = X_VAL.iloc[:300]                     # the districts of the curves of the check
REF_LIN_12 = make_pipeline(StandardScaler(), LinearRegression()).fit(X_TRAIN, Y_TRAIN)
ok_12 = {}
with checked("B6.12"):
    lin_12 = linear_12(X_TRAIN.copy(), Y_TRAIN.copy())
    if returned("B6.12", "linear_12", lin_12):
        steps_12 = [step for _, step in getattr(lin_12, "steps", [])]
        verdict("B6.12", isinstance(lin_12, Pipeline) and steps_12 and isinstance(steps_12[-1], LinearRegression)
                and any(isinstance(s, StandardScaler) for s in steps_12),
                "linear_12 rend un Pipeline StandardScaler → LinearRegression.",
                f"linear_12 doit rendre un Pipeline (StandardScaler, puis LinearRegression), reçu {type(lin_12).__name__} "
                f"{[type(s).__name__ for s in steps_12]}.")
        scaler_12 = next((s for s in steps_12 if isinstance(s, StandardScaler)), None)
        if scaler_12 is not None and not (scaler_12.with_mean and scaler_12.with_std):
            print("❌ Ex B6.12 : ton StandardScaler ne centre pas ou ne réduit pas (with_mean="
                  f"{scaler_12.with_mean}, with_std={scaler_12.with_std}) : le contrat veut ses réglages par défaut, "
                  "pour des coefficients qui mesurent l'effet d'un écart-type.")
        elif scaler_12 is not None and getattr(scaler_12, "mean_", None) is not None:
            verdict("B6.12", np.allclose(scaler_12.mean_, X_TRAIN.mean().to_numpy()),
                    "le StandardScaler a appris les moyennes de X_TRAIN.",
                    "le StandardScaler n'a pas appris les moyennes de X_TRAIN : sur quelles lignes est-il ajusté ?")
        c_12 = coefs_12(lin_12)
        want_c_12 = pd.Series(REF_LIN_12[-1].coef_, index=FEATURES)
        raw_12 = LinearRegression().fit(X_TRAIN, Y_TRAIN).coef_
        if not isinstance(c_12, pd.Series) or list(c_12.index) != FEATURES:
            print("❌ Ex B6.12 : coefs_12 doit rendre une pd.Series indexée par les 8 features, dans l'ordre de FEATURES "
                  f"(reçu {type(c_12).__name__}).")
        elif np.allclose(c_12.to_numpy(), raw_12):
            print("❌ Ex B6.12 : ce sont les coefficients des features brutes : chacun dépend de l'unité de sa feature ; "
                  "le contrat les veut sur des features standardisées.")
        else:
            ok_12["coefs"] = same("B6.12", "coefficients sur features standardisées", c_12.to_numpy(),
                                  want_c_12.to_numpy(), rtol=1e-6, atol=1e-9)

with checked("B6.12"):
    gb_12 = boosting_12(X_TRAIN.copy(), Y_TRAIN.copy())
    if returned("B6.12", "boosting_12", gb_12):
        if not isinstance(gb_12, HistGradientBoostingRegressor):
            print(f"❌ Ex B6.12 : boosting_12 doit rendre un HistGradientBoostingRegressor, reçu {type(gb_12).__name__}.")
            raise StopCheck
        want_p_12 = HistGradientBoostingRegressor(random_state=0).get_params()
        diff_12 = {k: v for k, v in gb_12.get_params().items() if want_p_12.get(k) != v}
        verdict("B6.12", not diff_12, "réglages : random_state=0, le reste par défaut.",
                f"réglages différents du contrat : {diff_12}.")
        same("B6.12", "prédictions de boosting_12 sur 200 districts de validation", gb_12.predict(X_VAL.iloc[:200]),
             GBR.predict(X_VAL.iloc[:200]), rtol=1e-9, atol=1e-9)

with checked("B6.12"):
    imp_12 = permutation_12(GBR, X_VAL.copy(), Y_VAL.copy())      # the reference model: a slip above does not spread
    want_i_12 = pd.Series(permutation_importance(GBR, X_VAL, Y_VAL, n_repeats=5, random_state=0).importances_mean,
                          index=FEATURES).sort_values(ascending=False)
    if not isinstance(imp_12, pd.Series) or sorted(imp_12.index) != sorted(FEATURES):
        print("❌ Ex B6.12 : permutation_12 doit rendre une pd.Series indexée par les 8 features "
              f"(reçu {type(imp_12).__name__}).")
        raise StopCheck
    verdict("B6.12", list(imp_12.index) == list(want_i_12.index) or bool((np.diff(imp_12.to_numpy()) <= 0).all()),
            "importances triées de la plus grande à la plus petite.",
            "permutation_12 doit trier les importances de la plus grande à la plus petite.")
    if not np.allclose(imp_12[FEATURES].to_numpy(), want_i_12[FEATURES].to_numpy(), atol=1e-9):
        on_train_12 = permutation_importance(GBR, X_TRAIN, Y_TRAIN, n_repeats=5, random_state=0).importances_mean
        if np.allclose(imp_12[FEATURES].to_numpy(), on_train_12, atol=1e-9):
            print("❌ Ex B6.12 : ces importances sont celles des districts d'entraînement : la permutation mesure ce que "
                  "le modèle perd sur des données qu'il n'a pas vues (relis 206.2).")
        else:
            same("B6.12", "importances par permutation (validation, 5 répétitions, random_state=0)",
                 imp_12[FEATURES].to_numpy(), want_i_12[FEATURES].to_numpy(), atol=1e-9)
    else:
        ok_12["perm"] = True
        print("✅ Ex B6.12 : importances par permutation sur la validation, comme attendu.")

with checked("B6.12"):
    out_12 = pd_ice_12(GBR, ROWS_12.copy(), "AveOccup")
    if not isinstance(out_12, dict) or set(out_12) != {"grid", "average", "individual"}:
        print("❌ Ex B6.12 : pd_ice_12 doit rendre un dictionnaire aux trois clés 'grid', 'average', 'individual'.")
        raise StopCheck
    ref_12 = partial_dependence(GBR, ROWS_12, ["AveOccup"], kind="both", grid_resolution=50)
    same("B6.12", "grid de AveOccup (50 valeurs)", out_12["grid"], ref_12["grid_values"][0])
    ind_12 = as_numpy(out_12["individual"])
    if ind_12 is not None and ind_12.ndim == 2 and ind_12.shape[1] > 1 and np.allclose(ind_12[:, 0], 0, atol=1e-9):
        print("❌ Ex B6.12 : tes courbes ICE partent toutes de 0 : c'est la version centrée (chaque courbe moins sa "
              "première valeur) ; le contrat veut les prédictions elles-mêmes.")
        raise StopCheck
    same("B6.12", "courbes ICE de AveOccup (300 districts × 50 valeurs)", out_12["individual"],
         ref_12["individual"][0], rtol=1e-9, atol=1e-9)
    if same("B6.12", "dépendance partielle de AveOccup", out_12["average"], ref_12["average"][0], rtol=1e-9, atol=1e-9):
        ok_12["pd"] = True
        verdict("B6.12", np.allclose(out_12["average"], np.asarray(out_12["individual"]).mean(axis=0)),
                "la dépendance partielle est la moyenne des courbes ICE.",
                "la dépendance partielle devrait être la moyenne des courbes ICE.")

if len(ok_12) == 3:                              # the figure, once your three results are right
    fig_12, axes_12 = plt.subplots(1, 4, figsize=(17, 3.8))
    order_12 = imp_12.index
    axes_12[0].barh(order_12[::-1], c_12[order_12[::-1]].abs(), color="tab:gray")
    axes_12[0].set_title("|coefficient| (features standardisées)", fontsize=10)
    axes_12[1].barh(order_12[::-1], imp_12[order_12[::-1]], color="tab:green")
    axes_12[1].set_title("permutation : baisse du R² (validation)", fontsize=10)
    for ax_12, feat_12 in zip(axes_12[2:], ("AveOccup", "HouseAge")):
        curves_12 = pd_ice_12(GBR, ROWS_12, feat_12)
        ax_12.plot(curves_12["grid"], np.asarray(curves_12["individual"])[:80].T, color="tab:blue", alpha=0.15, lw=0.8)
        ax_12.plot(curves_12["grid"], curves_12["average"], color="black", lw=3, label="dépendance partielle")
        ax_12.set_xlabel(f"{feat_12} imposé")
        ax_12.set_ylabel("prédiction (MedHouseVal)")
        ax_12.set_title(f"80 courbes ICE de {feat_12} et leur moyenne", fontsize=10)
        ax_12.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
    print("coefficients standardisés :", ", ".join(f"{k} {fr(v, 2)}" for k, v in c_12.items()))
    print("importances par permutation :", ", ".join(f"{k} {fr(v, 3)}" for k, v in imp_12.items()))

### Ex B6.13 — Prédire la forme des courbes ICE du revenu médian et de la latitude 🔮 ★ ⏱️ 10 min
**Objectif :** prédire ce que les courbes ICE d'un gradient boosting montrent de deux features, avant de les tracer.  
**Prérequis :** Ex B6.12 · fiche §206.2 (« La dépendance partielle », « Les courbes ICE ») · **Fil rouge :** California · **Parcours :** C

L'expérience trace, pour le gradient boosting de référence `GBR`, les courbes ICE de `MedInc` (le revenu médian du district, en dizaines de milliers de dollars) et de `Latitude` (de ≈ 33° au sud, vers San Diego, à ≈ 39° au nord de la grille) sur 500 districts de validation, avec leur moyenne, la dépendance partielle ; chaque grille a 50 valeurs, du 5ᵉ au 95ᵉ centile de la feature. La prédiction est en `MedHouseVal`, en centaines de milliers de dollars. **Avant d'exécuter**, écris tes quatre prédictions :
- a) `medinc_rise_13` : de combien la dépendance partielle de `MedInc` monte-t-elle du début à la fin de sa grille ? `"A"` moins de 0,5 ; `"B"` de 0,5 à 1,5 ; `"C"` de 1,5 à 3 ; `"D"` plus de 3 ;
- b) `lat_trend_13` : du sud au nord, la dépendance partielle de `Latitude` : `"A"` monte (de plus de 0,3) ; `"B"` reste à peu près plate (entre −0,3 et +0,3) ; `"C"` descend (de plus de 0,3) ;
- c) `lat_up_13` : quelle part des 500 courbes ICE de `Latitude` **finissent plus haut** qu'elles ne commencent ? `"A"` moins de 5 % ; `"B"` de 5 à 30 % ; `"C"` de 30 à 70 % ; `"D"` plus de 70 % ;
- d) `spread_13` : les courbes ICE de `Latitude` varient-elles **plus** d'un district à l'autre que celles de `MedInc` (un écart-type plus grand de leur variation, fin moins début) ? `True` ou `False`.

La cellule d'expérience n'affiche rien tant que tes quatre prédictions ne sont pas écrites ; elle les vérifie à la fin. Rien n'y dépend de ton code.

Dans tes notes, avant d'exécuter : ce que tu sais de la Californie (où sont les villes, la côte, les grandes régions) et ce que tu attends d'un modèle qui voit la latitude **et** la longitude.

In [ ]:
ROWS_13 = X_VAL.iloc[:500]                        # the 500 validation districts of the curves


def curves_13(feature):
    """(grid, average, individual) of the reference gradient boosting GBR for `feature` on ROWS_13: a grid of 50 values
    between the 5th and the 95th percentiles of the feature (written here without your code)."""
    result = partial_dependence(GBR, ROWS_13, [feature], kind="both", grid_resolution=50)
    return result["grid_values"][0], result["average"][0], result["individual"][0]

In [ ]:
medinc_rise_13, lat_trend_13, lat_up_13, spread_13 = "C", "C", "A", True

In [ ]:
wb.record("B6.13a", "C", mistakes={"lis la variation de la dépendance partielle de MedInc, sous la figure": {"A", "B", "D"}}, prediction=True)
wb.record("B6.13b", "C", mistakes={"suis la courbe épaisse de la latitude, du sud (à gauche) au nord (à droite)": "A", "lis la variation de la dépendance partielle de la latitude, sous la figure": "B"}, prediction=True)
wb.record("B6.13c", "A", mistakes={"lis la part des courbes ICE de la latitude qui finissent plus haut, sous la figure": {"B", "C", "D"}}, prediction=True)
wb.record("B6.13d", True, mistakes={"compare les deux écarts-types de la variation des ICE, sous la figure": False}, prediction=True)

In [ ]:
if any(answer is ... or answer is None for answer in [medinc_rise_13, lat_trend_13, lat_up_13, spread_13]):
    print("⏳ Ex B6.13 : écris d'abord tes quatre prédictions dans la cellule précédente.")
else:
    fig_13, axes_13 = plt.subplots(1, 2, figsize=(12, 4))
    seen_13 = {}
    for ax_13, feature_13 in zip(axes_13, ("MedInc", "Latitude")):
        grid_13, avg_13, ind_13 = curves_13(feature_13)
        ax_13.plot(grid_13, ind_13[:100].T, color="tab:blue", alpha=0.15, lw=0.8)
        ax_13.plot(grid_13, avg_13, color="black", lw=3, label="dépendance partielle")
        ax_13.set_xlabel(f"{feature_13} imposé (du 5e au 95e centile)")
        ax_13.set_ylabel("prédiction (MedHouseVal)")
        ax_13.set_title(f"100 courbes ICE de {feature_13} (sur 500) et leur moyenne", fontsize=10)
        ax_13.legend(fontsize=8)
        change_13 = ind_13[:, -1] - ind_13[:, 0]
        seen_13[feature_13] = {"rise": float(avg_13[-1] - avg_13[0]), "up": float(np.mean(change_13 > 0)),
                               "spread": float(np.std(change_13))}
    plt.tight_layout()
    plt.show()
    for feature_13, s_13 in seen_13.items():
        print(f"{feature_13:>8} : variation de la dépendance partielle (fin − début) {fr(s_13['rise'], 2)} ; "
              f"courbes ICE qui finissent plus haut {pct(s_13['up'])} ; écart-type de la variation des ICE {fr(s_13['spread'], 2)}")
    wb.check("B6.13a", medinc_rise_13)
    wb.check("B6.13b", lat_trend_13)
    wb.check("B6.13c", lat_up_13)
    wb.check("B6.13d", spread_13)

Dans tes notes : pour chaque prédiction ratée, qu'est-ce qui t'a trompé ? Que te disent les courbes ICE de la latitude que la dépendance partielle seule ne dit pas, et avec quelle autre feature la latitude interagit-elle sans doute ? Imposer une latitude à un district sans toucher à sa longitude : quelles combinaisons crée-t-on, et ces districts existent-ils (piège « une dépendance partielle hors des données ») ? Une courbe ICE dit-elle ce que **ferait** un déménagement du district vers le nord ?

### Ex B6.14 — shapley_values : Shapley exact par énumération des coalitions 🔨 ★★★ ⏱️ 45 min
**Objectif :** écrire le calcul exact des valeurs de Shapley interventionnelles, que shap approche.  
**Prérequis :** ✏️ B6.1 · ∂ B6.2 · fiche §206.3 (« Les features comme joueurs », « Un exemple à deux features », « Combien ça coûte ») · **Fil rouge :** California · **mylearn :** `explain.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** : ouvre `mon_travail/mylearn/explain.py` (créé par `python tools/start_chapter.py B6`), lis la docstring, remplace chaque `raise NotImplementedError(...)` de l'exercice par ton code et **enregistre**. NumPy seulement. La première cellule de vérification recharge ta librairie et l'essaie sur des cas visibles ; la seconde lance les tests de l'exercice. `python -m pytest tests/test_chb6_explain.py -q` les lance tous dans un terminal.

**Contrat** de `shapley_values(f, x, background, max_features=12)`, qui rend le couple `(phi, base_value)` :
- `f` ne reçoit que des tableaux 2-D de forme `(n, M)`, jamais une ligne seule, et rend une sortie de forme `(n,)`, par exemple `model.predict(X)` ou `model.predict_proba(X)[:, 1]` ; elle peut être appelée plusieurs fois ;
- $v(S)$ est la **moyenne de $f$** sur les $B$ lignes du fond dont les colonnes de $S$ prennent les valeurs de `x` ($f$ d'abord, la moyenne ensuite) ; **toutes** les lignes du fond comptent, sans sous-échantillonnage (contrairement au `max_samples=100` par défaut de shap) ;
- `phi` est un tableau de forme `(M,)` ; `base_value` vaut $v(\varnothing)$, la moyenne de $f$ sur le fond, un `float` Python ; l'efficacité `phi.sum() + base_value == f(x)` tient aux arrondis près ;
- `ValueError` si $M >$ `max_features` ($M =$ `max_features` est permis), si `x` n'est pas 1-D, ou si `background` n'est pas 2-D à $M$ colonnes ; `x` et `background` peuvent être des listes et ne sont jamais modifiés ;
- **coût** : la docstring prescrit l'énumération des $2^M$ coalitions, chaque $v(S)$ calculé **une seule fois** (un appel à `f` sur les $B$ lignes) ; les tests comptent les lignes passées à `f` et arrêtent ta fonction au-delà de `max(2·M·2**M·B, 500 000)` lignes, avec un message qui le dit. Ce budget laisse passer un $v(S)$ recalculé pour chaque feature, ou `f` appelée ligne par ligne ; il arrête une énumération des $M!$ ordres d'arrivée sans mémoire des $v(S)$ (40 320 ordres à $M = 8$), qui prendrait des minutes.

La première cellule de vérification regarde d'abord ce que reçoit `f`, puis essaie ta fonction sur les deux exemples de la docstring, sur un modèle linéaire à 5 features et un fond aléatoire de 20 lignes ($\phi_i = w_i(x_i - \bar b_i)$, ∂ B6.2), sur le gradient boosting de référence de California (8 features, fond de 50 lignes : efficacité, valeur de base, comparaison à `shap.explainers.Exact`, durée affichée), et sur quatre gardes. La seconde lance les 33 tests de l'exercice (`-k "test_shapley_values_"`).

Dans tes notes : combien de lignes ta fonction passe-t-elle à `f` pour $M = 8$ et $B = 50$, et combien en passerait une version qui recalcule chaque $v(S)$ pour chaque feature ? Pourquoi `f` doit-elle recevoir des tableaux 2-D (pense au coût d'un appel à `predict`) ? Que deviendrait la durée affichée avec 12 features (🧮 B6.7) ?

In [ ]:
with checked("B6.14"):
    E = mylearn.explain
    rng_14 = np.random.default_rng(14)                 # a linear model on a random background: w_i (x_i - mean_i)
    w5_14, x5_14, bg5_14 = rng_14.normal(size=5), rng_14.normal(size=5), rng_14.normal(size=(20, 5))
    shapes_14 = []

    def spy_14(X):                                     # first: what does f receive?
        shapes_14.append(np.shape(X))
        return np.asarray(X, dtype=float) @ w5_14

    E.shapley_values(spy_14, x5_14, bg5_14)
    bad_14 = sorted({s for s in shapes_14 if len(s) != 2 or s[1] != 5})
    verdict("B6.14", not bad_14, "f ne reçoit que des tableaux 2-D de 5 colonnes.",
            f"f a reçu les formes {bad_14[:3]} : appelle-la sur des tableaux (n, M), jamais sur une ligne seule.")
    rows_14 = sum(s[0] for s in shapes_14 if len(s) == 2)
    print(f"ℹ️ f a été appelée {len(shapes_14)} fois, sur {rows_14} lignes en tout (2**5 coalitions × 20 lignes = "
          f"{2 ** 5 * 20}).")
    w_14 = np.array([2.0, -1.0, 0.5])
    bg_14 = np.array([[0.0, 0.0, 0.0], [2.0, 2.0, 2.0]])
    phi_14, base_14 = pair("B6.14", "shapley_values", E.shapley_values(lambda X: X @ w_14, [3.0, 1.0, 1.0], bg_14.copy()),
                           names=("phi", "base_value"), floats=("base_value",))
    same("B6.14", "phi du modèle linéaire de la docstring", phi_14, [4.0, 0.0, 0.0])
    same("B6.14", "base_value du modèle linéaire de la docstring", base_14, 1.5)
    same("B6.14", "le produit x0 x1 de la docstring : l'interaction partagée",
         E.shapley_values(lambda X: X[:, 0] * X[:, 1], [1.0, 1.0], [[0.0, 0.0]])[0], [0.5, 0.5])

    same("B6.14", "modèle linéaire à 5 features, fond de 20 lignes : w_i (x_i − moyenne du fond)",
         E.shapley_values(lambda X: X @ w5_14 + 1.0, x5_14, bg5_14)[0], w5_14 * (x5_14 - bg5_14.mean(axis=0)),
         rtol=1e-8, atol=1e-10)

    X0_14, BG_14 = X_VAL.iloc[0].to_numpy(), X_TRAIN.iloc[:50].to_numpy()
    copies_14 = (X0_14.copy(), BG_14.copy())
    start_14 = time.perf_counter()
    phi_c_14, base_c_14 = pair("B6.14", "shapley_values", E.shapley_values(GBR.predict, X0_14, BG_14),
                               names=("phi", "base_value"), floats=("base_value",))
    seconds_14 = time.perf_counter() - start_14
    same("B6.14", "California, efficacité : somme des phi + base_value = f(x)", float(np.sum(phi_c_14)) + base_c_14,
         float(GBR.predict(X0_14[None])[0]), rtol=1e-9, atol=1e-9)
    same("B6.14", "California : base_value = moyenne de f sur les 50 lignes du fond", base_c_14,
         float(GBR.predict(BG_14).mean()), rtol=1e-9, atol=1e-9)
    if shap is not None:
        exact_14 = shap.explainers.Exact(GBR.predict, shap.maskers.Independent(BG_14, max_samples=len(BG_14)))(X0_14[None])
        same("B6.14", "California : phi contre shap.explainers.Exact (fond complet)", phi_c_14, exact_14.values[0],
             rtol=1e-7, atol=1e-9)
    print(f"ℹ️ California, 8 features, fond de 50 lignes (256 coalitions × 50 = 12 800 lignes) : {fr(seconds_14, 2)} s.")
    verdict("B6.14", np.array_equal(X0_14, copies_14[0]) and np.array_equal(BG_14, copies_14[1]),
            "x et le fond n'ont pas changé.", "shapley_values a modifié x ou le fond reçus.")

    try:
        E.shapley_values(lambda X: X.sum(axis=1), np.ones(3), np.zeros((2, 3)), max_features=3)
        print("✅ Ex B6.14 : M = max_features est permis.")
    except ValueError as exc_14:
        print(f"❌ Ex B6.14 : M = max_features doit être permis (seul M > max_features lève) : reçu ValueError ({exc_14}).")
    refuses("B6.14", "M = 4 > max_features = 3", lambda: E.shapley_values(lambda X: X.sum(axis=1), np.ones(4),
                                                                        np.zeros((2, 4)), max_features=3))
    refuses("B6.14", "x en 2-D", lambda: E.shapley_values(lambda X: X.sum(axis=1), np.ones((1, 3)), np.zeros((2, 3))))
    refuses("B6.14", "un fond de 2 colonnes pour x de 3", lambda: E.shapley_values(lambda X: X.sum(axis=1), np.ones(3),
                                                                                   np.zeros((2, 2))))

In [ ]:
if written(lambda: mylearn.explain.shapley_values(lambda X: X.sum(axis=1), [0.0], [[0.0]])):
    run_pytest_group(TEST_EXPLAIN, impl="ref", keyword="test_shapley_values_")   # 33 tests
else:
    print("⏳ Ex B6.14 : les tests attendent ta fonction shapley_values.")

### Ex B6.15 — SHAP sur un gradient boosting California : TreeExplainer, waterfall, beeswarm, dépendance 📦 ★★ ⏱️ 30 min
**Objectif :** expliquer un gradient boosting avec TreeSHAP, vérifier qu'il calcule le même Shapley que ta fonction, et lire les trois graphiques de shap.  
**Prérequis :** Ex B6.14 · ch. 14 · fiche §206.4 (« L'API de shap », « Les trois graphiques à connaître ») · **Fil rouge :** California · **Parcours :** R, C

**Contrat** des deux fonctions de la cellule suivante :
- `fit_15(X, y)` rend un `HistGradientBoostingRegressor(max_depth=3, random_state=0)` (des arbres peu profonds, les autres réglages par défaut), ajusté sur `(X, y)` ;
- `tree_shap_15(model, background, rows)` rend le `shap.Explanation` de `shap.TreeExplainer(model, data=background, feature_perturbation="interventional")` appliqué au DataFrame `rows` : `.values` de forme `(len(rows), 8)`, `.base_values`, `.feature_names`.

La cellule donnée fixe `BG_15`, le fond (les 50 premiers districts d'entraînement), et `ROWS_15`, les 1 000 premiers districts de validation à expliquer. La cellule de vérification ajuste ton modèle sur `X_TRAIN`, l'explique, contrôle l'efficacité sur les 1 000 districts, puis vérifie quatre grandeurs que tes fonctions calculent (rien à taper, aucune à arrondir) : a) la valeur de base ; b) la valeur SHAP de `MedInc` pour le premier district ; c) la moyenne des $\lvert\phi_{\text{MedInc}}\rvert$ sur les 1 000 districts ; d) si ta `shapley_values` (B6.14), avec le même fond de 50 lignes, redonne les valeurs de TreeSHAP sur le premier district à $10^{-6}$ près (`True` ou `False`). La cellule suivante trace le waterfall du premier district, le beeswarm et la dépendance de `MedInc`.

> ⚠️ **Piège classique — TreeSHAP et les données en float64** — `TreeExplainer` parcourt les arbres en **float32**. Un `HistGradientBoosting` pose certains seuils exactement sur des valeurs répétées des données (la latitude et la longitude de California en ont beaucoup) : une valeur égale à un seuil peut alors passer de l'autre côté, et TreeSHAP perd l'efficacité (des écarts jusqu'à 0,5 sur les données d'origine de scikit-learn, mesurés avec shap 0.54.1). C'est pourquoi `wbx.california()` arrondit les features en float32 : TreeSHAP interventionnel y redonne le Shapley exact à $10^{-8}$ près. Sur tes propres données, vérifie toujours l'efficacité (shap le fait : `check_additivity`).

Dans tes notes : pourquoi la valeur de base a) n'est-elle ni la moyenne des cibles d'entraînement, ni la prédiction moyenne des 1 000 districts expliqués ? Sur le waterfall, quelles features font monter et descendre la prédiction du premier district, et de combien ? Sur le beeswarm, dans quel sens joue un grand revenu, et quelles features ont une grande dispersion ? Sur la dépendance de `MedInc`, qu'indique la dispersion verticale à revenu égal ? Combien de temps a pris TreeSHAP sur 1 000 districts, et combien prendrait ta `shapley_values` (sa durée pour un district est affichée en B6.14) ?

In [ ]:
BG_15 = X_TRAIN.iloc[:50]          # the background: the first 50 training districts
ROWS_15 = X_VAL.iloc[:1000]        # the 1 000 validation districts to explain
MEDINC = FEATURES.index("MedInc")  # the column of MedInc (0)

In [ ]:
def fit_15(X, y):
    """HistGradientBoostingRegressor(max_depth=3, random_state=0), every other setting by default, fitted on (X, y)."""
    return HistGradientBoostingRegressor(max_depth=3, random_state=0).fit(X, y)


def tree_shap_15(model, background, rows):
    """The shap.Explanation of shap.TreeExplainer(model, data=background, feature_perturbation="interventional")
    on the DataFrame `rows`: .values of shape (len(rows), 8), .base_values, .feature_names."""
    explainer = shap.TreeExplainer(model, data=background, feature_perturbation="interventional")
    return explainer(rows)


SV_15 = MODEL_15 = None
BASE_OK_15 = False
with checked("B6.15"):
    if shap is None:
        print("⏳ Ex B6.15 : cet exercice a besoin de shap (wb.ensure('shap')).")
        raise StopCheck
    model_15 = fit_15(X_TRAIN.copy(), Y_TRAIN.copy())
    if not returned("B6.15", "fit_15", model_15):
        raise StopCheck
    if not isinstance(model_15, HistGradientBoostingRegressor):
        print(f"❌ Ex B6.15 : fit_15 doit rendre un HistGradientBoostingRegressor, reçu {type(model_15).__name__}.")
        raise StopCheck
    want_p_15 = HistGradientBoostingRegressor(max_depth=3, random_state=0).get_params()
    diff_15 = {k: v for k, v in model_15.get_params().items() if want_p_15.get(k) != v}
    verdict("B6.15", not diff_15, "réglages : max_depth=3, random_state=0, le reste par défaut.",
            f"réglages différents du contrat : {diff_15}.")
    ref_15 = HistGradientBoostingRegressor(max_depth=3, random_state=0).fit(X_TRAIN, Y_TRAIN)
    if not same("B6.15", "prédictions de fit_15 sur 200 districts de validation", model_15.predict(X_VAL.iloc[:200]),
                ref_15.predict(X_VAL.iloc[:200]), rtol=1e-9, atol=1e-9):
        raise StopCheck
    MODEL_15 = model_15
    start_15 = time.perf_counter()
    sv_15 = tree_shap_15(model_15, BG_15.copy(), ROWS_15.copy())
    print(f"ℹ️ tree_shap_15 sur les 1 000 districts : {fr(time.perf_counter() - start_15, 1)} s.")
    if not returned("B6.15", "tree_shap_15", sv_15):
        raise StopCheck
    if isinstance(sv_15, np.ndarray):
        print("❌ Ex B6.15 : tree_shap_15 rend un tableau NumPy : le contrat veut le shap.Explanation de explainer(rows) "
              "(avec .base_values et .feature_names), pas explainer.shap_values(rows).")
        raise StopCheck
    if np.shape(getattr(sv_15, "values", None)) != (len(ROWS_15), len(FEATURES)):
        print("❌ Ex B6.15 : tree_shap_15 doit rendre un shap.Explanation dont .values a la forme (1000, 8), reçu "
              f"{type(sv_15).__name__} de forme {np.shape(getattr(sv_15, 'values', None))}.")
        raise StopCheck
    if not getattr(sv_15, "feature_names", None):
        sv_15.feature_names = list(FEATURES)               # rows given as an array: the names for the plots
    SV_15 = sv_15
    gap_eff_15 = float(np.abs(sv_15.values.sum(axis=1) + np.ravel(sv_15.base_values) - model_15.predict(ROWS_15)).max())
    verdict("B6.15", gap_eff_15 < 1e-6, f"efficacité sur les 1 000 districts : écart maximal {gap_eff_15:.1e}.",
            f"efficacité : somme des valeurs SHAP + valeur de base s'écarte de la prédiction de {gap_eff_15:.2g} : "
            "l'explication porte-t-elle sur ce modèle et sur ces lignes ?")
    BASE_OK_15 = bool(abs(float(np.ravel(sv_15.base_values)[0]) - float(model_15.predict(BG_15).mean())) < 1e-6)
    verdict("B6.15", BASE_OK_15, "la valeur de base est la prédiction moyenne de ton modèle sur les 50 lignes de BG_15.",
            "la valeur de base n'est pas la prédiction moyenne de ton modèle sur les 50 lignes de BG_15 : tree_shap_15 "
            "reçoit-il tout le fond, en mode interventionnel ?")
    wb.check("B6.15a", float(np.ravel(sv_15.base_values)[0]), computed=True)
    wb.check("B6.15b", float(sv_15.values[0, MEDINC]), computed=True)
    wb.check("B6.15c", float(np.mean(np.abs(sv_15.values[:, MEDINC]))), computed=True)

with checked("B6.15"):
    if SV_15 is None:
        raise StopCheck
    if not BASE_OK_15:
        print("⏳ Ex B6.15d : corrige d'abord tree_shap_15 (sa valeur de base, ci-dessus) : la comparaison à ta "
              "shapley_values se fait avec le même fond de 50 lignes.")
        raise StopCheck
    if not shapley_ready("B6.15d"):
        raise StopCheck
    x0_15 = ROWS_15.iloc[0].to_numpy()
    phi_15, base_15 = pair("B6.15", "shapley_values", mylearn.explain.shapley_values(MODEL_15.predict, x0_15,
                                                                                     BG_15.to_numpy()),
                           names=("phi", "base_value"), floats=("base_value",))
    gap_15 = max(float(np.abs(np.asarray(phi_15, dtype=float) - SV_15.values[0]).max()),
                 abs(base_15 - float(np.ravel(SV_15.base_values)[0])))
    print(f"ℹ️ écart maximal entre TreeExplainer et ta shapley_values sur le premier district : {gap_15:.1e}")
    wb.check("B6.15d", bool(gap_15 <= 1e-6))

In [ ]:
# The right values, computed above with the solution; the classic slips, recomputed here
tree_path_15 = shap.TreeExplainer(MODEL_15)(ROWS_15.iloc[:1])                # without background: path-dependent
default_15 = shap.TreeExplainer(GBR, data=BG_15, feature_perturbation="interventional")(ROWS_15)   # the default boosting
bg100_15 = shap.TreeExplainer(MODEL_15, data=X_TRAIN.iloc[:100], feature_perturbation="interventional")(ROWS_15.iloc[:1])
path_bg_15 = shap.TreeExplainer(MODEL_15, data=BG_15, feature_perturbation="tree_path_dependent")(ROWS_15)   # same base
PATH_BG_15 = ("c'est le mode « chemins de l'arbre » (feature_perturbation=\"tree_path_dependent\"), même avec un fond : "
              "relis les deux modes de TreeExplainer (§206.4)")
base_r_15 = float(np.ravel(SV_15.base_values)[0])
phi_r_15 = float(SV_15.values[0, MEDINC])
mean_r_15 = float(np.mean(np.abs(SV_15.values[:, MEDINC])))


def distinct_15(right, slips):
    """The slips whose value differs from the right one, and from each other, at 4 decimals."""
    kept, seen = {}, {round(right, 4)}
    for message, value in slips.items():
        if round(value, 4) not in seen:
            kept[message] = value
            seen.add(round(value, 4))
    return kept


wb.record("B6.15a", base_r_15, decimals=4, mistakes=distinct_15(base_r_15, {
    "c'est la valeur de base sans fond (mode « chemins de l'arbre ») : relis les deux modes de TreeExplainer (§206.4)":
        float(np.ravel(tree_path_15.base_values)[0]),
    "c'est la moyenne des cibles d'entraînement : la valeur de base est la prédiction moyenne sur le fond": float(Y_TRAIN.mean()),
    "c'est la prédiction moyenne sur les districts expliqués : la valeur de base se calcule sur le fond": float(MODEL_15.predict(ROWS_15).mean()),
    "c'est le gradient boosting par défaut : le contrat veut max_depth=3": float(np.ravel(default_15.base_values)[0])}))
wb.record("B6.15b", phi_r_15, decimals=4, mistakes=distinct_15(phi_r_15, {
    "c'est la valeur sans fond (mode « chemins de l'arbre ») : relis les deux modes de TreeExplainer (§206.4)":
        float(tree_path_15.values[0, MEDINC]),
    "c'est le gradient boosting par défaut : le contrat veut max_depth=3": float(default_15.values[0, MEDINC]),
    "c'est un fond de 100 lignes : le contrat en prend 50": float(bg100_15.values[0, MEDINC]),
    PATH_BG_15: float(path_bg_15.values[0, MEDINC]),
    "vérifie le signe": -phi_r_15}))
tree_path_all_15 = shap.TreeExplainer(MODEL_15)(ROWS_15)
wb.record("B6.15c", mean_r_15, decimals=4, mistakes=distinct_15(mean_r_15, {
    "c'est la valeur sans fond (mode « chemins de l'arbre ») : relis les deux modes de TreeExplainer (§206.4)":
        float(np.mean(np.abs(tree_path_all_15.values[:, MEDINC]))),
    "c'est la moyenne des valeurs SHAP signées : on demande la moyenne de leurs valeurs absolues":
        float(np.mean(SV_15.values[:, MEDINC])),
    "c'est la moyenne des carrés : on demande la moyenne des valeurs absolues": float(np.mean(SV_15.values[:, MEDINC] ** 2)),
    "c'est le gradient boosting par défaut : le contrat veut max_depth=3": float(np.mean(np.abs(default_15.values[:, MEDINC]))),
    PATH_BG_15: float(np.mean(np.abs(path_bg_15.values[:, MEDINC])))}))
wb.record("B6.15d", True, mistakes={
    "TreeExplainer et ta shapley_values diffèrent de plus de 1e-6 alors que tree_shap_15 a la bonne valeur de base : ta "
    "fonction passe-t-elle ses tests (Ex B6.14), et garde-t-elle toutes les lignes du fond ?": False})

In [ ]:
if SV_15 is not None:                       # the three plots of 206.4, on your explanation
    shap.plots.waterfall(SV_15[0], show=False)
    plt.title("waterfall du premier district", fontsize=10)
    plt.show()
    shap.plots.beeswarm(SV_15, show=False)
    plt.title("beeswarm des 1 000 districts", fontsize=10)
    plt.show()
    shap.plots.scatter(SV_15[:, MEDINC], color=SV_15, show=False)
    plt.title("dépendance de MedInc (couleur : la feature qui interagit le plus)", fontsize=10)
    plt.show()
else:
    print("⏳ Ex B6.15 : les graphiques attendent ton explication (cellule précédente).")

### Ex B6.16 — Le fond change tout : trois fonds différents, puis une feature dupliquée 🔬 ★★★ ⏱️ 40 min
**Objectif :** mesurer ce que le choix du fond et une feature dupliquée font aux valeurs de Shapley d'un gradient boosting.  
**Prérequis :** Ex B6.15 · fiche §206.3 (« Le fond fait partie de la question ») et §206.4 (« Features corrélées : ce que SHAP répartit ») · 🧠 Q5, Q6 · **Fil rouge :** California · **Parcours :** C

Deux expériences sur le gradient boosting de référence `GBR`. **Ne présume pas l'issue : mesure-la.**

**1. Trois fonds.** La cellule donnée choisit `X_16`, le district de validation le plus proche du 75ᵉ centile du revenu, et trois fonds (`BACKGROUNDS_16`) : `"population"`, 50 districts d'entraînement tirés au hasard ; `"riches"`, 50 districts du décile de revenu le plus haut ; `"ligne type"`, **une** ligne, la médiane de chaque feature. Contrat de `explain_16(model, x, background)` : il rend le couple `(phi, base)`, les valeurs de Shapley interventionnelles de `model.predict` au district `x` (une `pd.Series` des 8 features) avec le fond `background` (un DataFrame), un tableau `(8,)`, et la valeur de base, un `float` ; ta `shapley_values` ou `shap.TreeExplainer` (les mêmes nombres, B6.15). La cellule de vérification compare tes valeurs à TreeSHAP pour les trois fonds, puis affiche le tableau des valeurs de base et des $\phi$.

**2. Une feature dupliquée.** `add_copy_16(X, noise, rng)` (donnée) ajoute à un DataFrame, juste après `MedInc`, la colonne `"MedInc_copy"` = `MedInc` + `noise` × (écart-type de `MedInc`) × un bruit gaussien tiré par `rng`. Contrat de `copy_experiment_16(noise, seed, n_rows=200)` :
- `rng = np.random.default_rng(seed)`, puis `A = add_copy_16(X_TRAIN, noise, rng)` et `V = add_copy_16(X_VAL, noise, rng)`, dans cet ordre, avec le même `rng` ;
- un `HistGradientBoostingRegressor(random_state=seed)` ajusté sur `(A, Y_TRAIN)` ;
- `shap.TreeExplainer` interventionnel, de fond les 50 premières lignes de `A`, sur les `n_rows` premières lignes de `V` ;
- il rend `{"MedInc": moyenne des |φ_MedInc|, "MedInc_copy": moyenne des |φ_copie|, "sum": moyenne des |φ_MedInc + φ_copie|}` (des floats).

La cellule d'expérience essaie d'abord ta fonction sur un petit cas (bruit 0,01, graine 0, 50 districts), puis la lance pour trois niveaux de bruit (0, 0,01 et 0,1 écart-type) et trois graines (≈ 10 s), à côté de la moyenne des $\lvert\phi_{\text{MedInc}}\rvert$ du modèle sans copie.

Dans tes notes : avec quel fond la valeur SHAP de `MedInc` change-t-elle de signe, et pourquoi ? Que change le fond « ligne type » par rapport à « population » ? Quelle phrase écrirais-tu sous un waterfall pour dire quel fond a servi ? Pour la copie : que reçoit-elle quand elle est exacte (bruit 0), et pourquoi ? Avec un bruit de 0,01, le partage entre `MedInc` et sa copie est-il stable d'une graine à l'autre ? Et la somme ? Que rapporterais-tu à un métier qui demande « quelle est l'importance du revenu ? » quand deux colonnes le portent ?

In [ ]:
# the district to explain: the validation district closest to the 75th percentile of MedInc
X_16 = X_VAL.iloc[int(np.argmin(np.abs(X_VAL["MedInc"].to_numpy() - X_VAL["MedInc"].quantile(0.75))))]
RICH_16 = X_TRAIN[X_TRAIN["MedInc"] >= X_TRAIN["MedInc"].quantile(0.9)]          # the top decile of income
BACKGROUNDS_16 = {"population": X_TRAIN.sample(50, random_state=0),             # 50 districts drawn at random
                  "riches": RICH_16.sample(50, random_state=0),                  # 50 districts of the top decile
                  "ligne type": X_TRAIN.median().to_frame().T.astype("float32").astype("float64")}   # one row: medians
SD_MEDINC_16 = float(X_TRAIN["MedInc"].std())
NOISES_16, SEEDS_16 = (0.0, 0.01, 0.1), (0, 1, 2)


def add_copy_16(X, noise, rng):
    """A copy of the DataFrame X with the column "MedInc_copy" = MedInc + noise * SD_MEDINC_16 * N(0, 1), drawn by
    rng.standard_normal(len(X)), placed right after MedInc and rounded to float32 like the other features (B6.15)."""
    out = X.copy()
    copy = X["MedInc"].to_numpy() + noise * SD_MEDINC_16 * rng.standard_normal(len(X))
    out.insert(1, "MedInc_copy", copy.astype("float32").astype("float64"))
    return out


print(f"district expliqué : MedInc {fr(X_16['MedInc'], 2)}, Latitude {fr(X_16['Latitude'], 2)}, prédiction "
      f"{fr(GBR.predict(X_16.to_frame().T)[0], 2)} ; fonds : " + ", ".join(f"{k} ({len(v)} ligne{'s' * (len(v) > 1)})"
                                                                            for k, v in BACKGROUNDS_16.items()))

In [ ]:
def explain_16(model, x, background):
    """(phi, base): the interventional Shapley values of model.predict at the district x (a pd.Series of the 8
    features) with the DataFrame `background`, as an array (8,), and the base value, a float."""
    explanation = shap.TreeExplainer(model, data=background, feature_perturbation="interventional")(x.to_frame().T)
    return explanation.values[0], float(np.ravel(explanation.base_values)[0])


def copy_experiment_16(noise, seed, n_rows=200):
    """The duplicated feature: rng = np.random.default_rng(seed); A = add_copy_16(X_TRAIN, noise, rng), then
    V = add_copy_16(X_VAL, noise, rng) (the same rng, in this order); HistGradientBoostingRegressor(random_state=seed)
    fitted on (A, Y_TRAIN); shap.TreeExplainer (interventional, background: the first 50 rows of A) on the first
    n_rows rows of V. Returns {"MedInc": mean |phi_MedInc|, "MedInc_copy": mean |phi_copy|,
    "sum": mean |phi_MedInc + phi_copy|} over these rows (floats)."""
    rng = np.random.default_rng(seed)
    A, V = add_copy_16(X_TRAIN, noise, rng), add_copy_16(X_VAL, noise, rng)
    model = HistGradientBoostingRegressor(random_state=seed).fit(A, Y_TRAIN)
    values = shap.TreeExplainer(model, data=A.iloc[:50], feature_perturbation="interventional")(V.iloc[:n_rows]).values
    i, j = list(A.columns).index("MedInc"), list(A.columns).index("MedInc_copy")
    return {"MedInc": float(np.mean(np.abs(values[:, i]))), "MedInc_copy": float(np.mean(np.abs(values[:, j]))),
            "sum": float(np.mean(np.abs(values[:, i] + values[:, j])))}


TABLE_16 = {}
with checked("B6.16"):
    if shap is None:
        print("⏳ Ex B6.16 : cet exercice a besoin de shap (wb.ensure('shap')).")
        raise StopCheck
    for name_16, bg_16 in BACKGROUNDS_16.items():
        got_16 = explain_16(GBR, X_16.copy(), bg_16.copy())
        phi_16, base_16 = pair("B6.16", "explain_16", got_16, names=("phi", "base"), floats=("base",))
        if len(bg_16) > 1 and abs(base_16 - float(GBR.predict(bg_16.iloc[:1])[0])) < 1e-9:
            print(f"❌ Ex B6.16 : fond « {name_16} » : ta valeur de base est la prédiction de la première ligne du fond : "
                  f"v(∅) est la moyenne de f sur ses {len(bg_16)} lignes.")
            continue
        want_16 = shap.TreeExplainer(GBR, data=bg_16, feature_perturbation="interventional")(X_16.to_frame().T)
        ok_16 = same("B6.16", f"fond « {name_16} » : phi", phi_16, want_16.values[0], rtol=1e-6, atol=1e-6)
        ok_16 &= same("B6.16", f"fond « {name_16} » : valeur de base", base_16, float(np.ravel(want_16.base_values)[0]),
                      rtol=1e-6, atol=1e-6)
        if ok_16:
            TABLE_16[name_16] = (base_16, np.asarray(phi_16, dtype=float))
if len(TABLE_16) == len(BACKGROUNDS_16):
    f_16 = float(GBR.predict(X_16.to_frame().T)[0])
    print(f"\nprédiction du district : {fr(f_16, 3)}")
    print(f"{'fond':>12} | {'base':>6} | " + " | ".join(f"{name[:9]:>9}" for name in FEATURES))
    for name_16, (base_16, phi_16) in TABLE_16.items():
        print(f"{name_16:>12} | {fr(base_16, 3):>6} | " + " | ".join(f"{fr(v, 3):>9}" for v in phi_16))

In [ ]:
COPIES_16 = {}
with checked("B6.16"):
    if shap is None:
        raise StopCheck
    got_16 = copy_experiment_16(0.01, 0, n_rows=50)      # a small first run, checked before the whole grid
    if not isinstance(got_16, dict) or set(got_16) != {"MedInc", "MedInc_copy", "sum"}:
        print("❌ Ex B6.16 : copy_experiment_16 doit rendre un dictionnaire aux clés 'MedInc', 'MedInc_copy', 'sum'.")
        raise StopCheck
    rng_16 = np.random.default_rng(0)
    A_16, V_16 = add_copy_16(X_TRAIN, 0.01, rng_16), add_copy_16(X_VAL, 0.01, rng_16)
    m_16 = HistGradientBoostingRegressor(random_state=0).fit(A_16, Y_TRAIN)
    v_16 = shap.TreeExplainer(m_16, data=A_16.iloc[:50], feature_perturbation="interventional")(V_16.iloc[:50]).values
    want_16 = [np.mean(np.abs(v_16[:, 0])), np.mean(np.abs(v_16[:, 1])), np.mean(np.abs(v_16[:, 0] + v_16[:, 1]))]
    if not same("B6.16", "copy_experiment_16(0.01, 0, n_rows=50)", [got_16[k] for k in ("MedInc", "MedInc_copy", "sum")],
                want_16, rtol=1e-6, atol=1e-6):
        raise StopCheck
    start_16 = time.perf_counter()
    for noise_16 in NOISES_16:
        for seed_16 in SEEDS_16:
            COPIES_16[noise_16, seed_16] = copy_experiment_16(noise_16, seed_16)
    print(f"ℹ️ {len(COPIES_16)} entraînements et explications en {time.perf_counter() - start_16:.0f} s.\n")
    alone_16 = shap.TreeExplainer(GBR, data=X_TRAIN.iloc[:50], feature_perturbation="interventional")(X_VAL.iloc[:200])
    print(f"sans copie : moyenne de |phi_MedInc| sur les 200 districts = {fr(np.mean(np.abs(alone_16.values[:, 0])), 3)}")
    for noise_16 in NOISES_16:
        cells_16 = [COPIES_16[noise_16, s] for s in SEEDS_16]
        print(f"bruit {fr(noise_16, 2)} écart-type : " + " ; ".join(
            f"graine {s} : MedInc {fr(c['MedInc'], 3)}, copie {fr(c['MedInc_copy'], 3)}, somme {fr(c['sum'], 3)}"
            for s, c in zip(SEEDS_16, cells_16)))

### Ex B6.17 — Penguins : SHAP par classe pour une régression logistique et une forêt aléatoire 📦 ★★ ⏱️ 25 min
**Objectif :** expliquer un classifieur à trois classes, une sortie à la fois, en logit puis en probabilité.  
**Prérequis :** Ex B6.15 · ∂ B6.2 · fiche §206.4 (« L'API de shap ») · ch. 13 et 14 · **Fil rouge :** Penguins · **Parcours :** C

Pourquoi ce manchot est-il un Gentoo ? La cellule donnée charge le découpage figé de Penguins (`wbx.penguins()` : 249 manchots d'entraînement, 84 de test, les 4 mesures en colonnes, l'espèce en label) et `GENTOO_17`, le premier Gentoo du test ; au ch. 0A tu as découvert ce dataset, au ch. 13 tu l'as classé. Un classifieur multiclasse a **une sortie par classe** : on explique une sortie à la fois, la probabilité de la classe $k$ ou son logit, et ces deux sorties ne donnent pas les mêmes valeurs (206.4). **Contrat** :
- `fit_logistic_17(X, y)` rend `make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))` ajusté ; `fit_forest_17(X, y)`, `RandomForestClassifier(n_estimators=200, random_state=0)` ajusté ;
- `class_output_17(model, k, kind="proba")` rend la fonction `f(X) -> (n,)` d'**une** sortie : la probabilité de la classe d'indice `k` (`predict_proba`) pour `kind="proba"`, son logit (`decision_function`) pour `kind="logit"` ;
- `shap_by_class_17(model, x, background, kind="proba")` rend le couple `(phi, base)` : `phi` de forme `(3, 4)`, une ligne par classe dans l'ordre de `model.classes_` (`SPECIES`), calculée par ta `shapley_values` sur la sortie de cette classe, et `base` de forme `(3,)`.

La cellule de vérification compare tes deux modèles à scikit-learn, puis explique `GENTOO_17` avec les modèles de référence et tout l'entraînement pour fond : en logit pour la régression logistique, elle compare tes valeurs au cas linéaire ($\phi_{ki} = w_{ki}(x_i - \bar b_i)$ dans les unités des mesures, ∂ B6.2) ; en probabilité, elle vérifie l'efficacité de chaque classe et les valeurs de base ; pour la forêt, elle compare tes valeurs à `shap.TreeExplainer` interventionnel, avec tout le fond (`shap.maskers.Independent(XP_TRAIN, max_samples=249)` : sinon il n'en garde que 100), à $2 \cdot 10^{-3}$ près, parce que sur une forêt TreeSHAP s'écarte lui-même de `predict_proba` de quelques $10^{-4}$. Puis elle trace les trois explications et affiche, pour chacune, la somme sur les trois classes de chaque feature.

Dans tes notes : quelles mesures font du manchot un Gentoo selon chaque explication ? Les logits et les probabilités classent-ils les mesures dans le même ordre ? Pourquoi les sommes sur les classes affichées valent-elles 0 pour les probabilités (pense à l'additivité et à ce que vaut la somme des trois probabilités) ? Elles valent 0 aussi pour les logits : que dit cela des coefficients de la régression logistique multinomiale de scikit-learn (affiche `LR_17[-1].coef_.sum(axis=0)`) ? La forêt et la régression logistique s'appuient-elles sur les mêmes mesures ?

In [ ]:
PENG = wbx.penguins()                               # frozen split of Penguins: 249 penguins, 84 for the test
XP_TRAIN, XP_TEST = PENG.X_train.to_numpy(), PENG.X_test.to_numpy()   # the 4 measures, as arrays (columns PENG.features)
YP_TRAIN, YP_TEST = PENG.y_train.to_numpy(), PENG.y_test.to_numpy()   # the species, strings
SPECIES = PENG.classes                               # ['Adelie', 'Chinstrap', 'Gentoo']: the order of predict_proba
GENTOO_17 = XP_TEST[np.flatnonzero(YP_TEST == "Gentoo")[0]]           # the penguin to explain: the first Gentoo
print("le manchot expliqué :", ", ".join(f"{name} {fr(v, 1)}" for name, v in zip(PENG.features, GENTOO_17)))

In [ ]:
def fit_logistic_17(X, y):
    """make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), fitted on (X, y), returned."""
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X, y)


def fit_forest_17(X, y):
    """RandomForestClassifier(n_estimators=200, random_state=0), fitted on (X, y), returned."""
    return RandomForestClassifier(n_estimators=200, random_state=0).fit(X, y)


def class_output_17(model, k, kind="proba"):
    """The function f(X) -> (n,) of ONE output of a classifier: the probability of the class of index k
    (predict_proba) for kind="proba", its logit (decision_function) for kind="logit"."""
    if kind == "proba":
        return lambda X: model.predict_proba(X)[:, k]
    if kind == "logit":
        return lambda X: model.decision_function(X)[:, k]
    raise ValueError(f"kind must be 'proba' or 'logit', got {kind!r}")


def shap_by_class_17(model, x, background, kind="proba"):
    """(phi, base): phi of shape (3, 4), the Shapley values of the 4 measures for each class (a row per class, in the
    order of model.classes_), base of shape (3,), with your shapley_values, one output at a time."""
    results = [mylearn.explain.shapley_values(class_output_17(model, k, kind), x, background)
               for k in range(len(model.classes_))]
    return np.array([phi for phi, _ in results]), np.array([base for _, base in results])


LR_17 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(XP_TRAIN, YP_TRAIN)   # references
RF_17 = RandomForestClassifier(n_estimators=200, random_state=0).fit(XP_TRAIN, YP_TRAIN)
G_17 = GENTOO_17[None]
with checked("B6.17"):
    lr_17 = fit_logistic_17(XP_TRAIN.copy(), YP_TRAIN.copy())
    if returned("B6.17", "fit_logistic_17", lr_17):
        steps_17 = [type(s).__name__ for _, s in getattr(lr_17, "steps", [])]
        verdict("B6.17", steps_17 == ["StandardScaler", "LogisticRegression"],
                "fit_logistic_17 : StandardScaler puis LogisticRegression.",
                f"fit_logistic_17 doit rendre le Pipeline (StandardScaler, LogisticRegression), reçu {steps_17 or type(lr_17).__name__}.")
        same("B6.17", "probabilités de fit_logistic_17 sur le test", lr_17.predict_proba(XP_TEST),
             LR_17.predict_proba(XP_TEST), rtol=1e-6, atol=1e-8)
    rf_17 = fit_forest_17(XP_TRAIN.copy(), YP_TRAIN.copy())
    if returned("B6.17", "fit_forest_17", rf_17):
        same("B6.17", "probabilités de fit_forest_17 sur le test", rf_17.predict_proba(XP_TEST), RF_17.predict_proba(XP_TEST))

with checked("B6.17"):
    f_p_17, f_l_17 = class_output_17(LR_17, 2, "proba"), class_output_17(LR_17, 2, "logit")
    if not (callable(f_p_17) and callable(f_l_17)):
        print("❌ Ex B6.17 : class_output_17 doit rendre une fonction f(X) -> (n,).")
        raise StopCheck
    same("B6.17", "class_output_17(LR, 2, 'proba') sur le test", f_p_17(XP_TEST), LR_17.predict_proba(XP_TEST)[:, 2])
    out_l_17 = f_l_17(XP_TEST)
    arr_l_17 = as_numpy(out_l_17)
    if (arr_l_17 is not None and arr_l_17.shape == (len(XP_TEST),)
            and np.allclose(arr_l_17, np.log(LR_17.predict_proba(XP_TEST))[:, 2], rtol=1e-6, atol=1e-8)):
        print("❌ Ex B6.17 : class_output_17(LR, 2, 'logit') rend le log de la probabilité : il diffère du logit "
              "(decision_function) d'une quantité qui change d'un manchot à l'autre, le log de la somme des "
              "exponentielles des trois logits.")
    else:
        same("B6.17", "class_output_17(LR, 2, 'logit') sur le test", out_l_17, LR_17.decision_function(XP_TEST)[:, 2])

RESULTS_17 = {}
ORDERS_17 = [[0, 2, 1], [1, 0, 2], [1, 2, 0], [2, 0, 1], [2, 1, 0]]   # the other orders of the three classes


def shifted_17(got, want):
    """True when the rows of `got` are those of `want` in another order of the classes."""
    arr = as_numpy(got)
    want = np.asarray(want, dtype=float)
    return (arr is not None and arr.shape == want.shape and not np.allclose(arr, want, rtol=1e-6, atol=1e-8)
            and any(np.allclose(arr, want[order], rtol=1e-6, atol=1e-8) for order in ORDERS_17))


with checked("B6.17"):
    if not shapley_ready("B6.17"):
        raise StopCheck
    phi_l_17, base_l_17 = pair("B6.17", "shap_by_class_17", shap_by_class_17(LR_17, GENTOO_17.copy(), XP_TRAIN.copy(), "logit"),
                               names=("phi", "base"))
    if np.shape(phi_l_17) != (3, 4) or np.shape(base_l_17) != (3,):
        print(f"❌ Ex B6.17 : attendu phi de forme (3, 4) et base de forme (3,), reçu {np.shape(phi_l_17)} et {np.shape(base_l_17)}.")
        raise StopCheck
    scaler_17, logreg_17 = LR_17[0], LR_17[-1]
    linear_17 = logreg_17.coef_ / scaler_17.scale_ * (GENTOO_17 - XP_TRAIN.mean(axis=0))   # w_ki (x_i - mean_i), ∂ B6.2
    if shifted_17(phi_l_17, linear_17):
        print("❌ Ex B6.17 : tes lignes sont celles d'autres classes : la ligne k doit expliquer la classe "
              "model.classes_[k] (SPECIES[k]).")
        raise StopCheck
    if same("B6.17", "logits : phi = w_ki (x_i − moyenne du fond), le cas linéaire", phi_l_17, linear_17, rtol=1e-6, atol=1e-8):
        RESULTS_17["régression logistique, logit"] = np.asarray(phi_l_17)
    phi_p_17, base_p_17 = pair("B6.17", "shap_by_class_17", shap_by_class_17(LR_17, GENTOO_17.copy(), XP_TRAIN.copy(), "proba"),
                               names=("phi", "base"))
    ok_p_17 = same("B6.17", "probabilités : efficacité de chaque classe", np.sum(phi_p_17, axis=1) + base_p_17,
                   LR_17.predict_proba(G_17)[0], rtol=1e-6, atol=1e-8)
    ok_p_17 &= same("B6.17", "probabilités : valeurs de base = probabilités moyennes sur le fond", base_p_17,
                    LR_17.predict_proba(XP_TRAIN).mean(axis=0), rtol=1e-6, atol=1e-8)
    if ok_p_17:
        RESULTS_17["régression logistique, probabilité"] = np.asarray(phi_p_17)
    phi_rf_17, base_rf_17 = pair("B6.17", "shap_by_class_17", shap_by_class_17(RF_17, GENTOO_17.copy(), XP_TRAIN.copy(), "proba"),
                                 names=("phi", "base"))
    if shap is not None:                  # the whole background: TreeExplainer keeps at most 100 rows by default
        tree_17 = shap.TreeExplainer(RF_17, data=shap.maskers.Independent(XP_TRAIN, max_samples=len(XP_TRAIN)),
                                     feature_perturbation="interventional")(G_17)
        own_17 = float(np.abs(tree_17.values[0].sum(axis=0) + tree_17.base_values[0] - RF_17.predict_proba(G_17)[0]).max())
        print(f"ℹ️ forêt : TreeSHAP s'écarte lui-même de predict_proba de {own_17:.1e} (son efficacité) ; la comparaison "
              "se fait donc à 2e-3 près.")
        if same("B6.17", "forêt : tes valeurs contre TreeExplainer (interventionnel, fond complet)", phi_rf_17,
                tree_17.values[0].T, rtol=0, atol=2e-3):
            RESULTS_17["forêt aléatoire, probabilité"] = np.asarray(phi_rf_17)
if len(RESULTS_17) == 3:
    fig_17, axes_17 = plt.subplots(1, 3, figsize=(16, 3.8))
    for ax_17, (title_17, phi_17) in zip(axes_17, RESULTS_17.items()):
        for k_17, species_17 in enumerate(SPECIES):
            ax_17.bar(np.arange(4) + 0.27 * (k_17 - 1), phi_17[k_17], width=0.27, label=species_17)
        ax_17.axhline(0, color="black", lw=0.8)
        ax_17.set_xticks(range(4), [name.replace("_mm", "").replace("_g", "") for name in PENG.features], fontsize=8)
        ax_17.set_title(title_17, fontsize=10)
        ax_17.legend(fontsize=8)
    plt.suptitle("valeurs de Shapley du premier Gentoo du test, une sortie à la fois (fond : les 249 manchots d'entraînement)",
                 fontsize=10)
    plt.tight_layout()
    plt.show()
    for title_17, phi_17 in RESULTS_17.items():
        print(f"{title_17:>36} : somme sur les classes de chaque feature "
              + ", ".join(fr(0.0 if abs(v) < 5e-5 else v, 4) for v in phi_17.sum(axis=0)))

## Partie B · Réseaux : gradients intégrés et Grad-CAM

Tu écris d'abord en NumPy les deux méthodes de ta librairie qui ne dépendent pas du réseau : les gradients intégrés à partir d'une fonction de gradient, et la combinaison Grad-CAM à partir d'activations et de gradients déjà extraits (B6.18). Puis tu les branches sur des réseaux : saillance, gradient × entrée et gradients intégrés sur un petit CNN CIFAR-10 (B6.19), Grad-CAM par hooks sur le ResNet fine-tuné du B1, pour des bonnes prédictions et des erreurs (B6.20). Au ch. 21, FGSM et la maximisation d'activation dérivaient déjà par rapport à l'entrée (🔁 R1) ; au ch. 20 et au B1, tu as posé des hooks.

La cellule suivante charge 500 images de test de CIFAR-10 (`X_CIFAR`, `Y_CIFAR` : 50 par classe, en uint8) et `CNN`, le petit CNN de référence de B6.19 : trois blocs de deux convolutions, entraîné une fois par `wbx.cifar_cnn` (≈ 40 s sur CPU la première fois, puis relu du disque), **gelé** et en mode `eval()` ; il reçoit des pixels dans $[0, 1]$, de forme `(N, 3, 32, 32)`, et se normalise lui-même : l'image noire est le tenseur nul. `show_rows` affiche une grille d'images et de cartes.

In [ ]:
# Tools of part B: CIFAR-10 and the small CNN of B6.19 (the ResNet of B6.20 is loaded there)
import torch
import torch.nn.functional as F

CLASSES = wb.datasets.CIFAR10_CLASSES
X_CIFAR, Y_CIFAR = wb.datasets.load_cifar10_subset(50, "test")   # 500 test images, uint8 (N, 32, 32, 3), int labels
CNN, CNN_INFO = wbx.cifar_cnn(FAST_MODE)   # trained once (≈ 40 s on CPU), then read back; frozen, in eval() mode


def show_rows(rows, row_titles, col_titles, cmaps, size=1.7, suptitle=None):
    """A grid of images: rows[r][c] is an image (H, W, 3) in [0, 1] or a map (H, W); cmaps[c] the colour map of the
    column c ("gray" for the images is ignored), None for an image."""
    nrow, ncol = len(rows), len(rows[0])
    fig, axes = plt.subplots(nrow, ncol, figsize=(size * ncol, size * nrow + (0.5 if suptitle else 0.1)), squeeze=False)
    for r in range(nrow):
        for c in range(ncol):
            ax = axes[r, c]
            ax.imshow(rows[r][c], cmap=cmaps[c], vmin=None if cmaps[c] is None else 0)
            ax.set_xticks([])
            ax.set_yticks([])
            if r == 0:
                ax.set_title(col_titles[c], fontsize=8)
        axes[r, 0].set_ylabel(row_titles[r], fontsize=8)
    if suptitle:
        fig.suptitle(suptitle, fontsize=10)
    plt.tight_layout()
    plt.show()


print(f"CIFAR-10 : {len(X_CIFAR)} images de test (50 par classe). Petit CNN de référence : "
      f"{sum(p.numel() for p in CNN.parameters())} paramètres, accuracy de test {pct(CNN_INFO['test_accuracy'], 1)}, "
      "entrée : des pixels dans [0, 1], (N, 3, 32, 32).")

### Ex B6.18 — integrated_gradients et grad_cam 🔨 ★★ ⏱️ 30 min
**Objectif :** écrire les gradients intégrés à partir d'une fonction de gradient, et la combinaison Grad-CAM de cartes et de gradients déjà extraits.  
**Prérequis :** ✏️ B6.3, B6.4 · fiche §206.5 (« Les gradients intégrés », « En pratique : une somme à la place de l'intégrale ») et §206.6 (« Un exemple à la main », « Avec PyTorch : deux hooks ») · **mylearn :** `explain.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** : ouvre `mon_travail/mylearn/explain.py` (créé par `python tools/start_chapter.py B6`), lis la docstring, remplace chaque `raise NotImplementedError(...)` de l'exercice par ton code et **enregistre**. NumPy seulement. La première cellule de vérification recharge ta librairie et l'essaie sur des cas visibles ; la seconde lance les tests de l'exercice. `python -m pytest tests/test_chb6_explain.py -q` les lance tous dans un terminal.

**Contrat** de `integrated_gradients(grad_fn, x, baseline=None, n_steps=50, method="trapezoid")` :
- les points du chemin sont $\mathbf{x}' + \alpha(\mathbf{x} - \mathbf{x}')$, $\mathbf{x}'$ la ligne de référence `baseline` (des zéros si `None`) ; la moyenne pondérée des gradients est multipliée par $\mathbf{x} - \mathbf{x}'$, et le résultat a la forme de `x`, en float, même pour un `x` entier ;
- `n_steps` compte les **intervalles**, un entier ≥ 1 : `'left'`, `'right'` et `'midpoint'` utilisent $n$ points de poids $1/n$, `'trapezoid'` les $n + 1$ points $k/n$ de poids $1/n$, divisés par deux aux extrémités (la docstring) ;
- `grad_fn` ne reçoit que des **batches** de forme `(n, *x.shape)`, même avec `n_steps = 1` (pour un seul point : `point[None]`, puis `[0]` sur le résultat) ; elle peut être appelée une fois ou sur plusieurs morceaux ;
- `ValueError` si `n_steps < 1`, si `method` est inconnue (les noms de captum, comme `'riemann_trapezoid'`, sont refusés), ou si `baseline` n'a pas **exactement** la forme de `x` (une forme seulement diffusable est refusée) ; `x` et `baseline` ne sont pas modifiés ;
- ⚠️ la note de la docstring sur captum est inexacte : son `riemann_trapezoid` à $n + 1$ points vaut ton `'trapezoid'` **multiplié par $n/(n+1)$**, parce que ses poids somment à moins de 1 (fiche, 🕰️ de 206.6 ; ✏️ B6.3 k) ; `'left'`, `'right'` et `'midpoint'` coïncident avec `riemann_left`, `riemann_right` et `riemann_middle` au même nombre de points.

**Contrat** de `grad_cam(activations, gradients, relu=True, normalize=True)` :
- les deux entrées ont exactement la même forme 4-D `(N, C, H, W)`, sinon `ValueError` ; le résultat a la forme `(N, H, W)` ;
- le poids d'un canal est la **moyenne** spatiale de ses gradients ; la carte est la somme pondérée des canaux, la ReLU appliquée **une fois**, à cette somme (`relu=True`) ;
- `normalize=True` divise **chaque** carte par sa propre plus grande valeur absolue, sans epsilon ; une carte nulle, y compris après la ReLU, reste nulle (pas de NaN) ;
- les entrées ne sont pas modifiées.

La première cellule de vérification essaie `integrated_gradients` sur les exemples de la docstring, sur $x_0 x_1 + x_2^3$ en $(1, 2, -1)$ avec les quatre règles et 6 intervalles (formes fermées), sur ce que reçoit `grad_fn`, sur la complétude de la sigmoïde de 206.5, un `x` entier et trois refus ; la seconde partie essaie `grad_cam` sur la docstring, sur trois images de 4 canaux (contre `np.einsum`), l'invariance à un facteur positif des gradients, `relu=False`, des gradients nuls et deux refus. La seconde cellule lance les 111 tests de l'exercice (`-k "test_integrated_gradients_ or test_grad_cam_"`) ; sans captum, 12 sont ignorés.

Dans tes notes : avec 6 intervalles, quelle règle donne la valeur exacte pour $x_0 x_1$, et pourquoi pas pour $x_2^3$ ? Pourquoi la complétude sert-elle de test à une implémentation ? Pourquoi `grad_cam` ne fait-elle que la combinaison, et pas l'extraction (qui reste côté PyTorch) ?

In [ ]:
with checked("B6.18"):
    E = mylearn.explain
    grad_q_18 = lambda X: np.stack([2 * X[:, 0], np.full(len(X), 3.0)], axis=1)       # f = x0**2 + 3 x1 (docstring)
    same("B6.18", "docstring : x0² + 3 x1 en (1, 1), n_steps=4, 'left'",
         E.integrated_gradients(grad_q_18, [1.0, 1.0], n_steps=4, method="left"), [0.75, 3.0])
    same("B6.18", "docstring : la même avec 'trapezoid'", E.integrated_gradients(grad_q_18, [1.0, 1.0], n_steps=4), [1.0, 3.0])

    # f = x0 x1 + x2**3 at x = (1, 2, -1) from a zero baseline: on the path a x, the gradient is (a x1, a x0, 3 a² x2²)
    grad_c_18 = lambda X: np.stack([X[:, 1], X[:, 0], 3 * X[:, 2] ** 2], axis=1)
    x_18 = np.array([1.0, 2.0, -1.0])
    k_18 = np.arange(7) / 6                                       # n = 6 intervals
    means_18 = {"left": (k_18[:-1].mean(), (k_18[:-1] ** 2).mean()), "right": (k_18[1:].mean(), (k_18[1:] ** 2).mean()),
                "midpoint": (((np.arange(6) + 0.5) / 6).mean(), (((np.arange(6) + 0.5) / 6) ** 2).mean()),
                "trapezoid": (0.5, 1 / 3 + 1 / (6 * 36))}         # the means of a and a² for each rule
    for method_18, (m1_18, m2_18) in means_18.items():
        want_18 = np.array([x_18[0] * x_18[1] * m1_18, x_18[1] * x_18[0] * m1_18, x_18[2] * 3 * x_18[2] ** 2 * m2_18])
        same("B6.18", f"x0 x1 + x2³ en (1, 2, −1), n_steps=6, '{method_18}'",
             E.integrated_gradients(grad_c_18, x_18.copy(), n_steps=6, method=method_18), want_18)
    shapes_18 = []

    def spy_18(X):
        shapes_18.append(np.shape(X))
        return grad_c_18(np.asarray(X, dtype=float))

    E.integrated_gradients(spy_18, x_18, n_steps=1, method="left")
    verdict("B6.18", shapes_18 and all(len(s) == 2 and s[1] == 3 for s in shapes_18),
            "grad_fn ne reçoit que des batches (n, 3), même avec n_steps=1.",
            f"grad_fn a reçu les formes {shapes_18} : elle attend un batch (n, *x.shape), même pour un seul point.")
    sig_18 = lambda X: (lambda s: np.stack([s * (1 - s) * X[:, 1], s * (1 - s) * X[:, 0]], axis=1))(
        1 / (1 + np.exp(-(X[:, 0] * X[:, 1] - 2))))                # the sigmoid of 206.5: sigma(x0 x1 - 2)
    gap_18 = 1 / (1 + np.exp(-4.0)) - 1 / (1 + np.exp(2.0))
    total_18 = float(np.sum(E.integrated_gradients(sig_18, [2.0, 3.0], n_steps=200)))
    verdict("B6.18", abs(total_18 - gap_18) < 1e-4, f"complétude sur la sigmoïde de 206.5 (200 intervalles) : somme {fr(total_18, 5)}.",
            f"complétude sur la sigmoïde de 206.5 : la somme vaut {total_18:.5g}, attendu f(x) − f(x′) ≈ {gap_18:.5g}.")
    ig_int_18 = E.integrated_gradients(grad_q_18, np.array([1, 1]), n_steps=4)
    verdict("B6.18", np.asarray(ig_int_18).dtype.kind == "f", "un x entier donne des attributions en float.",
            f"un x entier doit donner des attributions en float, reçu {np.asarray(ig_int_18).dtype}.")
    refuses("B6.18", "n_steps=0", lambda: E.integrated_gradients(grad_q_18, [1.0, 1.0], n_steps=0))
    refuses("B6.18", "method='riemann_trapezoid' (un nom de captum)",
            lambda: E.integrated_gradients(grad_q_18, [1.0, 1.0], method="riemann_trapezoid"))
    refuses("B6.18", "une ligne de référence de forme (1,) pour x de forme (2,)",
            lambda: E.integrated_gradients(grad_q_18, [1.0, 1.0], baseline=[0.0]))

with checked("B6.18"):
    E = mylearn.explain
    A_18 = np.array([[[[1.0, 2.0], [0.0, 1.0]], [[0.0, 1.0], [3.0, 0.0]]]])           # the docstring: N=1, C=2, 2 x 2
    G_18 = np.array([[[[1.0, 1.0], [1.0, 1.0]], [[-1.0, -1.0], [-1.0, -1.0]]]])
    copies_18 = (A_18.copy(), G_18.copy())
    same("B6.18", "docstring : grad_cam(A, G, relu=False, normalize=False)", E.grad_cam(A_18, G_18, relu=False, normalize=False),
         [[[1.0, 1.0], [-3.0, 1.0]]])
    same("B6.18", "docstring : grad_cam(A, G)", E.grad_cam(A_18, G_18), [[[1.0, 1.0], [0.0, 1.0]]])
    verdict("B6.18", np.array_equal(A_18, copies_18[0]) and np.array_equal(G_18, copies_18[1]),
            "les activations et les gradients reçus n'ont pas changé.", "grad_cam a modifié ses entrées.")
    rng_18 = np.random.default_rng(18)
    A2_18, G2_18 = rng_18.random((3, 4, 5, 5)), rng_18.normal(size=(3, 4, 5, 5))
    want_raw_18 = np.einsum("nc,nchw->nhw", G2_18.mean(axis=(2, 3)), A2_18)
    want_18 = np.maximum(want_raw_18, 0)
    peak_18 = want_18.max(axis=(1, 2), keepdims=True)
    same("B6.18", "3 images de 4 canaux 5 × 5 : ReLU de la somme pondérée, chaque carte ÷ son maximum",
         E.grad_cam(A2_18, G2_18), np.divide(want_18, peak_18, out=np.zeros_like(want_18), where=peak_18 > 0))
    same("B6.18", "les mêmes, gradients multipliés par 7 (même carte)", E.grad_cam(A2_18, 7 * G2_18),
         np.divide(want_18, peak_18, out=np.zeros_like(want_18), where=peak_18 > 0))
    same("B6.18", "relu=False : dans [−1, 1], ÷ le plus grand |.|", E.grad_cam(A2_18, G2_18, relu=False),
         want_raw_18 / np.abs(want_raw_18).max(axis=(1, 2), keepdims=True))
    same("B6.18", "gradients nuls : une carte nulle, sans NaN", E.grad_cam(A2_18, np.zeros_like(G2_18)), np.zeros((3, 5, 5)))
    refuses("B6.18", "activations et gradients de formes différentes", lambda: E.grad_cam(A2_18, G2_18[:, :3]))
    refuses("B6.18", "des tableaux 3-D", lambda: E.grad_cam(A2_18[0], G2_18[0]))

In [ ]:
if written(lambda: (mylearn.explain.integrated_gradients(lambda X: X, [1.0], n_steps=1), mylearn.explain.grad_cam(np.ones((1, 1, 1, 1)), np.ones((1, 1, 1, 1))))):
    run_pytest_group(TEST_EXPLAIN, impl="ref", keyword="test_integrated_gradients_ or test_grad_cam_")   # 111 tests, dont 12 ignorés sans captum
else:
    print("⏳ Ex B6.18 : les tests attendent tes fonctions.")

### Ex B6.19 — Saillance, gradient × entrée et gradients intégrés sur un CNN CIFAR-10 📦 ★★ ⏱️ 30 min
**Objectif :** calculer trois attributions pixel à pixel sur un CNN, vérifier la complétude et comparer deux lignes de référence.  
**Prérequis :** Ex B6.18 · ch. 24 · fiche §206.5 (« Le gradient comme explication », « Les gradients intégrés », « Le choix de la ligne de référence ») · **Fil rouge :** CIFAR-10 · **Parcours :** C

La cellule donnée prend 8 images de test (`IMAGES_19`, un tenseur `(8, 3, 32, 32)` dans $[0, 1]$), leurs classes (`LABELS_19`), les classes que prédit le CNN (`PRED_19`) et `blurred_19`, qui floute une image (la moyenne sur une fenêtre de 7 × 7 pixels). **Contrat** :
- `class_gradient_19(model, images, target)` rend un tableau NumPy `(N, 3, 32, 32)` : le gradient du **logit** (le score avant la softmax) de la classe `target[n]` de chaque image par rapport à ses pixels ; `images` est un tenseur ou un tableau `(N, 3, 32, 32)`, `target` une classe par image, pas forcément la prédite ; le modèle est gelé et en mode `eval()`, et doit le rester ; deux appels successifs rendent le même gradient ;
- `saliency_19(gradients)` rend la carte de saillance `(N, 32, 32)` : le plus grand $\lvert$gradient$\rvert$ des trois canaux de chaque pixel ;
- `grad_times_input_19(gradients, images)` rend `(N, 32, 32)` : gradient × entrée, sommé sur les trois canaux de chaque pixel, avec son signe ;
- `integrated_19(model, image, target, baseline, n_steps=50)` rend `(3, 32, 32)` : les gradients intégrés du logit de la classe `target` (un entier) pour **une** image `(3, 32, 32)`, depuis la ligne de référence `baseline` `(3, 32, 32)`, calculés par **ta** `integrated_gradients` (règle `'trapezoid'`) avec une `grad_fn` construite sur `class_gradient_19`.

La cellule de vérification compare chaque fonction à une référence (l'autograd de `wbx`) sur quatre images, avec des classes que le CNN ne prédit pas. L'expérience trace, pour 6 images et la vraie classe, la saillance, $\lvert$gradient × entrée$\rvert$, et $\lvert$IG$\rvert$ depuis l'image noire et depuis l'image floutée (la somme des valeurs absolues des trois canaux), puis compare la somme des attributions à $f(\mathbf{x}) - f(\mathbf{x}')$.

Dans tes notes : laquelle des trois cartes est la plus bruitée, et laquelle se concentre le plus sur l'objet ? La complétude tient-elle pour les gradients intégrés, et pour gradient × entrée ? Avec 50 intervalles, à combien près ? Que change la ligne de référence floutée : que vaut $f(\mathbf{x}')$, quels pixels reçoivent une attribution presque nulle, et pourquoi (le facteur $x_i - x'_i$) ? Quelle référence choisirais-tu pour un objet noir sur fond clair ?

In [ ]:
IMAGES_19 = wbx.pixels(X_CIFAR[:8])               # 8 test images, float32 tensor (8, 3, 32, 32) in [0, 1]
LABELS_19 = Y_CIFAR[:8]
PRED_19 = wbx.predict(CNN, IMAGES_19).argmax(axis=1)  # the classes the CNN predicts (computed without your code)


def blurred_19(x):
    """A blurred copy of images (N, 3, 32, 32): the mean over a 7 x 7 window around each pixel."""
    return F.avg_pool2d(torch.as_tensor(x), 7, stride=1, padding=3, count_include_pad=False)


print("classes vraies :", [CLASSES[k] for k in LABELS_19])
print("classes prédites :", [CLASSES[k] for k in PRED_19])

In [ ]:
def class_gradient_19(model, images, target):
    """np.ndarray (N, 3, 32, 32): the gradient of the logit of the class target[n] of each image with respect to its
    pixels. images: a float tensor or array (N, 3, 32, 32) in [0, 1]; target: one class per image (N ints)."""
    x = torch.as_tensor(np.asarray(images), dtype=torch.float32).clone().requires_grad_(True)
    t = torch.as_tensor(np.asarray(target)).long().reshape(-1)
    scores = model(x).gather(1, t[:, None])          # (N, 1): the logit of the class of each image
    scores.sum().backward()                          # each logit depends on its own image only
    return x.grad.numpy()


def saliency_19(gradients):
    """(N, 32, 32): the saliency map, the largest |gradient| over the 3 channels of each pixel."""
    return np.abs(np.asarray(gradients)).max(axis=1)


def grad_times_input_19(gradients, images):
    """(N, 32, 32): gradient × input, summed over the 3 channels of each pixel (signed)."""
    return (np.asarray(gradients) * np.asarray(images)).sum(axis=1)


def integrated_19(model, image, target, baseline, n_steps=50):
    """(3, 32, 32): the integrated gradients of the logit of the class `target` (an int) for ONE image (3, 32, 32),
    from `baseline` (3, 32, 32), with your integrated_gradients (rule 'trapezoid') and a grad_fn built on
    class_gradient_19."""
    grad_fn = lambda points: class_gradient_19(model, points, [target] * len(points))
    return mylearn.explain.integrated_gradients(grad_fn, np.asarray(image, dtype=float), np.asarray(baseline, dtype=float),
                                                n_steps=n_steps, method="trapezoid")


ok_19 = {}
with checked("B6.19"):
    x_19 = IMAGES_19[:4]
    t_19 = np.array([(p + 3) % 10 for p in PRED_19[:4]])     # classes that the CNN does NOT predict
    want_19 = wbx.input_gradients(CNN, x_19, t_19)
    g_19 = class_gradient_19(CNN, x_19.clone(), t_19.copy())
    if CNN.training or any(p.requires_grad for p in CNN.parameters()):
        print("❌ Ex B6.19 : après class_gradient_19, le modèle n'est plus gelé ou n'est plus en mode eval() (ses "
              "batchnorms changent alors le calcul) : ta fonction ne doit pas le changer.")
        raise StopCheck
    if not returned("B6.19", "class_gradient_19", g_19):
        raise StopCheck
    arr_19 = np.asarray(g_19.detach() if torch.is_tensor(g_19) else g_19, dtype=float)
    if arr_19.shape != want_19.shape:
        print(f"❌ Ex B6.19 : class_gradient_19 doit rendre un tableau (4, 3, 32, 32), reçu la forme {arr_19.shape}.")
        raise StopCheck
    if not np.allclose(arr_19, want_19, rtol=1e-4, atol=1e-6):
        probe_19 = x_19.clone().requires_grad_(True)
        probs_19 = torch.softmax(CNN(probe_19), dim=1)
        (g_prob_19,) = torch.autograd.grad(probs_19.gather(1, torch.as_tensor(t_19)[:, None]).sum(), probe_19)
        if np.allclose(arr_19, wbx.input_gradients(CNN, x_19, PRED_19[:4]), rtol=1e-4, atol=1e-6):
            print("❌ Ex B6.19 : c'est le gradient de la classe prédite : le contrat veut celui de target[n], qui n'est "
                  "pas forcément la prédite.")
        elif np.allclose(arr_19, g_prob_19.numpy(), rtol=1e-4, atol=1e-6):
            print("❌ Ex B6.19 : c'est le gradient de la probabilité (après la softmax) : le contrat veut celui du logit.")
        elif np.allclose(arr_19, sum(wbx.input_gradients(CNN, x_19, k) for k in range(10)), rtol=1e-4, atol=1e-5):
            print("❌ Ex B6.19 : c'est le gradient de la somme des 10 logits : ne garde que celui de target[n].")
        else:
            same("B6.19", "class_gradient_19 contre l'autograd de référence", arr_19, want_19, rtol=1e-4, atol=1e-6)
        raise StopCheck
    print("✅ Ex B6.19 : class_gradient_19 : le gradient du logit de la classe demandée, image par image.")
    again_19 = np.asarray(class_gradient_19(CNN, x_19.clone(), t_19.copy()), dtype=float)
    verdict("B6.19", np.allclose(again_19, arr_19, rtol=1e-6, atol=1e-8), "un second appel rend le même gradient (rien ne s'accumule).",
            "un second appel ne rend pas le même gradient : un .grad s'accumule-t-il d'un appel à l'autre ?")
    print("✅ Ex B6.19 : le modèle reste gelé, en mode eval().")
    ok_19["grad"] = True
CNN.eval().requires_grad_(False)                       # restored for the cells that follow, whatever your code did

with checked("B6.19"):
    if not ok_19:
        raise StopCheck
    s_19 = saliency_19(want_19.copy())
    arr_s_19 = as_numpy(s_19)
    if arr_s_19 is not None and arr_s_19.shape == (4, 32, 32) and np.allclose(arr_s_19, np.abs(want_19).mean(axis=1)):
        print("❌ Ex B6.19 : c'est la moyenne des |gradients| des canaux : la carte de saillance de 206.5 prend le maximum.")
    elif same("B6.19", "saliency_19 : le plus grand |gradient| des 3 canaux", s_19, np.abs(want_19).max(axis=1)):
        ok_19["saliency"] = True
    gi_19 = grad_times_input_19(want_19.copy(), x_19.numpy().copy())
    if same("B6.19", "grad_times_input_19 : somme sur les canaux de gradient × pixel", gi_19, (want_19 * x_19.numpy()).sum(axis=1),
            rtol=1e-6, atol=1e-8):
        ok_19["gxi"] = True

with checked("B6.19"):
    if not ok_19.get("grad"):
        raise StopCheck
    ig_19 = integrated_19(CNN, IMAGES_19[0].numpy(), int(LABELS_19[0]), np.zeros((3, 32, 32)), n_steps=20)
    if not returned("B6.19", "integrated_19", ig_19):
        raise StopCheck
    ref_ig_19 = wbx.integrated_gradients(CNN, IMAGES_19[:1], [int(LABELS_19[0])], n_steps=20)[0]
    arr_ig_19 = as_numpy(ig_19)
    if (arr_ig_19 is not None and arr_ig_19.shape == ref_ig_19.shape and not np.allclose(arr_ig_19, ref_ig_19, rtol=1e-4, atol=1e-6)
            and np.allclose(arr_ig_19 * IMAGES_19[0].numpy(), ref_ig_19, rtol=1e-4, atol=1e-6)):
        print("❌ Ex B6.19 : integrated_19 rend la moyenne des gradients sur le chemin : les gradients intégrés la "
              "multiplient par (x − x′).")
    elif same("B6.19", "integrated_19 (image noire, 20 intervalles) contre la référence", ig_19, ref_ig_19, rtol=1e-4, atol=1e-6):
        ok_19["ig"] = True
    else:
        print("ℹ️ Ex B6.19 : la ligne de référence est ici l'image noire, le tenseur nul (le CNN reçoit des pixels dans "
              "[0, 1] et se normalise lui-même), et la règle celle des trapèzes, en 20 intervalles.")

In [ ]:
if len(ok_19) == 4:
    start_19 = time.perf_counter()
    black_19, blur_19 = torch.zeros_like(IMAGES_19), blurred_19(IMAGES_19)
    grads_19 = class_gradient_19(CNN, IMAGES_19, LABELS_19)
    sal_19, gxi_19 = saliency_19(grads_19), grad_times_input_19(grads_19, IMAGES_19.numpy())
    ig_black_19 = np.stack([integrated_19(CNN, IMAGES_19[i].numpy(), int(LABELS_19[i]), black_19[i].numpy())
                            for i in range(6)])
    ig_blur_19 = np.stack([integrated_19(CNN, IMAGES_19[i].numpy(), int(LABELS_19[i]), blur_19[i].numpy())
                           for i in range(6)])
    rows_19 = []
    for i in range(6):
        rows_19.append([IMAGES_19[i].permute(1, 2, 0).numpy(), sal_19[i], np.abs(gxi_19[i]),
                        np.abs(ig_black_19[i]).sum(axis=0), blur_19[i].permute(1, 2, 0).numpy(),
                        np.abs(ig_blur_19[i]).sum(axis=0)])
    show_rows(rows_19, [f"{CLASSES[LABELS_19[i]]} (prédit {CLASSES[PRED_19[i]]})" for i in range(6)],
              ["image", "saillance", "|gradient × entrée|", "|IG|, réf. noire", "réf. floutée", "|IG|, réf. floutée"],
              [None, "magma", "magma", "magma", None, "magma"],
              suptitle="attributions du logit de la vraie classe (IG : somme des |attributions| sur les 3 canaux)")
    with torch.no_grad():
        logit_19 = lambda x: CNN(x).gather(1, torch.as_tensor(LABELS_19[:6])[:, None])[:, 0].double().numpy()
        f_x_19, f_black_19, f_blur_19 = logit_19(IMAGES_19[:6]), logit_19(black_19[:6]), logit_19(blur_19[:6])
    print("complétude (somme des attributions contre f(x) − f(x′), logit de la vraie classe) :")
    for i in range(6):
        print(f"  {CLASSES[LABELS_19[i]]:>10} : réf. noire {fr(ig_black_19[i].sum(), 3)} contre {fr(f_x_19[i] - f_black_19[i], 3)} ; "
              f"réf. floutée {fr(ig_blur_19[i].sum(), 3)} contre {fr(f_x_19[i] - f_blur_19[i], 3)} ; "
              f"somme de gradient × entrée {fr(gxi_19[i].sum(), 3)}")
    print(f"ℹ️ {time.perf_counter() - start_19:.1f} s")
else:
    print("⏳ Ex B6.19 : l'expérience attend tes quatre fonctions (cellule précédente).")

### Ex B6.20 — Grad-CAM par hooks sur le ResNet fine-tuné du B1 : bonnes prédictions et erreurs 📦 ★★★ ⏱️ 45 min
**Objectif :** extraire activations et gradients par deux hooks, construire les cartes Grad-CAM d'un ResNet et les lire sur des bonnes prédictions et des erreurs.  
**Prérequis :** Ex B6.18 · B1 (B1.17, B1.27) · fiche §206.6 (« Avec PyTorch : deux hooks ») · ch. 20 (hooks) · **Fil rouge :** CIFAR-10 · **Parcours :** R, C

Le modèle est **ton** ResNet-18 du B1 si tu l'as publié en B1.27 : `wbx.find_b1_folder()` cherche le dossier `resnet18_cifar10_b1` (avec `model.safetensors` et `config.json`) à côté de tes notebooks de `mon_travail/` ; sinon, ou si tu mets `B1_FOLDER = None`, c'est un ResNet-18 de référence, fine-tuné une fois par la recette de B1.17 (`layer4` et une nouvelle tête, 100 images par classe à 96 pixels, 3 epochs : ≈ 35 s sur CPU la première fois, puis relu du disque). La cellule de chargement dit lequel est utilisé. Le modèle est **gelé** (`requires_grad=False` partout) et en mode `eval()`, comme un backbone du B1 ; il attend des images préparées par `wbx.model_input` (agrandies à `SIZE_20` pixels, normalisées par les statistiques d'ImageNet), et `layer4` y est en `SIZE_20 // 32` × `SIZE_20 // 32` (3 × 3 à 96 pixels). La cellule de chargement choisit aussi, parmi 200 images de test, 8 bonnes prédictions et 8 erreurs (`X_20`, 16 images préparées).

**Contrat** :
- `activations_and_gradients_20(model, layer, images, target)` rend le couple `(A, G)` de tableaux NumPy `(N, C, H, W)` : `A`, la sortie du module `layer` pour le batch préparé `images`, et `G`, le gradient de la somme, sur les images, du logit de `target[n]` par rapport à `A` (une classe par image, pas forcément la prédite) ; deux hooks, l'un sur le module (passe avant), l'autre sur son tenseur de sortie ; le hook de passe avant ne rend rien, et il est retiré avant de rendre la main, même en cas d'erreur ;
- `cam_20(model, layer, images, target, size)` rend `(N, size, size)` : les cartes de **ta** `grad_cam` (ReLU et normalisation par défaut), agrandies par `F.interpolate(..., mode="bilinear", align_corners=False)`, la convention qui a agrandi les images.

La cellule de vérification compare `A`, `G` et les cartes à une référence (`wbx`) sur 4 images, dont une avec une autre classe que la vraie, et vérifie qu'aucun hook ne reste posé. L'expérience trace la carte de la classe prédite sur les 8 bonnes prédictions, puis, sur les 8 erreurs, celle de la classe prédite et celle de la vraie classe, et affiche la corrélation entre ces deux cartes.

Dans tes notes : sur les bonnes prédictions, la carte tombe-t-elle sur l'objet, sur le fond, sur un bord ? Sur les erreurs, les cartes de la classe prédite et de la vraie classe regardent-elles au même endroit, et qu'est-ce que cela suggère de l'erreur (un objet ambigu, un contexte trompeur) ? Que peut-on localiser avec une carte de 3 × 3 sur une image de 96 pixels ? Que verrais-tu avec `layer3`, et pourquoi Grad-CAM vise-t-il la dernière couche convolutive (🧠 Q8) ? Une carte qui « tombe sur l'objet » prouve-t-elle que le modèle a raison pour de bonnes raisons (206.7) ?

In [ ]:
B1_FOLDER = None   # the solutions notebook never reads the learner's model: the reference ResNet


def activations_and_gradients_20(model, layer, images, target):
    """(A, G): np.ndarray (N, C, H, W) twice, the output A of `layer` (a module of model) for the prepared batch
    `images`, and the gradient G of the sum over the images of the logit of target[n] with respect to A; two hooks
    (one on the module, one on its output tensor), removed before returning."""
    store = {}

    def save(module, inputs, output):                 # returns None: the output is left unchanged
        store["A"] = output.detach()
        output.register_hook(lambda grad: store.__setitem__("G", grad.detach()))

    x = torch.as_tensor(images, dtype=torch.float32).clone().requires_grad_(True)   # frozen model: the input needs grad
    t = torch.as_tensor(np.asarray(target)).long().reshape(-1)
    handle = layer.register_forward_hook(save)
    try:
        scores = model(x)
        scores.gather(1, t[:, None]).sum().backward()
    finally:
        handle.remove()
    return store["A"].numpy(), store["G"].numpy()


def cam_20(model, layer, images, target, size):
    """(N, size, size): the Grad-CAM maps of your grad_cam (relu and normalize by default), resized to size x size by
    F.interpolate(..., mode="bilinear", align_corners=False)."""
    A, G = activations_and_gradients_20(model, layer, images, target)
    cams = torch.as_tensor(mylearn.explain.grad_cam(A, G), dtype=torch.float32)
    return F.interpolate(cams[:, None], size=(size, size), mode="bilinear", align_corners=False)[:, 0].numpy()

In [ ]:
RESNET, RESNET_INFO = wbx.b1_resnet(FAST_MODE, folder=B1_FOLDER)   # frozen, in eval() mode
SIZE_20 = RESNET_INFO["size"]                                          # the side of its input images (96 in fast mode)
POOL_20 = X_CIFAR[:200], Y_CIFAR[:200]                                 # 200 test images (20 per class)
PRED_20 = wbx.predict(RESNET, POOL_20[0], RESNET_INFO).argmax(axis=1)
RIGHT_20 = np.flatnonzero(PRED_20 == POOL_20[1])[:8]                   # 8 well-classified images
WRONG_20 = np.flatnonzero(PRED_20 != POOL_20[1])[:8]                   # 8 errors
X_20 = wbx.model_input(POOL_20[0][np.concatenate([RIGHT_20, WRONG_20])], RESNET_INFO)   # (16, 3, SIZE_20, SIZE_20)
print(f"ResNet-18 {'de référence' if RESNET_INFO['source'] == 'reference' else 'du B1 (le tien)'} : {SIZE_20} pixels, layer4 en {SIZE_20 // 32} × {SIZE_20 // 32} ; accuracy sur "
      f"les 200 images : {pct(np.mean(PRED_20 == POOL_20[1]), 1)} ; 8 bonnes prédictions et 8 erreurs retenues.")

In [ ]:
ok_20 = {}
wbx.clear_hooks(RESNET)                                # hooks left on RESNET by an earlier run of your code
with checked("B6.20"):
    x_20 = X_20[[0, 1, 8, 9]]
    true_20 = POOL_20[1][np.concatenate([RIGHT_20, WRONG_20])][[0, 1, 8, 9]]
    t_20 = np.array([true_20[0], (true_20[1] + 1) % 10, true_20[2], true_20[3]])      # true classes, and one other
    A_ref_20, G_ref_20, _ = wbx.activations_and_gradients(RESNET, RESNET.layer4, x_20, t_20)
    try:
        got_20 = activations_and_gradients_20(RESNET, RESNET.layer4, x_20.clone(), t_20.copy())
    except RuntimeError as exc_20:
        if "require" in str(exc_20) and "grad" in str(exc_20):
            print("❌ Ex B6.20 : ton code lève RuntimeError : " + str(exc_20)[:160] + " ; deux causes possibles : les "
                  "poids du modèle sont gelés (requires_grad=False), relis le dernier détail de « Avec PyTorch : deux "
                  "hooks » (§206.6) ; ou ton hook de passe avant rend une valeur, qui remplace la sortie du module, et "
                  "un tenseur détaché coupe le graphe (le hook ne rend rien, 206.6).")
            raise StopCheck
        raise
    A_20, G_20 = pair("B6.20", "activations_and_gradients_20", got_20, names=("A", "G"))
    left_20 = wbx.leftover_hooks(RESNET)
    if left_20:
        print("❌ Ex B6.20 : un hook reste posé sur RESNET après l'appel ("
              + ", ".join(f"{name or 'le modèle'} : {', '.join(kinds)}" for name, kinds in left_20.items())
              + ") : retire chaque hook de module (handle.remove(), dans un finally) ; sinon chaque passe suivante le "
              "déclenche encore.")
        wbx.clear_hooks(RESNET)
        raise StopCheck
    if same("B6.20", "A : la sortie de layer4", A_20, A_ref_20, rtol=1e-5, atol=1e-6):
        ok_20["A"] = True
    if np.shape(G_20) == G_ref_20.shape and not np.allclose(np.asarray(G_20, dtype=float), G_ref_20, rtol=1e-4, atol=1e-7):
        pred_t_20 = wbx.predict(RESNET, x_20).argmax(axis=1)
        _, G_pred_20, _ = wbx.activations_and_gradients(RESNET, RESNET.layer4, x_20, pred_t_20)
        if np.allclose(np.asarray(G_20, dtype=float), G_pred_20, rtol=1e-4, atol=1e-7):
            print("❌ Ex B6.20 : G est le gradient de la classe prédite : le contrat veut celui de target[n] "
                  "(l'image 2 reçoit une autre classe).")
            raise StopCheck
    if same("B6.20", "G : le gradient du logit de target[n] par rapport à A", G_20, G_ref_20, rtol=1e-4, atol=1e-7):
        ok_20["G"] = True
RESNET.eval().requires_grad_(False)                    # restored for the cells that follow, whatever your code did
wbx.clear_hooks(RESNET)

with checked("B6.20"):
    if len(ok_20) < 2:
        raise StopCheck
    cams_20 = cam_20(RESNET, RESNET.layer4, x_20.clone(), t_20.copy(), SIZE_20)
    want_cam_20 = wbx.grad_cam(RESNET, RESNET.layer4, x_20, t_20, size=SIZE_20)
    arr_cam_20 = np.asarray(cams_20, dtype=float)
    if arr_cam_20.shape == want_cam_20.shape and not np.allclose(arr_cam_20, want_cam_20, rtol=1e-4, atol=1e-5):
        small_20 = torch.as_tensor(wbx.grad_cam(RESNET, RESNET.layer4, x_20, t_20))[:, None].float()
        corners_20 = F.interpolate(small_20, size=(SIZE_20, SIZE_20), mode="bilinear", align_corners=True)[:, 0].numpy()
        if np.allclose(arr_cam_20, corners_20, rtol=1e-4, atol=1e-5):
            print("❌ Ex B6.20 : la carte est agrandie avec align_corners=True : les images le sont avec "
                  "align_corners=False (wb.transfer.to_input) ; avec deux conventions, la carte est décalée sur l'image.")
            raise StopCheck
        if arr_cam_20.min() < -1e-9:
            print("❌ Ex B6.20 : la carte a des valeurs négatives : grad_cam garde-t-il sa ReLU (relu=True par défaut) ?")
            raise StopCheck
        nearest_20 = F.interpolate(small_20, size=(SIZE_20, SIZE_20), mode="nearest")[:, 0].numpy()
        if np.allclose(arr_cam_20, nearest_20, rtol=1e-4, atol=1e-5):
            print("❌ Ex B6.20 : la carte est agrandie par plus proche voisin (des blocs) : le contrat veut "
                  "mode=\"bilinear\", align_corners=False, la convention qui a agrandi les images.")
            raise StopCheck
        wbx.clear_hooks(RESNET)
        if np.allclose(arr_cam_20, wbx.grad_cam(RESNET, RESNET.layer3, x_20, t_20, size=SIZE_20), rtol=1e-4, atol=1e-5):
            print("❌ Ex B6.20 : ce sont les cartes de layer3 : cam_20 doit pondérer les activations du module `layer` "
                  "qu'elle reçoit.")
            raise StopCheck
    if same("B6.20", f"cam_20 : cartes agrandies à {SIZE_20} × {SIZE_20}", arr_cam_20, want_cam_20, rtol=1e-4, atol=1e-5):
        ok_20["cam"] = True
wbx.clear_hooks(RESNET)

In [ ]:
if len(ok_20) == 3:
    labels_20 = POOL_20[1][np.concatenate([RIGHT_20, WRONG_20])]
    preds_20 = PRED_20[np.concatenate([RIGHT_20, WRONG_20])]
    cam_pred_20 = cam_20(RESNET, RESNET.layer4, X_20, preds_20, SIZE_20)        # the predicted class
    cam_true_20 = cam_20(RESNET, RESNET.layer4, X_20, labels_20, SIZE_20)       # the true class (the same when right)
    shown_20 = wbx.model_input(POOL_20[0][np.concatenate([RIGHT_20, WRONG_20])], {"size": SIZE_20, "mean": (0, 0, 0),
                                                                                  "std": (1, 1, 1)})
    pics_20 = shown_20.permute(0, 2, 3, 1).clamp(0, 1).numpy()                  # the images at SIZE_20 pixels, in [0, 1]

    def overlay_20(image, cam):
        heat = plt.cm.jet(cam)[..., :3]
        return np.clip(0.55 * image + 0.45 * heat, 0, 1)

    def corr_20(a, b):
        """The correlation of two maps, or None when one of them is constant (all zero after the ReLU)."""
        a, b = a.ravel(), b.ravel()
        return None if a.std() == 0 or b.std() == 0 else float(np.corrcoef(a, b)[0, 1])

    rows_20 = [[pics_20[i], overlay_20(pics_20[i], cam_pred_20[i])] for i in range(8)]
    show_rows(rows_20, [f"{CLASSES[labels_20[i]]}" for i in range(8)], ["image", "Grad-CAM (prédite)"],
              [None, None], size=1.6, suptitle="bonnes prédictions : Grad-CAM de la classe prédite (layer4)")
    rows_w_20 = [[pics_20[i], overlay_20(pics_20[i], cam_pred_20[i]), overlay_20(pics_20[i], cam_true_20[i])]
                 for i in range(8, 16)]
    show_rows(rows_w_20, [f"{CLASSES[labels_20[i]]} → {CLASSES[preds_20[i]]}" for i in range(8, 16)],
              ["image", "classe prédite", "vraie classe"], [None, None, None], size=1.6,
              suptitle="erreurs (vraie → prédite) : Grad-CAM de la classe prédite, puis de la vraie classe")
    same_20 = [corr_20(cam_pred_20[i], cam_true_20[i]) for i in range(8, 16)]
    print("erreurs : corrélation entre la carte de la classe prédite et celle de la vraie classe :",
          ", ".join("carte nulle" if v is None else fr(v, 2) for v in same_20))
    empty_20 = [f"{CLASSES[labels_20[i]]} ({which})" for i in range(16)
                for which, cams in (("classe prédite", cam_pred_20), ("vraie classe", cam_true_20)) if cams[i].max() == 0]
    if empty_20:
        print("cartes nulles (la ReLU a tout éteint) :", ", ".join(empty_20))
else:
    print("⏳ Ex B6.20 : l'expérience attend tes deux fonctions (cellules précédentes).")

### Ex B6.21 — Grad-CAM à l'envers : mauvaise couche, mauvaise classe, ReLU oubliée, carte mal alignée 🐛 ★★ ⏱️ 25 min
**Objectif :** trouver dans le code, par des diagnostics enchaînés, les quatre erreurs classiques d'un Grad-CAM.  
**Prérequis :** Ex B6.20 · ✏️ B6.4 · fiche §206.6 (« L'idée », « Avec PyTorch : deux hooks ») et les pièges de la fiche · **Fil rouge :** CIFAR-10 · **Parcours :** C

Un collègue a écrit le Grad-CAM de 206.6 pour `RESNET` (le modèle de B6.20) en quatre morceaux : `LAYER_21`, la couche dont on pondère les activations ; `score_21(logits, target)`, le scalaire que l'on dérive ; `combine_21(A, G)`, la combinaison des activations et des gradients ; `upsample_21(cam, size)`, l'agrandissement de la carte à la taille des images. L'extraction par hooks (`extract_21`, dans la cellule donnée) est juste, comme ta fonction de B6.20. Quatre défauts se sont glissés dans les morceaux, ceux du titre. **Répare la cellule suivante en place** (mêmes noms, mêmes signatures).

La cellule de vérification enchaîne cinq diagnostics, dans l'ordre où un défaut peut en masquer un autre : 1. la résolution des activations de la couche ; 2. la dépendance des gradients à la classe demandée (la vraie classe, puis une autre) ; 3. le signe des cartes avant agrandissement ; 4. l'alignement de la carte agrandie sur l'image, mesuré sur une carte « rampe » dont les cases valent 0, 1, 2 de gauche à droite ; 5. les cartes finales, comparées à la référence `wbx.grad_cam`. Le premier qui échoue arrête la chaîne (⏳ pour les suivants) ; un message dit ce qu'il mesure, pas ce qu'il faut changer. La cellule d'expérience affiche tes cartes à côté de la référence, sur 6 images (3 bonnes prédictions et 3 erreurs de B6.20, la vraie classe) : lance-la une première fois **avant** de réparer, pour voir les symptômes.

Dans tes notes : pour chaque défaut, le symptôme que tu as vu (sur les cartes, puis dans le message) et la ligne fautive. Pourquoi le point 2 compare-t-il les gradients de deux classes plutôt que de regarder une seule carte ? Pourquoi chercher un décalage avec une rampe plutôt qu'avec la carte d'une vraie image ? Lequel de ces défauts une carte « qui a l'air juste » cacherait-elle le mieux ?

In [ ]:
# Six images of B6.20 (three right, three errors), their true classes, another class for each, and the tools of the
# exercise: the extraction by hooks (correct, as in B6.20), the chain of five diagnoses, and the display
PICK_21 = [0, 1, 2, 8, 9, 10]
X_21 = X_20[PICK_21]                                                     # (6, 3, SIZE_20, SIZE_20), prepared
TRUE_21 = POOL_20[1][np.concatenate([RIGHT_20, WRONG_20])][PICK_21]     # their true classes
PRED_21 = PRED_20[np.concatenate([RIGHT_20, WRONG_20])][PICK_21]        # the classes RESNET predicts
OTHER_21 = np.array([next(k for k in range(10) if k not in (t, p)) for t, p in zip(TRUE_21, PRED_21)])
H_21 = SIZE_20 // 32                                                     # the side of the output of layer4


def extract_21(model, layer, images, target, score_fn):
    """(A, G): float64 tensors (N, C, H, W), the output of `layer` for the prepared images and the gradient of
    score_fn(logits, target) with respect to it (two hooks, removed afterwards; the input requires grad)."""
    store = {}

    def save(module, inputs, output):
        store["A"] = output
        output.register_hook(lambda grad: store.__setitem__("G", grad))

    x = torch.as_tensor(images, dtype=torch.float32).clone().requires_grad_(True)
    handle = layer.register_forward_hook(save)
    try:
        score_fn(model(x), torch.as_tensor(np.asarray(target)).long()).backward()
    finally:
        handle.remove()
    if "A" not in store:
        raise ValueError("la couche n'est pas utilisée par la passe avant de RESNET")
    if "G" not in store:
        raise ValueError("le score ne dépend pas de la sortie de la couche (aucun gradient ne lui parvient)")
    return store["A"].detach().double(), store["G"].detach().double()


def pipeline_21(layer, score_fn, combine_fn, upsample_fn, images, target):
    """The whole Grad-CAM: extraction at `layer`, combination, resizing to SIZE_20 x SIZE_20."""
    A, G = extract_21(RESNET, layer, images, target, score_fn)
    return upsample_fn(combine_fn(A, G), SIZE_20)


def diagnose_21(layer, score_fn, combine_fn, upsample_fn):
    """Five diagnoses of the Grad-CAM, in a row: the first one that fails stops the chain (one bug can hide another).
    Returns True when the five pass."""
    total = lambda logits, t: logits.sum()               # a score that does not depend on score_fn

    def point1():
        A, _ = extract_21(RESNET, layer, X_21[:2], TRUE_21[:2], total)
        h, w = A.shape[-2:]
        if (h, w) != (H_21, H_21):
            return False, (f"les activations de ta couche font {h} × {w} sur une image de {SIZE_20} pixels (une case "
                           f"pour {SIZE_20 // h} × {SIZE_20 // w} pixels) : la carte sera très fine, et ses canaux "
                           "répondent à des bords et à des textures plutôt qu'à des objets"), \
                "quelle couche Grad-CAM vise-t-il, et pourquoi (206.6, 🧠 Q8) ?"
        A4, _ = extract_21(RESNET, RESNET.layer4, X_21[:2], TRUE_21[:2], total)
        if A.shape != A4.shape or not torch.allclose(A, A4):
            return False, (f"ta couche est en {h} × {w}, mais sa sortie n'est pas celle du dernier bloc entier de "
                           "RESNET (après sa connexion résiduelle et sa ReLU)"), "relis la liste des modules de RESNET"
        return True, f"les activations font {h} × {w} sur une image de {SIZE_20} pixels", ""

    def point2():
        _, G1 = extract_21(RESNET, layer, X_21, TRUE_21, score_fn)
        _, G2 = extract_21(RESNET, layer, X_21, OTHER_21, score_fn)
        if torch.allclose(G1, G2):
            return False, ("les gradients sont les mêmes quand on demande la vraie classe ou une autre classe, sur les "
                           f"{len(X_21)} images : la carte ne dépend pas de la classe étudiée"), \
                "de quelle classe ton score prend-il le logit ?"
        _, G_ref, _ = wbx.activations_and_gradients(RESNET, layer, X_21, TRUE_21)
        G1 = G1.numpy()
        scale = float((G1 * G_ref).sum() / (G_ref * G_ref).sum())
        if not (scale > 0 and np.allclose(G1, scale * G_ref, rtol=1e-4, atol=1e-7 * np.abs(G_ref).max())):
            return False, ("les gradients changent avec la classe demandée, mais ne sont pas ceux du logit de target[n] "
                           "de chaque image"), "que dérive exactement ton score, image par image ?"
        return True, "les gradients sont ceux du logit de la classe demandée, image par image", ""

    def point3():
        A, G = extract_21(RESNET, layer, X_21, TRUE_21, score_fn)
        cams = torch.as_tensor(combine_fn(A.clone(), G.clone())).double()
        if tuple(cams.shape) != (len(X_21), H_21, H_21):
            return False, f"combine_21 rend la forme {tuple(cams.shape)}", f"attendu ({len(X_21)}, {H_21}, {H_21})"
        low = float(cams.min())
        if low < -1e-9:
            i = int(cams.amin(dim=(1, 2)).argmin())
            return False, (f"les cartes ont des valeurs négatives (jusqu'à {fr(low, 2)} sur l'image {i}) : les régions "
                           "dont l'activation fait baisser le score de la classe y restent"), \
                "relis la formule de Grad-CAM (206.6) : que garde-t-elle de la somme pondérée ?"
        raw = np.einsum("nc,nchw->nhw", G.numpy().mean(axis=(2, 3)), A.numpy())
        want = np.maximum(raw, 0)
        peak = np.abs(want).max(axis=(1, 2), keepdims=True)
        want = np.divide(want, peak, out=np.zeros_like(want), where=peak > 0)
        if not np.allclose(cams.numpy(), want, rtol=1e-6, atol=1e-9):
            return False, (f"les cartes sont positives, mais s'écartent de la formule de 206.6 de "
                           f"{fr(float(np.abs(cams.numpy() - want).max()), 3)}"), "relis combine_21 ligne à ligne"
        return True, "les cartes sont la ReLU de la somme pondérée, chacune divisée par son maximum", ""

    def point4():
        ramp = torch.arange(H_21, dtype=torch.float64).repeat(1, H_21, 1)       # 0, 1, 2 from left to right
        big = torch.as_tensor(upsample_fn(ramp.clone(), SIZE_20)).double()
        if tuple(big.shape) != (1, SIZE_20, SIZE_20):
            return False, f"upsample_21 rend la forme {tuple(big.shape)}", f"attendu (1, {SIZE_20}, {SIZE_20})"
        want = F.interpolate(ramp[:, None], size=(SIZE_20, SIZE_20), mode="bilinear", align_corners=False)[:, 0]
        if torch.allclose(big, want, atol=1e-6):
            return True, ("agrandie, une carte de cases 0, 1, 2 vaut 0, 1 et 2 aux centres de ses cases, comme les "
                          "pixels des images"), ""
        cell = SIZE_20 // H_21
        first = float(big[0, 0, cell // 2 - 1: cell // 2 + 1].mean())          # the centre of the first column of cells
        if abs(first) > 0.01:
            return False, (f"une carte de {H_21} × {H_21} cases qui valent 0, 1, 2 de gauche à droite, agrandie, vaut "
                           f"{fr(first, 2)} au centre de la première case de l'image (pixels {cell // 2 - 1} et "
                           f"{cell // 2}), au lieu de 0 : la carte est décalée de {fr(first * cell, 0)} pixels sur "
                           "l'image à cet endroit"), \
                "à quelle position de l'image chaque case correspond-elle, et comment les images ont-elles été agrandies (wbx.model_input, B6.20) ?"
        return False, ("agrandie, une carte de cases 0, 1, 2 ne passe pas progressivement d'une case à la suivante "
                       "comme les pixels des images"), "comment les images ont-elles été agrandies (wbx.model_input, B6.20) ?"

    def point5():
        mine = torch.as_tensor(pipeline_21(layer, score_fn, combine_fn, upsample_fn, X_21, TRUE_21)).double().numpy()
        want = wbx.grad_cam(RESNET, RESNET.layer4, X_21, TRUE_21, size=SIZE_20)
        if mine.shape == want.shape and np.allclose(mine, want, rtol=1e-4, atol=1e-5):
            return True, f"les {len(X_21)} cartes sont celles de la référence (wbx.grad_cam)", ""
        gap = float(np.abs(mine - want).max()) if mine.shape == want.shape else float("nan")
        return False, f"tes cartes s'écartent de la référence de {fr(gap, 3)}", "un autre changement que les quatre défauts ?"

    points = [("la couche", point1), ("la classe étudiée", point2), ("le signe des cartes", point3),
              ("l'alignement sur l'image", point4), ("les cartes finales", point5)]
    for k, (name, run) in enumerate(points, start=1):
        try:
            ok, measured, why = run()
        except Exception as error:  # noqa: BLE001 - the learner's code: one ❌ line, then the chain stops
            ok, measured, why = False, f"une erreur {type(error).__name__} : {str(error)[:200]}", "relis la fonction en cause"
        finally:
            wbx.clear_hooks(RESNET)                          # a hook left behind by an earlier run
        if ok:
            print(f"✅ Ex B6.21 : point {k}, {name} ({measured}).")
        else:
            print(f"❌ Ex B6.21 : point {k}, {name} : {measured} ; {why}")
            for later in range(k + 1, len(points) + 1):
                print(f"⏳ Ex B6.21 : point {later} : corrige d'abord le point {k}.")
            return False
    print("✅ Ex B6.21 : les cinq diagnostics passent.")
    return True


def overlay_21(image, cam):
    """An image (H, W, 3) in [0, 1] under a map (H, W) in [0, 1] (colour map jet)."""
    return np.clip(0.55 * image + 0.45 * plt.cm.jet(np.clip(cam, 0, 1))[..., :3], 0, 1)


PICS_21 = wbx.model_input(POOL_20[0][np.concatenate([RIGHT_20, WRONG_20])][PICK_21],
                          {"size": SIZE_20, "mean": (0, 0, 0), "std": (1, 1, 1)}).permute(0, 2, 3, 1).clamp(0, 1).numpy()

In [ ]:
LAYER_21 = RESNET.layer4       # 1. the last convolutional block (3 x 3 at 96 pixels), not layer2 (12 x 12)


def score_21(logits, target):
    """The scalar to differentiate: the sum over the images of the logit of the class target[n] ((N,) int64)."""
    return logits.gather(1, target.reshape(-1, 1)).sum()            # 2. the class studied, not the predicted one


def combine_21(A, G):
    """Grad-CAM maps (N, H, W) from the activations A and the gradients G, tensors (N, C, H, W): channel weights =
    spatial mean of G, ReLU of the weighted sum, each map divided by its largest absolute value (a zero map stays 0)."""
    weights = G.mean(dim=(2, 3), keepdim=True)
    cam = torch.relu((weights * A).sum(dim=1))                      # 3. the ReLU, once, on the weighted sum
    peak = cam.abs().amax(dim=(1, 2), keepdim=True)
    return torch.where(peak > 0, cam / torch.where(peak > 0, peak, 1.0), 0.0)


def upsample_21(cam, size):
    """The maps (N, H, W) resized to (N, size, size), the way the images were resized (wbx.model_input)."""
    return F.interpolate(cam[:, None], size=(size, size), mode="bilinear", align_corners=False)[:, 0]   # 4.


repaired_21 = False
with checked("B6.21"):
    repaired_21 = diagnose_21(LAYER_21, score_21, combine_21, upsample_21)

In [ ]:
# Your Grad-CAM, as it is now (it shows the symptoms before the repair), next to the reference of wbx (layer4)
try:
    mine_21 = torch.as_tensor(pipeline_21(LAYER_21, score_21, combine_21, upsample_21, X_21, TRUE_21)).double().numpy()
except Exception as error_21:  # noqa: BLE001 - the learner's code
    mine_21 = None
    print(f"⏳ Ex B6.21 : ton Grad-CAM lève {type(error_21).__name__} ({str(error_21)[:160]}) : rien à montrer.")
finally:
    wbx.clear_hooks(RESNET)
if mine_21 is not None and mine_21.shape == (len(X_21), SIZE_20, SIZE_20):
    ref_21 = wbx.grad_cam(RESNET, RESNET.layer4, X_21, TRUE_21, size=SIZE_20)
    rows_21 = [[PICS_21[i], overlay_21(PICS_21[i], (mine_21[i] - min(0, mine_21[i].min())) / max(np.ptp(mine_21[i]), 1e-12)),
                overlay_21(PICS_21[i], ref_21[i])] for i in range(len(X_21))]
    show_rows(rows_21, [f"{CLASSES[TRUE_21[i]]} (prédit {CLASSES[PRED_21[i]]})" for i in range(len(X_21))],
              ["image", "ton Grad-CAM " + ("réparé" if repaired_21 else "tel quel"), "référence"], [None, None, None],
              size=1.6, suptitle="Grad-CAM de la vraie classe")
    print("valeurs de tes cartes : de", fr(float(mine_21.min()), 2), "à", fr(float(mine_21.max()), 2),
          "; écart maximal à la référence :", fr(float(np.abs(mine_21 - ref_21).max()), 3))
elif mine_21 is not None:
    print(f"⏳ Ex B6.21 : ton Grad-CAM rend la forme {mine_21.shape}, attendu ({len(X_21)}, {SIZE_20}, {SIZE_20}).")

### Ex B6.22 — Test de santé d'Adebayo : randomiser les poids couche par couche, la carte change-t-elle ? 🔬 ★★★ ⏱️ 40 min
**Objectif :** soumettre quatre cartes d'attribution au test de santé par randomisation des poids, et lire leurs corrélations avec leur dispersion.  
**Prérequis :** Ex B6.19, B6.20 · fiche §206.7 (« Une explication peut avoir l'air juste et être fausse », « Les tests de santé ») · 🧠 Q9 · **Fil rouge :** CIFAR-10 · **Parcours :** C

Une carte qui décrit le modèle doit **changer quand le modèle change** (206.7). Tu fais le test d'Adebayo et coll. sur le petit CNN de B6.19, puis sur le ResNet de B6.20. **Ne présume pas l'issue : mesure-la.** **Contrat** :
- `randomize_22(model, names, seed=0)` rend une **copie** gelée (mode `eval()`, `requires_grad=False` partout) de `model`, dans laquelle chaque `Conv2d` et chaque `Linear` des parties nommées (`model.get_submodule(name)` : des noms comme `"fc"` ou `"features.17"`) reprend son `reset_parameters()`, l'initialisation par défaut de PyTorch, tirée après `torch.manual_seed(seed)` **dans** `torch.random.fork_rng()` : `model` et le générateur aléatoire global ne bougent pas, les batchnorms et les autres parties gardent leurs valeurs, et la même graine redonne les mêmes poids ;
- `sanity_22(model, explain, names, mode="cascade", seed=0, absolute=False)` rend un tableau `(len(names), N)` : sa ligne $k$ est la corrélation de rangs de Spearman, image par image, entre les cartes `explain(model)` et `explain(copie)`, la copie randomisant les $k + 1$ premières parties de `names` (`mode="cascade"`, de la sortie vers l'entrée) ou la partie $k$ seule (`mode="independent"`) ; `explain(m)` rend les cartes `(N, …)` des $N$ images pour le modèle `m` ; `absolute=True` compare leurs valeurs absolues ; une carte constante (par exemple toute nulle après la ReLU de Grad-CAM) donne `nan`. `scipy.stats.spearmanr` ou `wbx.spearman_per_image` font la corrélation.

La cellule donnée choisit 16 images de test que le CNN classe bien (`X_22`, `T_22`) et `NAMES_22`, les parties du CNN qui ont des poids, de la sortie vers l'entrée (`wbx.cascade_layers`). La vérification essaie `randomize_22` sur deux parties du CNN (une copie, le générateur, le gel, les parties changées et les autres intactes, la graine) et sur un bloc du ResNet (ses batchnorms restent), puis compare `sanity_22` à un calcul direct sur le gradient du logit de 4 images, dans les deux modes. L'expérience (≈ 25 s) mesure quatre cartes du CNN, pour la vraie classe : Grad-CAM du dernier bloc convolutif (`features[19]`, la ReLU qui suit la dernière convolution, 8 × 8), les gradients intégrés (20 intervalles, valeurs absolues sommées sur les canaux), le gradient (la saillance de B6.19) et un **contrôle** qui ne regarde pas le modèle, les bords de l'image (un filtre de Sobel) ; en cascade et une partie à la fois, pour trois graines des poids aléatoires ; puis le Grad-CAM de `layer4` du ResNet, en cascade, sur les 16 images de B6.20. Elle trace la médiane et les quartiles des corrélations (16 images × 3 graines) et affiche un tableau, avec l'étendue des médianes d'une graine à l'autre et le nombre de cartes constantes.

Dans tes notes : quelles cartes changent quand on randomise, lesquelles non, et dès quelle partie ? Que montre le contrôle, et pourquoi le mettre ? Les gradients intégrés gardent-ils une part de l'image elle-même (pense au facteur $x_i - x'_i$) ? Que veut dire une carte de Grad-CAM qui devient constante ? La dispersion d'une image à l'autre et d'une graine à l'autre change-t-elle tes conclusions ? Qu'est-ce qu'un test réussi prouve, et qu'est-ce qu'il ne prouve pas (🧠 Q9) ?

In [ ]:
# 16 test images that the small CNN classifies right, their classes, and the parts of the CNN that hold weights, from
# the output down (the order of a cascading randomisation)
RIGHT_CNN_22 = np.flatnonzero(wbx.predict(CNN, X_CIFAR).argmax(axis=1) == Y_CIFAR)[:16]
X_22 = wbx.pixels(X_CIFAR[RIGHT_CNN_22])            # (16, 3, 32, 32) in [0, 1]
T_22 = Y_CIFAR[RIGHT_CNN_22]                         # their true (and predicted) classes
NAMES_22 = wbx.cascade_layers(CNN)                  # ['fc', 'features.17', 'features.14', ..., 'features.0']
print("parties du CNN, de la sortie vers l'entrée :", NAMES_22)

In [ ]:
import copy

from scipy.stats import spearmanr


def randomize_22(model, names, seed=0):
    """A frozen copy (eval() mode, requires_grad=False everywhere) of model, in which every Conv2d and Linear module
    inside the parts `names` (model.get_submodule(name)) gets new weights from its own reset_parameters(), drawn after
    torch.manual_seed(seed) inside torch.random.fork_rng(); model itself and the global random generator do not change."""
    out = copy.deepcopy(model)
    with torch.random.fork_rng():                     # the draws of the following cells do not move
        torch.manual_seed(seed)
        for name in names:
            for module in out.get_submodule(name).modules():
                if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear)):
                    module.reset_parameters()
    return out.eval().requires_grad_(False)


def sanity_22(model, explain, names, mode="cascade", seed=0, absolute=False):
    """np.ndarray (len(names), N): row k is the Spearman rank correlation, image by image, between the maps
    explain(model) and explain(copy), copy = randomize_22(model, names[:k + 1], seed) for mode="cascade" (the k + 1
    first parts), randomize_22(model, [names[k]], seed) for mode="independent" (part k alone). explain(m) returns the
    maps (N, ...) of the N images for the model m; absolute=True compares their absolute values; a constant map: nan."""
    if mode not in ("cascade", "independent"):
        raise ValueError(f"mode must be 'cascade' or 'independent', got {mode!r}")
    flat = lambda maps: np.abs(np.asarray(maps, dtype=float)) if absolute else np.asarray(maps, dtype=float)
    before = flat(explain(model))
    rows = []
    for k in range(len(names)):
        parts = names[:k + 1] if mode == "cascade" else [names[k]]
        after = flat(explain(randomize_22(model, parts, seed)))
        row = []
        for u, v in zip(before.reshape(len(before), -1), after.reshape(len(after), -1)):
            row.append(np.nan if np.ptp(u) == 0 or np.ptp(v) == 0 else float(spearmanr(u, v).statistic))
        rows.append(row)
    return np.array(rows)


ok_22 = {}
with checked("B6.22"):
    _chk_before_22 = {k: v.clone() for k, v in CNN.state_dict().items()}
    _chk_rng_22 = torch.get_rng_state()
    _chk_copy_22 = randomize_22(CNN, ["fc", "features.17"], seed=3)
    _chk_changed_22 = [k for k, v in CNN.state_dict().items() if not torch.equal(v, _chk_before_22[k])]
    if _chk_changed_22:
        CNN.load_state_dict(_chk_before_22)                 # the CNN of the following cells stays the trained one
        CNN.eval().requires_grad_(False)
    if not returned("B6.22", "randomize_22", _chk_copy_22):
        raise StopCheck
    if _chk_copy_22 is CNN:
        print("❌ Ex B6.22 : randomize_22 rend le modèle reçu lui-même : le contrat veut une copie (copy.deepcopy).")
        raise StopCheck
    if _chk_changed_22:
        print(f"❌ Ex B6.22 : randomize_22 a modifié le modèle reçu ({_chk_changed_22[0]}…) : travaille sur une copie.")
        raise StopCheck
    verdict("B6.22", torch.equal(torch.get_rng_state(), _chk_rng_22), "le générateur aléatoire global n'a pas bougé.",
            "randomize_22 a déplacé le générateur aléatoire global : tire les poids dans torch.random.fork_rng().")
    verdict("B6.22", not _chk_copy_22.training and not any(p.requires_grad for p in _chk_copy_22.parameters()),
            "la copie est gelée, en mode eval().", "la copie doit être en mode eval() et gelée (requires_grad_(False)).")
    _chk_new_22, _chk_old_22 = _chk_copy_22.state_dict(), CNN.state_dict()
    _chk_inside_22 = [k for k in _chk_old_22 if (k.startswith("fc.") or k.startswith("features.17.")) and k.endswith(("weight", "bias"))]
    _chk_outside_22 = [k for k in _chk_old_22 if k not in _chk_inside_22]
    verdict("B6.22", all(not torch.equal(_chk_new_22[k], _chk_old_22[k]) for k in _chk_inside_22),
            "les poids et biais de fc et de features.17 ont été retirés.",
            "un poids ou un biais de fc ou de features.17 n'a pas changé : chaque Conv2d et chaque Linear des parties "
            "nommées doit reprendre son reset_parameters().")
    verdict("B6.22", all(torch.equal(_chk_new_22[k], _chk_old_22[k]) for k in _chk_outside_22),
            "les autres parties (et les batchnorms) sont intactes.",
            "une autre partie que fc et features.17 a changé (" + next((k for k in _chk_outside_22 if not torch.equal(_chk_new_22[k], _chk_old_22[k])), "") + ").")
    _chk_block_22 = randomize_22(RESNET, ["layer4.1"], seed=3).state_dict()     # a block: two convolutions, two batchnorms
    _chk_old_r_22 = RESNET.state_dict()
    _chk_bn_22 = [k for k in _chk_old_r_22 if k.startswith("layer4.1.bn")]
    verdict("B6.22", all(torch.equal(_chk_block_22[k], _chk_old_r_22[k]) for k in _chk_bn_22)
            and not torch.equal(_chk_block_22["layer4.1.conv1.weight"], _chk_old_r_22["layer4.1.conv1.weight"]),
            "dans un bloc du ResNet (layer4.1), les convolutions changent, les batchnorms restent.",
            "dans le bloc layer4.1 du ResNet, seules les Conv2d (et les Linear) doivent reprendre leur reset_parameters() : "
            "les batchnorms gardent leurs poids et leurs statistiques.")
    _chk_again_22 = randomize_22(CNN, ["fc", "features.17"], seed=3).state_dict()
    _chk_other_22 = randomize_22(CNN, ["fc", "features.17"], seed=4).state_dict()
    verdict("B6.22", all(torch.equal(_chk_again_22[k], _chk_new_22[k]) for k in _chk_new_22)
            and not all(torch.equal(_chk_other_22[k], _chk_new_22[k]) for k in _chk_inside_22),
            "la même graine redonne les mêmes poids, une autre graine d'autres poids.",
            "randomize_22 doit être reproductible (même graine, mêmes poids) et changer avec la graine.")
    ok_22["randomize"] = True

with checked("B6.22"):
    if not ok_22:
        raise StopCheck
    probe_22 = lambda m: wbx.input_gradients(m, X_22[:4], T_22[:4])     # a cheap map: the gradient of the logit
    names_22 = ["fc", "features.17", "features.14"]
    for mode_22 in ("cascade", "independent"):
        for absolute_22 in (False, True):
            got_22 = sanity_22(CNN, probe_22, names_22, mode=mode_22, seed=5, absolute=absolute_22)
            if not returned("B6.22", "sanity_22", got_22):
                raise StopCheck
            ref_maps_22 = probe_22(CNN)
            want_22 = np.array([wbx.spearman_per_image(ref_maps_22, probe_22(randomize_22(
                CNN, names_22[:k + 1] if mode_22 == "cascade" else [names_22[k]], seed=5)), absolute=absolute_22)
                for k in range(len(names_22))])
            arr_22 = np.asarray(got_22, dtype=float)
            if arr_22.shape != want_22.shape:
                print(f"❌ Ex B6.22 : sanity_22 doit rendre un tableau (3, 4) (une ligne par partie, une colonne par image), "
                      f"reçu la forme {arr_22.shape}.")
                raise StopCheck
            if not same("B6.22", f"sanity_22, mode {mode_22}, absolute={absolute_22}", arr_22, want_22, rtol=1e-9, atol=1e-9):
                raise StopCheck
    ok_22["sanity"] = True

In [ ]:
# The sanity checks of Adebayo et al. on the small CNN (16 images, the true class): four maps, cascading and independent
# randomisation, three seeds of the random weights; then the Grad-CAM of the ResNet of B6.20 in cascade
SOBEL_22 = torch.tensor([[1.0, 0.0, -1.0], [2.0, 0.0, -2.0], [1.0, 0.0, -1.0]])


def edges_22(model):
    """The control: the edges of the images (Sobel), a map that does not look at the model."""
    gray = X_22.mean(dim=1, keepdim=True)
    gx = F.conv2d(gray, SOBEL_22.view(1, 1, 3, 3), padding=1)
    gy = F.conv2d(gray, SOBEL_22.T.reshape(1, 1, 3, 3), padding=1)
    return (gx ** 2 + gy ** 2).sqrt()[:, 0].numpy()


METHODS_22 = {   # name: (explain(model) -> maps, absolute)
    "Grad-CAM (features.19, 8 × 8)": (lambda m: wbx.grad_cam(m, m.features[19], X_22, T_22), False),
    "gradients intégrés (20 intervalles)": (lambda m: wbx.integrated_gradients(m, X_22, T_22, n_steps=20).sum(axis=1), True),
    "gradient (saillance)": (lambda m: np.abs(wbx.input_gradients(m, X_22, T_22)).max(axis=1), False),
    "contrôle : bords de l'image": (edges_22, False),
}
SEEDS_22 = (0, 1, 2)
if len(ok_22) == 2:
    start_22 = time.perf_counter()
    RESULTS_22 = {}
    for method_22, (explain_22, absolute_22) in METHODS_22.items():
        for mode_22 in ("cascade", "independent"):
            RESULTS_22[method_22, mode_22] = np.stack([sanity_22(CNN, explain_22, NAMES_22, mode=mode_22, seed=s,
                                                                 absolute=absolute_22) for s in SEEDS_22])
    names_r_22 = wbx.cascade_layers(RESNET)
    resnet_cam_22 = lambda m: wbx.grad_cam(m, m.layer4, X_20, POOL_20[1][np.concatenate([RIGHT_20, WRONG_20])])
    RESNET_22 = np.stack([sanity_22(RESNET, resnet_cam_22, names_r_22, mode="cascade", seed=s) for s in SEEDS_22])

    fig_22, axes_22 = plt.subplots(1, 3, figsize=(15, 4))
    for ax_22, mode_22 in zip(axes_22[:2], ("cascade", "independent")):
        for method_22 in METHODS_22:
            rho_22 = RESULTS_22[method_22, mode_22].reshape(-1, len(NAMES_22), 16).transpose(1, 0, 2).reshape(len(NAMES_22), -1)
            med_22 = np.nanmedian(rho_22, axis=1)
            q1_22, q3_22 = np.nanpercentile(rho_22, 25, axis=1), np.nanpercentile(rho_22, 75, axis=1)
            line_22, = ax_22.plot(range(len(NAMES_22)), med_22, marker="o", label=method_22)
            ax_22.fill_between(range(len(NAMES_22)), q1_22, q3_22, color=line_22.get_color(), alpha=0.15)
        ax_22.set_xticks(range(len(NAMES_22)), NAMES_22, rotation=45, ha="right", fontsize=8)
        ax_22.axhline(0, color="gray", lw=0.5)
        ax_22.set_ylim(-0.6, 1.05)
        ax_22.set_title(f"petit CNN, randomisation {'en cascade (jusqu’à la partie)' if mode_22 == 'cascade' else 'indépendante (cette partie seule)'}", fontsize=9)
        ax_22.set_ylabel("corrélation de Spearman (médiane, quartiles)")
    rho_r_22 = RESNET_22.transpose(1, 0, 2).reshape(len(names_r_22), -1)
    axes_22[2].plot(range(len(names_r_22)), np.nanmedian(rho_r_22, axis=1), marker="o", color="C0", label="Grad-CAM (layer4)")
    axes_22[2].fill_between(range(len(names_r_22)), np.nanpercentile(rho_r_22, 25, axis=1),
                            np.nanpercentile(rho_r_22, 75, axis=1), color="C0", alpha=0.15)
    axes_22[2].set_xticks(range(len(names_r_22)), names_r_22, rotation=45, ha="right", fontsize=8)
    axes_22[2].axhline(0, color="gray", lw=0.5)
    axes_22[2].set_ylim(-0.6, 1.05)
    axes_22[2].set_title("ResNet de B6.20, randomisation en cascade (16 images)", fontsize=9)
    axes_22[0].legend(fontsize=7)
    plt.tight_layout()
    plt.show()

    print("médiane [1er quartile ; 3e quartile] sur 16 images × 3 graines ; entre parenthèses, l'étendue des médianes "
          "des 3 graines ; nan : cartes constantes (comptées à part)")
    for method_22 in METHODS_22:
        for mode_22 in ("cascade", "independent"):
            r_22 = RESULTS_22[method_22, mode_22]                      # (seeds, parts, images)
            cells_22 = []
            for k_22 in (0, 1, 3, len(NAMES_22) - 1):
                v_22 = r_22[:, k_22, :]
                per_seed_22 = np.nanmedian(v_22, axis=1)
                cells_22.append(f"{NAMES_22[k_22]} {fr(np.nanmedian(v_22), 2)} [{fr(np.nanpercentile(v_22, 25), 2)} ; "
                                f"{fr(np.nanpercentile(v_22, 75), 2)}] ({fr(per_seed_22.min(), 2)} à {fr(per_seed_22.max(), 2)})"
                                + (f", {int(np.isnan(v_22).sum())} nan" if np.isnan(v_22).any() else ""))
            print(f"  {method_22}, {mode_22} : " + " · ".join(cells_22))
    for k_22 in (0, 1, 2, len(names_r_22) - 1):
        v_22 = RESNET_22[:, k_22, :]
        print(f"  ResNet, Grad-CAM, cascade jusqu'à {names_r_22[k_22]} : {fr(np.nanmedian(v_22), 2)} "
              f"[{fr(np.nanpercentile(v_22, 25), 2)} ; {fr(np.nanpercentile(v_22, 75), 2)}]"
              + (f", {int(np.isnan(v_22).sum())} nan" if np.isnan(v_22).any() else ""))
    print(f"ℹ️ {time.perf_counter() - start_22:.0f} s")
else:
    print("⏳ Ex B6.22 : l'expérience attend tes deux fonctions (cellule précédente).")

## Partie C · Équité : auditer, corriger, documenter

Au MP3, tu as prédit un revenu de plus de 50 000 $ sur *Adult* avec un pipeline sans fuite, audité ses erreurs par sexe et par âge, et mesuré sur les folds le retrait de `sex`. Ici, tu écris d'abord les indicateurs de ta librairie (B6.23), tu refais l'audit **avec des intervalles** (B6.24), tu lis un tableau d'audit (B6.25), tu prédis ce que change le retrait de `sex` (B6.26), tu essaies trois corrections et tu chiffres leur coût (B6.27), tu documentes le modèle dans une fiche (B6.28), puis tu relèves le défi (B6.29).

La cellule suivante charge le découpage figé d'*Adult* (`wbx.adult()` : `XA_TRAIN`, `YA_TRAIN`, 29 304 personnes d'entraînement ; `XA_VAL`, `YA_VAL`, 9 769 de validation ; le test, celui du MP3, reste à part jusqu'au défi), `adult_pipeline(model=None, drop=())`, le pipeline du MP3 en encodage ordinal (`income.build_pipeline(..., encoding="ordinal")` : `fnlwgt` et `education` écartés, et les colonnes de `drop`), `age_band`, les tranches d'âge du MP3, et `HGB_C`, le gradient boosting de référence de la partie (pipeline du MP3, `random_state=0`, ajusté sur l'entraînement), avec ses scores `P_VAL`, ses décisions `PRED_VAL` au seuil 0,5 sur la validation, `SEX_VAL` et `AGE_VAL`. `sk_audit(y, p, g)` rend les taux par groupe et les écarts, calculés avec scikit-learn : c'est la référence des vérifications et des expériences données, qui ne dépendent jamais de ton code. ≈ 5 s.

In [ ]:
# Tools of part C: Adult (frozen split; its test part, the one of the MP3, stays aside until the challenge), the
# pipeline of the MP3, the age bands of the MP3, and the reference model of the part (with its validation scores)
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.metrics import confusion_matrix as sk_confusion_matrix
from sklearn.preprocessing import OrdinalEncoder

ADULT = wbx.adult()                                   # 29 304 / 9 769 / 9 769 people (the test part: the one of the MP3)
XA_TRAIN, YA_TRAIN = ADULT.X_train, ADULT.y_train.to_numpy()
XA_VAL, YA_VAL = ADULT.X_val, ADULT.y_val.to_numpy()
NUMERIC_C = ["age", "education-num", "capital-gain", "capital-loss", "hours-per-week"]
CATEGORICAL_C = ["workclass", "marital-status", "occupation", "relationship", "race", "sex", "native-country"]
AGE_BINS_C, AGE_LABELS_C = [25, 35, 45, 55, 65], ["<25", "25-34", "35-44", "45-54", "55-64", "65+"]


def adult_pipeline(model=None, drop=()):
    """The pipeline of the MP3 (income.build_pipeline(model, encoding="ordinal")), not fitted: the categorical columns
    coded 0, 1, 2… (OrdinalEncoder, categories seen fewer than 20 times grouped, missing and unknown values as NaN),
    then the numeric columns unchanged; fnlwgt and education left out, and the columns of `drop` too. model: by
    default HistGradientBoostingClassifier(random_state=0), told which columns are categorical."""
    categorical = [c for c in CATEGORICAL_C if c not in drop]
    numeric = [c for c in NUMERIC_C if c not in drop]
    encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan, min_frequency=20)
    prep = ColumnTransformer([("categorical", encoder, categorical), ("numeric", "passthrough", numeric)],
                             remainder="drop", verbose_feature_names_out=False, sparse_threshold=0.0)
    model = HistGradientBoostingClassifier(random_state=0) if model is None else model
    if isinstance(model, HistGradientBoostingClassifier):
        model.set_params(categorical_features=list(range(len(categorical))) or None)
    return Pipeline([("prep", prep), ("model", model)])


def age_band(age):
    """The age band of each person (the bands of the MP3): "<25", "25-34", …, "65+" (25 is in "25-34")."""
    return np.asarray(AGE_LABELS_C, dtype=object)[np.digitize(np.asarray(age, dtype=float), AGE_BINS_C)]


def sk_audit(y, p, g):
    """({group: {"n", "positives", "selection_rate", "tpr", "fpr", "accuracy"}}, {"dp", "tpr", "fpr", "eo"}): the rates
    of each group and the gaps (largest minus smallest), from scikit-learn's confusion matrix: the reference of the
    checks and of the experiments given in part C (they never depend on your code)."""
    y, p, g = np.asarray(y, dtype=int), np.asarray(p, dtype=int), np.asarray(g)
    rates = {}
    for key in np.unique(g):
        m = g == key
        tn, fp, fn, tp = sk_confusion_matrix(y[m], p[m], labels=[0, 1]).ravel()
        n = int(m.sum())
        rates[key] = {"n": n, "positives": int(tp + fn), "selection_rate": (tp + fp) / n,
                      "tpr": tp / (tp + fn) if tp + fn else float("nan"),
                      "fpr": fp / (fp + tn) if fp + tn else float("nan"), "accuracy": (tp + tn) / n}
    spread = lambda k: float(max(r[k] for r in rates.values()) - min(r[k] for r in rates.values()))
    gaps = {"dp": spread("selection_rate"), "tpr": spread("tpr"), "fpr": spread("fpr")}
    gaps["eo"] = max(gaps["tpr"], gaps["fpr"])
    return rates, gaps


HGB_C = adult_pipeline().fit(XA_TRAIN, YA_TRAIN)      # the reference model of part C, fitted on the training part
P_VAL = HGB_C.predict_proba(XA_VAL)[:, 1]             # its scores on the validation part
PRED_VAL = (P_VAL >= 0.5).astype(int)                 # its decisions, threshold 0.5
SEX_VAL, AGE_VAL = XA_VAL["sex"].to_numpy(), age_band(XA_VAL["age"])
print(f"Adult : {len(XA_TRAIN)} personnes d'entraînement, {len(XA_VAL)} de validation (le test du MP3, "
      f"{len(ADULT.X_test)} personnes, reste à part jusqu'au défi) ; taux de base {fr(YA_TRAIN.mean(), 3)}. Gradient "
      f"boosting de référence (pipeline du MP3), sur la validation : accuracy {fr(np.mean(PRED_VAL == YA_VAL), 4)} au "
      f"seuil 0,5, ROC-AUC {fr(roc_auc_score(YA_VAL, P_VAL), 4)}.")

### Ex B6.23 — group_rates, demographic_parity_difference et equalized_odds_difference 🔨 ★★ ⏱️ 25 min
**Objectif :** écrire les taux par groupe et les deux écarts d'équité de ta librairie, avec leurs cas limites.  
**Prérequis :** ✏️ B6.5 · ch. 3 (`confusion_matrix`) · 🔁 R3 · fiche §206.9 (« Des taux par groupe », « Les critères ») · **mylearn :** `explain.py` · **Parcours :** R, C

> **Mode d'emploi mylearn** : ouvre `mon_travail/mylearn/explain.py` (créé par `python tools/start_chapter.py B6`), lis la docstring, remplace chaque `raise NotImplementedError(...)` de l'exercice par ton code et **enregistre**. NumPy seulement. La première cellule de vérification recharge ta librairie et l'essaie sur des cas visibles ; la seconde lance les tests de l'exercice. `python -m pytest tests/test_chb6_explain.py -q` les lance tous dans un terminal.

**Contrat** de `group_rates(y_true, y_pred, groups)`, qui rend `{groupe: {'n', 'base_rate', 'selection_rate', 'tpr', 'fpr', 'precision', 'accuracy'}}` :
- les clés sont les valeurs des groupes, dans l'ordre de `np.unique` (`rates[10]` pour le groupe entier 10 ; un scalaire Python ou NumPy convient) ; `'n'` est un entier, les taux des floats, et un taux dont le dénominateur est nul vaut `nan` (un groupe sans positif a un TPR `nan`) ;
- chaque groupe a sa matrice de confusion 2 × 2, celle de ta `confusion_matrix` du ch. 3 avec `labels=[0, 1]` : sans `labels`, un groupe qui n'a qu'un seul label donne une matrice 1 × 1, que `tn, fp, fn, tp = C.ravel()` ne sait pas défaire ; `C.ravel()` rend `TN, FP, FN, TP` (lignes = vérité, colonnes = prédiction, la disposition de scikit-learn ; le livre met TP en haut à gauche) ;
- un seul groupe est permis ; `ValueError` si les longueurs diffèrent ou si un label ou une prédiction n'est pas 0 ou 1 (NaN, 0,5, 2 ou −1 sont refusés) ; les entrées ne sont pas modifiées.

**Contrat** de `demographic_parity_difference(y_pred, groups)` et de `equalized_odds_difference(y_true, y_pred, groups)` :
- le plus grand moins le plus petit taux de sélection, sur **tous** les groupes ; pour l'equalized odds, le plus grand de deux écarts, celui des TPR et celui des FPR, chacun le plus grand moins le plus petit sur tous les groupes ;
- `ValueError` s'il y a moins de deux groupes, et, pour l'equalized odds, si un groupe n'a aucun positif ou aucun négatif (son TPR ou son FPR est indéfini ; fairlearn rend alors un nombre, fiche 🕰️) ; les mêmes refus que `group_rates`.

La première cellule de vérification essaie tes fonctions sur l'exemple de la docstring, sur trois groupes entiers non triés (contre `sklearn.metrics.confusion_matrix`), sur un groupe sans positif, sur un groupe dont les labels et les prédictions valent tous 0, sur les deux écarts à trois groupes et sur trois refus ; la seconde lance les 42 tests de l'exercice (`-k "test_group_rates_ or test_demographic_parity_difference_ or test_equalized_odds_difference_"`).

Dans tes notes : pourquoi l'ordre `TN, FP, FN, TP` est-il un piège quand on passe d'un livre à scikit-learn ? Pourquoi ton equalized odds refuse-t-il un groupe sans positif, quand fairlearn rend un nombre ? Avec trois groupes, l'écart dit-il entre quels groupes il se trouve ?

In [ ]:
with checked("B6.23"):
    E = mylearn.explain                        # inside the block: a library not created yet gives ⏳
    y_23, p_23, g_23 = [1, 0, 1, 0, 1, 1, 0, 0], [1, 0, 0, 1, 1, 1, 0, 0], ["a"] * 4 + ["b"] * 4     # the docstring
    r_23 = E.group_rates(list(y_23), list(p_23), list(g_23))
    if returned("B6.23", "group_rates", r_23):
        if not isinstance(r_23, dict) or list(r_23) != ["a", "b"]:
            print(f"❌ Ex B6.23 : group_rates doit rendre un dictionnaire dont les clés sont les groupes, dans l'ordre de "
                  f"np.unique (['a', 'b']), reçu {type(r_23).__name__} {list(r_23)[:4] if isinstance(r_23, dict) else ''}.")
            raise StopCheck
        keys_23 = ["accuracy", "base_rate", "fpr", "n", "precision", "selection_rate", "tpr"]
        verdict("B6.23", sorted(r_23["a"]) == keys_23, "les sept clés de chaque groupe sont là.",
                f"chaque groupe doit avoir les sept clés {keys_23}, reçu {sorted(r_23['a'])}.")
        got_23 = (r_23["a"]["n"], r_23["a"]["tpr"], r_23["a"]["fpr"], r_23["b"]["n"], r_23["b"]["tpr"], r_23["b"]["fpr"])
        verdict("B6.23", got_23 == (4, 0.5, 0.5, 4, 1.0, 0.0) and isinstance(r_23["a"]["n"], (int, np.integer)),
                "l'exemple de la docstring : (4, 0,5, 0,5) et (4, 1,0, 0,0), 'n' entier.",
                f"l'exemple de la docstring donne {got_23} au lieu de (4, 0.5, 0.5, 4, 1.0, 0.0) (n, TPR, FPR des groupes a et b).")

with checked("B6.23"):     # three groups of different sizes, labels drawn at random: every rate against scikit-learn
    E = mylearn.explain                        # inside the block: a library not created yet gives ⏳
    rng_23 = np.random.default_rng(23)
    g3_23 = np.repeat([30, 10, 20], [40, 25, 35])                    # integer groups, not sorted
    y3_23, p3_23 = rng_23.integers(0, 2, 100), rng_23.integers(0, 2, 100)
    r3_23 = E.group_rates(y3_23.copy(), p3_23.copy(), g3_23.copy())
    if returned("B6.23", "group_rates", r3_23):
        if list(r3_23) != [10, 20, 30]:
            print(f"❌ Ex B6.23 : avec les groupes 30, 10 et 20, les clés doivent être 10, 20, 30 (l'ordre de np.unique), "
                  f"reçu {list(r3_23)}.")
            raise StopCheck
        bad_23 = []
        for key_23 in (10, 20, 30):
            m_23 = g3_23 == key_23
            tn, fp, fn, tp = sk_confusion_matrix(y3_23[m_23], p3_23[m_23], labels=[0, 1]).ravel()
            want_23 = {"n": tn + fp + fn + tp, "base_rate": (tp + fn) / m_23.sum(), "selection_rate": (tp + fp) / m_23.sum(),
                       "tpr": tp / (tp + fn), "fpr": fp / (fp + tn), "precision": tp / (tp + fp), "accuracy": (tp + tn) / m_23.sum()}
            bad_23 += [f"{key}[{key_23}] = {fr(float(r3_23[key_23][key]), 4)} au lieu de {fr(float(value), 4)}"
                       for key, value in want_23.items() if not np.isclose(float(r3_23[key_23][key]), float(value))]
        if bad_23:
            swapped_23 = any("tpr" in b for b in bad_23) and any("fpr" in b for b in bad_23)
            print("❌ Ex B6.23 : contre scikit-learn, " + " ; ".join(bad_23[:3])
                  + (" : C.ravel() rend TN, FP, FN, TP (lignes = vérité, comme scikit-learn)" if swapped_23 else "") + ".")
        else:
            print("✅ Ex B6.23 : trois groupes entiers non triés : les sept taux de chaque groupe sont ceux de scikit-learn.")

with checked("B6.23"):     # a group without positives: nan for its TPR, and no crash
    E = mylearn.explain                        # inside the block: a library not created yet gives ⏳
    r4_23 = E.group_rates([0, 0, 0, 1, 0, 1], [0, 1, 0, 1, 0, 0], ["p", "p", "p", "q", "q", "q"])
    if returned("B6.23", "group_rates", r4_23):
        tpr_p_23 = r4_23["p"]["tpr"]
        verdict("B6.23", isinstance(tpr_p_23, float) and math.isnan(tpr_p_23) and math.isclose(r4_23["p"]["fpr"], 1 / 3),
                "un groupe sans positif : TPR nan, FPR 1/3.",
                f"le groupe p n'a aucun positif : son TPR vaut nan (dénominateur nul) et son FPR 1/3, reçu {tpr_p_23} et "
                f"{r4_23['p']['fpr']}.")
with checked("B6.23"):     # a group whose labels AND predictions are all 0: a 1 x 1 matrix without labels=[0, 1]
    E = mylearn.explain                        # inside the block: a library not created yet gives ⏳
    try:
        r5_23 = E.group_rates([0, 0, 1, 0], [0, 0, 1, 1], ["q", "q", "r", "r"])
    except ValueError as exc_23:
        if "unpack" in str(exc_23):
            print("❌ Ex B6.23 : un groupe dont les labels et les prédictions valent tous 0 fait échouer ta fonction ("
                  + str(exc_23) + ") : sa matrice de confusion n'a qu'une case ; relis le contrat : labels=[0, 1].")
            raise StopCheck
        raise
    if returned("B6.23", "group_rates", r5_23):
        q_23 = r5_23["q"]
        verdict("B6.23", q_23["n"] == 2 and math.isnan(q_23["tpr"]) and q_23["fpr"] == 0 and math.isnan(q_23["precision"]),
                "un groupe sans positif ni sélectionné : TPR et precision nan, FPR 0.",
                f"le groupe q (labels et prédictions tous à 0) doit avoir n = 2, un TPR et une precision nan, un FPR 0 ; "
                f"reçu {q_23}.")
with checked("B6.23"):
    E = mylearn.explain                        # inside the block: a library not created yet gives ⏳
    dp_23 = E.demographic_parity_difference([1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0], ["x"] * 4 + ["y"] * 4 + ["z"] * 4)
    eo_23 = E.equalized_odds_difference([1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1],
                                        ["x"] * 4 + ["y"] * 4 + ["z"] * 4)
    if returned("B6.23", "demographic_parity_difference", dp_23) and returned("B6.23", "equalized_odds_difference", eo_23):
        verdict("B6.23", math.isclose(float(dp_23), 0.5), "parité démographique sur trois groupes : 0,75 − 0,25 = 0,5.",
                f"trois groupes de taux de sélection 0,5, 0,25 et 0,75 : l'écart vaut le plus grand moins le plus petit, "
                f"reçu {dp_23}.")
        verdict("B6.23", math.isclose(float(eo_23), 1.0), "equalized odds sur trois groupes : le plus grand des écarts de TPR et de FPR.",
                f"trois groupes de TPR 1, 0,5 et 1 et de FPR 0,5, 0 et 1 : l'écart vaut le plus grand des deux écarts "
                f"(max − min sur tous les groupes), reçu {eo_23}.")
        refuses("B6.23", "equalized_odds_difference d'un groupe sans positif", lambda: E.equalized_odds_difference(
            [0, 0, 1, 0], [0, 1, 1, 0], ["p", "p", "q", "q"]))
        refuses("B6.23", "demographic_parity_difference d'un seul groupe", lambda: E.demographic_parity_difference([1, 0], ["a", "a"]))
        refuses("B6.23", "group_rates avec un label 2", lambda: E.group_rates([0, 2], [0, 1], ["a", "b"]))

In [ ]:
if written(lambda: mylearn.explain.group_rates([0, 1], [0, 1], [0, 0])):
    run_pytest_group(TEST_EXPLAIN, impl="ref", keyword="test_group_rates_ or test_demographic_parity_difference_ or test_equalized_odds_difference_")   # 42 tests
else:
    print("⏳ Ex B6.23 : les tests attendent tes trois fonctions.")

### Ex B6.24 — Audit d'équité sur Adult : indicateurs par sexe et par âge, intervalles de confiance bootstrap 📦 ★★★ ⏱️ 45 min
**Objectif :** auditer un classifieur groupe par groupe, avec un intervalle bootstrap pour chaque taux et chaque écart.  
**Prérequis :** Ex B6.23 · ch. 2 (`bootstrap_ci`, Ex 2.22) · CP3 (MP3.7) · fiche §206.8 et §206.9 (« Des écarts avec leurs intervalles ») · **Parcours :** R, C

Au MP3, ton audit donnait des taux par groupe sans intervalles (tes intervalles bootstrap portaient sur le test entier). Un écart entre deux taux ne dit rien sans son incertitude, surtout pour un petit groupe (206.9, « Des écarts avec leurs intervalles »). **Contrat** :
- `rate_ci_24(y_true, y_pred, rate, n_boot=500, rng=None)` rend `(low, high)`, l'intervalle bootstrap à 95 % (centiles 2,5 et 97,5) du taux `rate` (`"selection_rate"`, `"tpr"` ou `"fpr"`) d'**un** groupe, calculé par ta `bootstrap_ci` (ch. 2) sur les paires `(y_true, y_pred)` du groupe, tirées ensemble ; sur un tirage sans dénominateur (aucun positif tiré, pour un TPR), la statistique rend `nan`, que ta `bootstrap_ci` refuse (`ValueError`, ch. 2) : sur les groupes d'*Adult*, cela n'arrive pas (le plus petit dénominateur, les 12 positifs des moins de 25 ans, n'est presque jamais tiré vide) ; pour un petit groupe, il faudrait `bootstrap_distribution` puis `np.nanpercentile`, en disant la part des tirages écartés ;
- `audit_24(y_true, y_pred, groups, n_boot=500, seed=0)` rend un `pd.DataFrame`, une ligne par groupe (dans l'ordre de `np.unique`, index nommé `"group"`), les colonnes `COLUMNS_24` : `n`, `positives` (les y = 1 du groupe), les taux de ta `group_rates` (`base_rate`, `selection_rate`, `tpr`, `fpr`, `precision`, `accuracy`), et les bornes `selection_rate_low`, `selection_rate_high`, `tpr_low`, `tpr_high`, `fpr_low`, `fpr_high` de `rate_ci_24` sur les lignes du groupe, avec un seul `rng = np.random.default_rng(seed)` pour tout le tableau ;
- `gap_ci_24(y_true, y_pred, groups, kind, n_boot=500, seed=0)` rend `(gap, low, high)` : l'écart `kind` sur les groupes (`"dp"` : ta `demographic_parity_difference` ; `"eo"` : ta `equalized_odds_difference` ; `"tpr"` ou `"fpr"` : le plus grand moins le plus petit de ce taux) et son intervalle à 95 % ; à chaque tirage, les lignes de **chaque groupe** sont tirées avec remise dans leur groupe (un bootstrap stratifié par groupe : chaque tirage garde les effectifs ; un seul `rng = np.random.default_rng(seed)` pour tous les tirages), puis l'écart est recalculé.

La cellule de vérification audite `PRED_VAL` (le modèle de référence, seuil 0,5, sur la validation) par sexe et par tranche d'âge : les taux sont comparés à scikit-learn ; les intervalles doivent contenir leur taux et avoir la largeur attendue, celle de l'approximation normale $\pm 1{,}96\sqrt{r(1-r)/m}$ à 30 % près, quand le dénominateur $m$ dépasse 100 ; la même graine doit redonner le même tableau, et deux groupes identiques des intervalles différents (un seul générateur) ; puis `gap_ci_24` est comparé aux écarts de scikit-learn, et ses tirages doivent garder l'effectif de chaque groupe. L'expérience affiche les deux tableaux avec leurs intervalles et les quatre écarts par sexe et par âge, et trace les taux par groupe. `N_BOOT_24` vaut 500 en mode rapide (≈ 30 s pour la vérification et l'expérience), 2 000 sinon.

Dans tes notes : quel groupe a les intervalles les plus larges, et pourquoi ? Où sont les écarts par sexe : dans les TPR, dans les FPR, dans la sélection ? Leurs intervalles excluent-ils 0 ? Les écarts par âge viennent-ils plutôt des erreurs du modèle ou des taux de base ? Pourquoi tirer dans chaque groupe plutôt que dans toute la validation ? Que deviendraient tes conclusions avec 24 sous-groupes croisés (sexe × âge × deux catégories raciales) ?

In [ ]:
COLUMNS_24 = ["n", "positives", "base_rate", "selection_rate", "selection_rate_low", "selection_rate_high", "tpr",
              "tpr_low", "tpr_high", "fpr", "fpr_low", "fpr_high", "precision", "accuracy"]
N_BOOT_24 = 500 if FAST_MODE else 2000                # bootstrap draws (the fiche: 2 000)

In [ ]:
def _rate_24(y, p, rate):
    """A rate of one group from its labels and decisions (nan when its denominator is 0)."""
    y, p = np.asarray(y), np.asarray(p)
    if rate == "selection_rate":
        return float(p.mean()) if len(p) else float("nan")
    keep = y == (1 if rate == "tpr" else 0)
    return float(p[keep].mean()) if keep.any() else float("nan")


def rate_ci_24(y_true, y_pred, rate, n_boot=500, rng=None):
    """(low, high): the 95 % percentile bootstrap interval of the rate `rate` ("selection_rate", "tpr" or "fpr") of
    ONE group, from its pairs (y_true, y_pred), with your bootstrap_ci (ch. 2); the statistic gives nan on a draw
    without a denominator, which bootstrap_ci refuses (ValueError): it does not happen on the groups of Adult."""
    pairs = np.column_stack([np.asarray(y_true, dtype=int), np.asarray(y_pred, dtype=int)])   # rows stay paired
    return mylearn.stats.bootstrap_ci(pairs, lambda s: _rate_24(s[:, 0], s[:, 1], rate), n_boot=n_boot, rng=rng)


def audit_24(y_true, y_pred, groups, n_boot=500, seed=0):
    """pd.DataFrame, one row per group (the order of np.unique, index named "group"), the columns COLUMNS_24: n,
    positives (the y = 1 of the group), the rates of your group_rates, and the interval of selection_rate, tpr and fpr
    (rate_ci_24 on the rows of the group, one rng = np.random.default_rng(seed) for the whole table)."""
    y, p, g = np.asarray(y_true, dtype=int), np.asarray(y_pred, dtype=int), np.asarray(groups)
    rates = mylearn.explain.group_rates(y, p, g)
    rng = np.random.default_rng(seed)
    rows = {}
    for key, r in rates.items():
        m = g == key
        row = {"n": r["n"], "positives": int(y[m].sum()), **{k: r[k] for k in ("base_rate", "selection_rate", "tpr", "fpr",
                                                                               "precision", "accuracy")}}
        for rate in ("selection_rate", "tpr", "fpr"):
            row[f"{rate}_low"], row[f"{rate}_high"] = rate_ci_24(y[m], p[m], rate, n_boot=n_boot, rng=rng)
        rows[key] = row
    table = pd.DataFrame.from_dict(rows, orient="index")[COLUMNS_24]
    table.index.name = "group"
    return table


def gap_ci_24(y_true, y_pred, groups, kind, n_boot=500, seed=0):
    """(gap, low, high): the gap `kind` over the groups ("dp": your demographic_parity_difference, "eo": your
    equalized_odds_difference, "tpr" or "fpr": the largest minus the smallest rate) and its 95 % bootstrap interval,
    the rows of each group resampled with replacement within their group (rng = np.random.default_rng(seed))."""
    y, p, g = np.asarray(y_true, dtype=int), np.asarray(y_pred, dtype=int), np.asarray(groups)
    E = mylearn.explain

    def gap(yy, pp, gg):
        if kind == "dp":
            return float(E.demographic_parity_difference(pp, gg))
        if kind == "eo":
            return float(E.equalized_odds_difference(yy, pp, gg))
        values = [r[kind] for r in E.group_rates(yy, pp, gg).values()]
        return float(max(values) - min(values))

    rng = np.random.default_rng(seed)
    members = [np.flatnonzero(g == key) for key in np.unique(g)]
    draws = []
    for _ in range(n_boot):
        idx = np.concatenate([m[rng.integers(0, len(m), size=len(m))] for m in members])   # within each group
        draws.append(gap(y[idx], p[idx], g[idx]))
    low, high = np.percentile(draws, [2.5, 97.5])
    return gap(y, p, g), float(low), float(high)


ok_24 = {}
TWIN_Y_24 = np.concatenate([YA_VAL[:400], YA_VAL[:400]])          # two identical groups of 400 people
TWIN_P_24 = np.concatenate([PRED_VAL[:400], PRED_VAL[:400]])
TWIN_G_24 = np.array(["a"] * 400 + ["b"] * 400)


with checked("B6.24"):
    start_24 = time.perf_counter()
    lo_24, hi_24 = rate_ci_24(YA_VAL[SEX_VAL == "Female"].copy(), PRED_VAL[SEX_VAL == "Female"].copy(), "tpr",
                              n_boot=N_BOOT_24, rng=np.random.default_rng(0))
    m_24 = (SEX_VAL == "Female") & (YA_VAL == 1)
    tpr_f_24 = PRED_VAL[m_24].mean()
    half_24 = 1.96 * math.sqrt(tpr_f_24 * (1 - tpr_f_24) / m_24.sum())          # the normal approximation
    verdict("B6.24", lo_24 <= tpr_f_24 <= hi_24 and 0.7 < (hi_24 - lo_24) / (2 * half_24) < 1.3,
            f"rate_ci_24 : TPR des femmes {fr(tpr_f_24, 3)}, intervalle [{fr(lo_24, 3)} ; {fr(hi_24, 3)}], de la largeur "
            "attendue (approximation normale ± 30 %).",
            f"rate_ci_24 : TPR des femmes {fr(tpr_f_24, 3)}, intervalle [{fr(lo_24, 3)} ; {fr(hi_24, 3)}] ; attendu un "
            f"intervalle qui le contient, large d'environ {fr(2 * half_24, 3)} (approximation normale) : sur quelles lignes "
            "tires-tu, et quels centiles gardes-tu ?")
    for name_24, groups_24 in (("sexe", SEX_VAL), ("âge", AGE_VAL)):
        table_24 = audit_24(YA_VAL.copy(), PRED_VAL.copy(), groups_24.copy(), n_boot=N_BOOT_24, seed=0)
        if not isinstance(table_24, pd.DataFrame) or list(table_24.index) != list(np.unique(groups_24)):
            print(f"❌ Ex B6.24 : audit_24 doit rendre un DataFrame indexé par les groupes, dans l'ordre de np.unique "
                  f"({list(np.unique(groups_24))}), reçu {type(table_24).__name__} "
                  f"{list(getattr(table_24, 'index', []))[:6]}.")
            raise StopCheck
        missing_24 = [c for c in COLUMNS_24 if c not in table_24.columns]
        if missing_24:
            print(f"❌ Ex B6.24 : audit_24 ({name_24}) : il manque les colonnes {missing_24}.")
            raise StopCheck
        want_24 = {k: {c: r[c] for c in ("n", "positives", "selection_rate", "tpr", "fpr")}
                   for k, r in sk_audit(YA_VAL, PRED_VAL, groups_24)[0].items()}
        bad_24 = [f"{c}[{k}] = {table_24.loc[k, c]} au lieu de {fr(float(v), 4)}" for k, w in want_24.items()
                  for c, v in w.items() if not np.isclose(float(table_24.loc[k, c]), float(v))]
        if bad_24:
            print(f"❌ Ex B6.24 : audit_24 ({name_24}) : " + " ; ".join(bad_24[:3]) + ".")
            raise StopCheck
        outside_24 = [f"{rate}[{k}]" for k in want_24 for rate in ("selection_rate", "tpr", "fpr")
                      if not table_24.loc[k, f"{rate}_low"] - 1e-9 <= table_24.loc[k, rate] <= table_24.loc[k, f"{rate}_high"] + 1e-9]
        widths_24 = []
        for k, w in want_24.items():
            m_k = groups_24 == k
            den_24 = {"selection_rate": m_k.sum(), "tpr": (m_k & (YA_VAL == 1)).sum(), "fpr": (m_k & (YA_VAL == 0)).sum()}
            for rate in ("selection_rate", "tpr", "fpr"):
                r_24 = float(w[rate])
                if den_24[rate] >= 100 and 0.05 < r_24 < 0.95:
                    widths_24.append(((table_24.loc[k, f"{rate}_high"] - table_24.loc[k, f"{rate}_low"])
                                      / (2 * 1.96 * math.sqrt(r_24 * (1 - r_24) / den_24[rate])), f"{rate}[{k}]"))
        odd_24 = [label for ratio, label in widths_24 if not 0.7 < ratio < 1.3]
        verdict("B6.24", not outside_24 and not odd_24,
                f"audit_24 ({name_24}) : taux de scikit-learn, intervalles qui les contiennent et de la largeur attendue.",
                f"audit_24 ({name_24}) : les taux sont justes, mais les intervalles de {(outside_24 + odd_24)[:3]} "
                + ("ne contiennent pas leur taux" if outside_24 else "n'ont pas la largeur attendue (approximation normale "
                   "± 30 %) : tires-tu dans les lignes du groupe, et gardes-tu les centiles 2,5 et 97,5 ?") + ".")
        if not outside_24 and not odd_24:
            ok_24[name_24] = table_24
    again_24 = audit_24(YA_VAL.copy(), PRED_VAL.copy(), SEX_VAL.copy(), n_boot=N_BOOT_24, seed=0)
    other_24 = audit_24(YA_VAL.copy(), PRED_VAL.copy(), SEX_VAL.copy(), n_boot=N_BOOT_24, seed=1)
    verdict("B6.24", again_24.equals(ok_24.get("sexe", again_24)) and not np.allclose(
        other_24["tpr_low"].to_numpy(float), again_24["tpr_low"].to_numpy(float)),
        "la même graine redonne le même tableau, une autre graine d'autres intervalles.",
        "audit_24 doit être reproductible (même graine, même tableau) et ses intervalles changer avec la graine.")
    twin_24 = audit_24(TWIN_Y_24.copy(), TWIN_P_24.copy(), TWIN_G_24.copy(), n_boot=50, seed=0)
    bounds_24 = ["selection_rate_low", "selection_rate_high", "tpr_low", "tpr_high", "fpr_low", "fpr_high"]
    verdict("B6.24", not np.allclose(twin_24.loc["a", bounds_24].to_numpy(float), twin_24.loc["b", bounds_24].to_numpy(float)),
            "deux groupes identiques ont des intervalles différents : un seul générateur pour tout le tableau.",
            "audit_24 donne exactement les mêmes intervalles à deux groupes identiques : chaque groupe repart-il de la "
            "même graine ? Le contrat veut un seul rng = np.random.default_rng(seed) pour tout le tableau.")

with checked("B6.24"):
    if len(ok_24) < 2:
        raise StopCheck
    want_s_24 = sk_audit(YA_VAL, PRED_VAL, SEX_VAL)[0]
    tpr_gap_24 = want_s_24["Male"]["tpr"] - want_s_24["Female"]["tpr"]
    gap_24 = gap_ci_24(YA_VAL.copy(), PRED_VAL.copy(), SEX_VAL.copy(), "tpr", n_boot=N_BOOT_24, seed=0)
    if not returned("B6.24", "gap_ci_24", gap_24):
        raise StopCheck
    g_24, l_24, h_24 = (float(v) for v in gap_24)
    half_g_24 = 1.96 * math.sqrt(sum(w["tpr"] * (1 - w["tpr"]) / w["positives"] for w in want_s_24.values()))
    verdict("B6.24", math.isclose(g_24, abs(tpr_gap_24)) and l_24 <= g_24 <= h_24 and 0.7 < (h_24 - l_24) / (2 * half_g_24) < 1.4,
            f"gap_ci_24 : écart de TPR {fr(g_24, 3)}, intervalle [{fr(l_24, 3)} ; {fr(h_24, 3)}].",
            f"gap_ci_24 (« tpr ») rend {fr(g_24, 3)} [{fr(l_24, 3)} ; {fr(h_24, 3)}] ; attendu l'écart {fr(abs(tpr_gap_24), 3)} "
            f"(le plus grand moins le plus petit TPR), dans un intervalle large d'environ {fr(2 * half_g_24, 3)} : "
            "rééchantillonnes-tu chaque groupe dans ses propres lignes ?")
    for kind_24, want_k_24 in (("dp", sk_audit(YA_VAL, PRED_VAL, SEX_VAL)[1]["dp"]),
                               ("eo", sk_audit(YA_VAL, PRED_VAL, SEX_VAL)[1]["eo"])):
        gk_24 = [float(v) for v in gap_ci_24(YA_VAL.copy(), PRED_VAL.copy(), SEX_VAL.copy(), kind_24, n_boot=100, seed=0)]
        verdict("B6.24", math.isclose(gk_24[0], float(want_k_24)) and gk_24[1] <= gk_24[0] <= gk_24[2],
                f"gap_ci_24 (« {kind_24} ») : {fr(gk_24[0], 3)}, dans son intervalle.",
                f"gap_ci_24 (« {kind_24} ») rend {[fr(v, 3) for v in gk_24]} ; attendu l'écart de ta fonction de B6.23 "
                f"({fr(float(want_k_24), 3)}, celui de scikit-learn), dans son intervalle.")
    twin_gap_24 = [float(v) for v in gap_ci_24(TWIN_Y_24.copy(), TWIN_P_24.copy(), TWIN_G_24.copy(), "tpr", n_boot=50, seed=0)]
    verdict("B6.24", twin_gap_24[2] > 0, "deux groupes identiques reçoivent des tirages différents (un seul générateur).",
            "gap_ci_24 : sur deux groupes identiques, tous tes tirages donnent un écart nul : chaque groupe repart-il de "
            "la même graine ? Le contrat veut un seul rng = np.random.default_rng(seed) pour tous les tirages.")
    real_dp_24, sizes_24 = mylearn.explain.demographic_parity_difference, []

    def spy_dp_24(*args, **kwargs):              # what your gap_ci_24 passes to your demographic_parity_difference
        groups = kwargs["groups"] if "groups" in kwargs else args[1]
        sizes_24.append(tuple(np.unique(np.asarray(groups), return_counts=True)[1].tolist()))
        return real_dp_24(*args, **kwargs)

    try:
        mylearn.explain.demographic_parity_difference = spy_dp_24
        gap_ci_24(YA_VAL.copy(), PRED_VAL.copy(), SEX_VAL.copy(), "dp", n_boot=30, seed=0)
    finally:
        mylearn.explain.demographic_parity_difference = real_dp_24
    want_sizes_24 = tuple(np.unique(SEX_VAL, return_counts=True)[1].tolist())
    if len(sizes_24) > 1:                        # (seen only when gap_ci_24 calls mylearn.explain.<function>)
        odd_sizes_24 = next((s for s in sizes_24 if s != want_sizes_24), None)
        verdict("B6.24", odd_sizes_24 is None,
                "gap_ci_24 : chaque tirage garde l'effectif de chaque groupe (un bootstrap stratifié).",
                f"gap_ci_24 : un de tes tirages a des groupes de {odd_sizes_24} personnes au lieu de {want_sizes_24} : le "
                "contrat tire les lignes de chaque groupe dans son propre groupe, pour que chaque tirage garde ses effectifs.")
    ok_24["gap"] = True
    print(f"ℹ️ {time.perf_counter() - start_24:.1f} s")

In [ ]:
# The audit of the reference model on the validation part (threshold 0.5), by sex and by age band, with your functions
if len(ok_24) == 3:
    SHOW_24 = ["n", "positives", "base_rate", "selection_rate", "tpr", "fpr", "precision"]

    def with_ci_24(table):
        out = table[["n", "positives"]].copy()
        for rate in ("base_rate", "selection_rate", "tpr", "fpr", "precision"):
            out[rate] = table[rate].map(lambda v: fr(v, 3))
            if f"{rate}_low" in table:
                out[rate] = [f"{fr(v, 3)} [{fr(lo, 3)} ; {fr(hi, 3)}]" for v, lo, hi in
                             zip(table[rate], table[f"{rate}_low"], table[f"{rate}_high"])]
        return out

    AUDIT_24 = {"sexe": ok_24["sexe"], "âge": ok_24["âge"]}
    for name_24, table_24 in AUDIT_24.items():
        print(f"audit par {name_24} (validation, {N_BOOT_24} tirages ; [intervalle à 95 %]) :")
        print(with_ci_24(table_24).to_string())
        print()
    GAPS_24 = {}
    for name_24, groups_24 in (("sexe", SEX_VAL), ("âge", AGE_VAL)):
        for kind_24 in ("dp", "eo", "tpr", "fpr"):
            GAPS_24[name_24, kind_24] = gap_ci_24(YA_VAL, PRED_VAL, groups_24, kind_24, n_boot=N_BOOT_24, seed=0)
    labels_24 = {"dp": "ΔDP (sélection)", "eo": "ΔEO", "tpr": "écart de TPR", "fpr": "écart de FPR"}
    for name_24 in ("sexe", "âge"):
        print(f"écarts par {name_24} : " + " ; ".join(f"{labels_24[k]} {fr(GAPS_24[name_24, k][0], 3)} "
                                                       f"[{fr(GAPS_24[name_24, k][1], 3)} ; {fr(GAPS_24[name_24, k][2], 3)}]"
                                                       for k in labels_24))
    fig_24, axes_24 = plt.subplots(1, 3, figsize=(14, 3.6))
    for ax_24, rate_24, title_24 in zip(axes_24, ("selection_rate", "tpr", "fpr"), ("taux de sélection", "TPR", "FPR")):
        tables_24 = [ok_24["sexe"], ok_24["âge"]]
        names_24 = [*ok_24["sexe"].index, *ok_24["âge"].index]
        values_24 = np.concatenate([t[rate_24].to_numpy(float) for t in tables_24])
        lows_24 = np.concatenate([t[f"{rate_24}_low"].to_numpy(float) for t in tables_24])
        highs_24 = np.concatenate([t[f"{rate_24}_high"].to_numpy(float) for t in tables_24])
        colors_24 = ["C1"] * len(ok_24["sexe"]) + ["C0"] * len(ok_24["âge"])
        ax_24.errorbar(range(len(names_24)), values_24, yerr=[values_24 - lows_24, highs_24 - values_24], fmt="none",
                       ecolor="gray", capsize=3)
        ax_24.scatter(range(len(names_24)), values_24, c=colors_24, zorder=3)
        ax_24.set_xticks(range(len(names_24)), names_24, rotation=45, fontsize=8)
        ax_24.set_title(f"{title_24} par groupe (intervalles à 95 %)", fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print("⏳ Ex B6.24 : l'audit attend tes trois fonctions (cellule précédente) et ta librairie (B6.23, ch. 2).")

### Ex B6.25 — Lire un tableau d'audit : où est l'écart, est-il significatif, que conclure ? 📈 ★★ ⏱️ 15 min
**Objectif :** lire un tableau d'audit avec effectifs et intervalles, et en tirer une conclusion prudente.  
**Prérequis :** Ex B6.24 · fiche §206.9 (« Des écarts avec leurs intervalles », « L'impossibilité ») · **Parcours :** C

Le tableau suivant est l'audit du modèle de référence de la partie (`HGB_C`, seuil 0,5) sur la validation, par tranche d'âge et par sexe, avec 2 000 tirages bootstrap (la solution de B6.24, calculée une fois) ; le second tableau donne les écarts avec leur intervalle à 95 %. Réponds **à partir du tableau** :
- a) `lowest_tpr_25` : la lettre de la tranche d'âge au plus petit TPR : `"A"` <25, `"B"` 25-34, `"C"` 35-44, `"D"` 45-54, `"E"` 55-64, `"F"` 65+ ;
- b) `positives_25` : sur combien de positifs de la validation ce TPR repose-t-il ? (un entier)
- c) `widest_25` : parmi les **cinq autres** tranches, la lettre de celle dont l'intervalle du TPR est le plus **large** (mêmes lettres) ;
- d) `tpr_sex_25` : l'intervalle à 95 % de l'écart de TPR entre les sexes exclut-il 0 ? `True` ou `False` ;
- e) `overlap_25` : les intervalles du TPR des femmes et des hommes se chevauchent-ils ? `True` ou `False` ;
- f) `conclusion_25` : la conclusion que le tableau permet :
  - `"A"` le modèle est équitable par âge, puisque les intervalles des TPR des tranches se chevauchent deux à deux ;
  - `"B"` il faut égaliser les taux de sélection des tranches d'âge, puisque ΔDP par âge est le plus grand écart du tableau ;
  - `"C"` le TPR le plus bas repose sur peu de positifs et son intervalle est large : l'écart de TPR par âge est mesurable, sa taille très incertaine, et les écarts de sélection suivent surtout les taux de base ;
  - `"D"` les écarts par sexe ne sont pas significatifs, car les deux groupes sont grands.

Dans tes notes : pourquoi un intervalle large ne veut-il pas dire « pas d'écart » ? Quelle ligne rapporterais-tu en premier à un métier, avec quelle phrase ? Avec six tranches, combien de comparaisons deux à deux ferais-tu, et qu'en dit 206.9 (les comparaisons multiples) ?

In [ ]:
# The audit table to read: the reference model of part C on the validation part (threshold 0.5), by sex and by age
# band, 2 000 bootstrap draws (the solution of B6.24, computed once by tools/chapters/history_chB6.py)
TABLE_25 = pd.DataFrame({"groupe": ["<25", "25-34", "35-44", "45-54", "55-64", "65+", "femmes", "hommes"], "n": [1647, 2596, 2422, 1708, 983, 413, 3165, 6604], "positifs": [12, 471, 790, 661, 314, 89, 350, 1987], "taux de base": ["0,007", "0,181", "0,326", "0,387", "0,319", "0,215", "0,111", "0,301"], "sélection [IC 95 %]": ["0,003 [0,001 ; 0,006]", "0,124 [0,112 ; 0,138]", "0,289 [0,272 ; 0,308]", "0,347 [0,325 ; 0,371]", "0,272 [0,244 ; 0,299]", "0,136 [0,102 ; 0,167]", "0,076 [0,067 ; 0,087]", "0,258 [0,247 ; 0,268]"], "TPR [IC 95 %]": ["0,417 [0,125 ; 0,714]", "0,535 [0,490 ; 0,582]", "0,684 [0,650 ; 0,717]", "0,716 [0,681 ; 0,750]", "0,631 [0,579 ; 0,684]", "0,551 [0,448 ; 0,651]", "0,569 [0,516 ; 0,619]", "0,663 [0,642 ; 0,684]"], "FPR [IC 95 %]": ["0,000 [0,000 ; 0,000]", "0,033 [0,026 ; 0,041]", "0,099 [0,084 ; 0,113]", "0,115 [0,096 ; 0,135]", "0,103 [0,080 ; 0,128]", "0,022 [0,006 ; 0,039]", "0,015 [0,011 ; 0,020]", "0,083 [0,075 ; 0,091]"]}).set_index("groupe")
GAPS_25 = pd.DataFrame({"écart": ["ΔDP (sélection)", "ΔEO", "écart de TPR", "écart de FPR"], "par sexe [IC 95 %]": ["0,181 [0,167 ; 0,196]", "0,095 [0,064 ; 0,153]", "0,095 [0,042 ; 0,153]", "0,068 [0,059 ; 0,077]"], "par âge [IC 95 %]": ["0,344 [0,321 ; 0,368]", "0,299 [0,164 ; 0,578]", "0,299 [0,164 ; 0,578]", "0,115 [0,101 ; 0,136]"]}).set_index("écart")
print(TABLE_25.to_string())
print()
print(GAPS_25.to_string())

In [ ]:
lowest_tpr_25, positives_25, widest_25 = "A", 12, "F"
tpr_sex_25, overlap_25, conclusion_25 = True, False, "C"

In [ ]:
wb.record("B6.25a", "A", mistakes={"suis la colonne du TPR, tranche par tranche": {"B", "C", "D", "E", "F"}})
wb.record("B6.25b", 12, fractional="un nombre de personnes est un entier : lis la colonne des positifs", mistakes={"relis la question : on demande les positifs (y = 1), pas tout l'effectif de la tranche": 1647})
wb.record("B6.25c", "F", mistakes={"relis la question : on cherche parmi les cinq autres tranches que celle de a)": "A", "compare la largeur (borne haute moins borne basse) des intervalles du TPR des cinq autres tranches": {"B", "C", "D", "E"}})
wb.record("B6.25d", True, mistakes={"lis les deux bornes de l'intervalle de l'écart de TPR par sexe : 0 est-il entre elles ?": False})
wb.record("B6.25e", False, mistakes={"compare la borne haute de l'un et la borne basse de l'autre, dans la colonne du TPR": True})
wb.record("B6.25f", "C", mistakes={"des intervalles qui se chevauchent prouvent-ils l'égalité ? Regarde l'intervalle de l'écart": "A", "un écart de sélection suit aussi les taux de base : le tableau dit-il qu'il faut les égaliser ?": "B", "lis l'intervalle des écarts par sexe : 0 y est-il ?": "D"})

### Ex B6.26 — Prédire : retirer la variable « sexe » rend-il le modèle équitable ? 🔮 ★ ⏱️ 10 min
**Objectif :** prédire de combien le retrait d'un attribut sensible change les écarts d'un modèle, et ce qui le remplace.  
**Prérequis :** Ex B6.24 · MP3 (MP3.7) · fiche §206.8 (« L'équité par aveuglement ») et §206.10 · 💼 B6.E5 · **Parcours :** R, C

Retirer `sex` des features et réentraîner : le modèle devient-il équitable ? La fiche dit seulement que cela se **mesure** (206.8, « L'équité par aveuglement »). L'expérience mesure, sur la validation, quatre grandeurs avec la recette du modèle de référence (`adult_pipeline`). **Avant d'exécuter**, écris tes quatre prédictions :
- a) `proxy_auc_26` : la ROC-AUC d'un gradient boosting (la même recette) qui prédit le **sexe** à partir des 11 autres features du pipeline (`fnlwgt` et `education` en sont déjà écartés) : `"A"` moins de 0,6 ; `"B"` de 0,6 à 0,75 ; `"C"` de 0,75 à 0,9 ; `"D"` plus de 0,9 ;
- b) `proxy_auc_few_26` : la même chose sans `relationship` ni `marital-status` (9 features) : mêmes lettres ;
- c) `eo_ratio_26` : le ΔEO par sexe du modèle réentraîné sans `sex`, divisé par celui du modèle de référence : `"A"` moins de 0,5 (l'écart baisse de plus de moitié) ; `"B"` de 0,5 à 1,5 ; `"C"` plus de 1,5 ;
- d) `four_fifths_26` : pour le modèle sans `sex`, le taux de sélection des femmes divisé par celui des hommes (la règle des quatre cinquièmes, 206.9) : `"A"` moins de 0,4 ; `"B"` de 0,4 à 0,6 ; `"C"` de 0,6 à 0,8 ; `"D"` 0,8 ou plus.

La cellule d'expérience n'affiche rien tant que tes quatre prédictions ne sont pas écrites ; elle entraîne trois modèles (≈ 10 s), affiche les deux AUC et un tableau des taux par sexe avec et sans `sex`, puis vérifie tes prédictions. Rien n'y dépend de ton code.

Dans tes notes, avant d'exécuter : quelles features d'*Adult* portent, selon toi, l'information du sexe, et comment le vérifierais-tu ?

In [ ]:
proxy_auc_26, proxy_auc_few_26, eo_ratio_26, four_fifths_26 = "D", "C", "B", "A"

In [ ]:
wb.record("B6.26a", "D", mistakes={"lis la ROC-AUC du modèle qui prédit le sexe à partir des 11 autres features": {"A", "B", "C"}}, prediction=True)
wb.record("B6.26b", "C", mistakes={"lis la ROC-AUC sans relationship ni marital-status": {"A", "B", "D"}}, prediction=True)
wb.record("B6.26c", "B", mistakes={"lis le rapport des deux ΔEO, sous le tableau": {"A", "C"}}, prediction=True)
wb.record("B6.26d", "A", mistakes={"lis la colonne F / M du modèle sans sex": {"B", "C", "D"}}, prediction=True)

In [ ]:
if any(answer is ... or answer is None for answer in [proxy_auc_26, proxy_auc_few_26, eo_ratio_26, four_fifths_26]):
    print("⏳ Ex B6.26 : écris d'abord tes quatre prédictions dans la cellule précédente.")
else:
    start_26 = time.perf_counter()
    NOSEX_26 = adult_pipeline(drop=("sex",)).fit(XA_TRAIN, YA_TRAIN)              # the reference recipe, without sex
    PRED_NOSEX_26 = (NOSEX_26.predict_proba(XA_VAL)[:, 1] >= 0.5).astype(int)
    female_tr_26, female_va_26 = (XA_TRAIN["sex"] == "Female").to_numpy(int), (SEX_VAL == "Female").astype(int)
    AUC_26 = {}
    for name_26, drop_26 in (("les 11 autres features", ("sex",)),
                             ("sans relationship ni marital-status", ("sex", "relationship", "marital-status"))):
        spy_26 = adult_pipeline(drop=drop_26).fit(XA_TRAIN, female_tr_26)          # predicts the sex
        AUC_26[name_26] = roc_auc_score(female_va_26, spy_26.predict_proba(XA_VAL)[:, 1])
    rows_26 = {}
    for name_26, pred_26 in (("avec sex (référence)", PRED_VAL), ("sans sex", PRED_NOSEX_26)):
        rates_26, gaps_26 = sk_audit(YA_VAL, pred_26, SEX_VAL)
        rows_26[name_26] = {"accuracy": np.mean(pred_26 == YA_VAL), "sélection F": rates_26["Female"]["selection_rate"],
                            "sélection M": rates_26["Male"]["selection_rate"],
                            "F / M": rates_26["Female"]["selection_rate"] / rates_26["Male"]["selection_rate"],
                            "TPR F": rates_26["Female"]["tpr"], "TPR M": rates_26["Male"]["tpr"],
                            "FPR F": rates_26["Female"]["fpr"], "FPR M": rates_26["Male"]["fpr"],
                            "ΔDP": gaps_26["dp"], "ΔEO": gaps_26["eo"]}
    print("prédire le sexe à partir des autres features (validation) : " + " ; ".join(
        f"{name} : ROC-AUC {fr(v, 3)}" for name, v in AUC_26.items()))
    print(pd.DataFrame(rows_26).T.map(lambda v: fr(v, 3)).to_string())
    eo_ratio_value_26 = rows_26["sans sex"]["ΔEO"] / rows_26["avec sex (référence)"]["ΔEO"]
    print(f"ΔEO sans sex / ΔEO avec : {fr(eo_ratio_value_26, 2)} ; ℹ️ {time.perf_counter() - start_26:.0f} s")
    print()
    for id_26, answer_26 in (("B6.26a", proxy_auc_26), ("B6.26b", proxy_auc_few_26), ("B6.26c", eo_ratio_26),
                             ("B6.26d", four_fifths_26)):
        wb.check(id_26, answer_26)

Dans tes notes : pour chaque prédiction ratée, qu'est-ce qui t'a trompé ? Quelles colonnes portent l'information du sexe, et que reste-t-il sans elles ? Si le modèle n'utilise pas `sex`, d'où viennent les écarts qui restent : des variables de substitution, des taux de base, des labels ? Que perdrait-on, pour l'audit et pour la correction, en effaçant `sex` des données ?

### Ex B6.27 — Atténuer : retrait des variables de substitution, repondération, seuils par groupe, et ce que cela coûte 🔬 ★★★ ⏱️ 50 min
**Objectif :** comparer trois familles de corrections d'équité sur un même modèle, réauditer tous les groupes et chiffrer le coût en accuracy.  
**Prérequis :** Ex B6.24, B6.26 · fiche §206.10 (« Trois moments pour agir », « Le prix à payer ») · **Parcours :** R, C

Trois familles de corrections (206.10), essayées sur le même modèle, et leur coût. **Contrat** :
- `reweigh_27(groups, y)` rend le poids $w(g, y) = P(g)\,P(y) / P(g, y)$ de chaque exemple (Kamiran et Calders), les probabilités estimées par les proportions dans ces données ;
- `tpr_thresholds_27(scores, y, groups, target_tpr)` rend `{groupe: seuil}` (les groupes de `np.unique`) : pour chaque groupe, le quantile `1 - target_tpr` des scores de ses **positifs** (`np.quantile`, comme dans 206.10), pour que son TPR vaille à peu près `target_tpr` sur ces données ;
- `decide_27(scores, groups, thresholds)` rend les décisions 0/1 (des entiers) : 1 quand le score atteint le seuil du groupe de l'exemple (`>=`).

La cellule donnée coupe la validation en deux moitiés : `CAL_27`, où se choisissent les seuils, et `EVAL_27`, où tout se mesure (choisir et mesurer sur les mêmes personnes flatterait les seuils). La vérification essaie tes trois fonctions sur de petits cas. L'expérience (≈ 20 s) compare cinq stratégies, toutes mesurées sur `EVAL_27` : le modèle de référence au seuil 0,5 ; réentraîné sans `sex` ; sans `sex`, `relationship` ni `marital-status` (des variables de substitution, B6.26) ; repondéré sur le sexe et le label (`fit(..., model__sample_weight=w)`) ; des seuils par sexe choisis sur `CAL_27` pour le TPR qu'y obtient la référence. Pour chacune : l'accuracy, ΔDP et ΔEO par sexe (avec leur intervalle, ta `gap_ci_24` de B6.24, 200 tirages), ΔDP et ΔEO par tranche d'âge (l'audit refait après chaque correction), et les TPR et FPR des femmes et des hommes. **Ne présume pas l'issue : mesure-la.**

Dans tes notes : quelle stratégie réduit le plus ΔEO par sexe, et à quel coût en accuracy ? Les seuils par sexe rapprochent les TPR : que deviennent les FPR, et donc ΔEO ? La repondération change-t-elle les écarts autant que tu l'attendais ? Une correction par sexe déplace-t-elle les écarts par âge ? Lesquelles de ces stratégies utilisent le sexe dans la **décision**, et qu'en dit 206.10 ? Laquelle recommanderais-tu pour la statistique publique du MP3, et pourquoi ?

In [ ]:
# The validation part cut in two halves: the thresholds by group are chosen on CAL_27, every strategy is measured on
# EVAL_27 (choosing and measuring on the same people would flatter the thresholds)
order_27 = np.random.default_rng(27).permutation(len(XA_VAL))
CAL_27, EVAL_27 = np.sort(order_27[: len(order_27) // 2]), np.sort(order_27[len(order_27) // 2:])
print(f"moitié de calibrage : {len(CAL_27)} personnes ; moitié d'évaluation : {len(EVAL_27)} personnes")

In [ ]:
def reweigh_27(groups, y):
    """np.ndarray (n,): the weight w(g, y) = P(g) P(y) / P(g, y) of each example (Kamiran and Calders, 206.10), the
    probabilities estimated by the shares in these data."""
    g, y = np.asarray(groups), np.asarray(y)
    w = np.empty(len(y), dtype=float)
    for key in np.unique(g):
        for label in np.unique(y):
            cell = (g == key) & (y == label)
            if cell.any():
                w[cell] = np.mean(g == key) * np.mean(y == label) / np.mean(cell)
    return w


def tpr_thresholds_27(scores, y, groups, target_tpr):
    """{group: threshold} (the groups of np.unique): for each group, the quantile 1 - target_tpr of the scores of its
    positives (np.quantile, as in 206.10), so that its TPR on these data is about target_tpr."""
    s, y, g = np.asarray(scores, dtype=float), np.asarray(y), np.asarray(groups)
    return {key: float(np.quantile(s[(g == key) & (y == 1)], 1 - target_tpr)) for key in np.unique(g)}


def decide_27(scores, groups, thresholds):
    """np.ndarray of 0/1 ints: 1 when the score of an example reaches the threshold of its group
    (scores >= thresholds[group])."""
    s, g = np.asarray(scores, dtype=float), np.asarray(groups)
    return (s >= np.array([thresholds[key] for key in g])).astype(int)


ok_27 = {}
with checked("B6.27"):
    g_27 = np.array(["u"] * 6 + ["v"] * 4)
    y_27 = np.array([1, 1, 1, 1, 0, 0, 1, 0, 0, 0])
    w_27 = reweigh_27(g_27.copy(), y_27.copy())
    if returned("B6.27", "reweigh_27", w_27):
        w_27 = np.asarray(w_27, dtype=float)
        # P(u) = 0.6, P(y=1) = 0.5: u-positives 0.6 * 0.5 / 0.4 = 0.75, u-negatives 0.6 * 0.5 / 0.2 = 1.5, ...
        want_w_27 = np.array([0.75] * 4 + [1.5] * 2 + [2.0] + [2 / 3] * 3)
        if w_27.shape != want_w_27.shape:
            print(f"❌ Ex B6.27 : reweigh_27 doit rendre un poids par exemple, forme (10,), reçu {w_27.shape}.")
        elif np.allclose(w_27, 1 / want_w_27):
            print("❌ Ex B6.27 : ce sont les inverses des poids : w(g, y) = P(g) P(y) / P(g, y), les cases rares pèsent plus.")
        else:
            ok_27["reweigh"] = same("B6.27", "reweigh_27 sur deux groupes de 6 et 4", w_27, want_w_27, rtol=1e-9, atol=1e-12)
        base_27 = [np.average(y_27[g_27 == k], weights=w_27[g_27 == k]) for k in ("u", "v")] if w_27.shape == (10,) else []
        verdict("B6.27", len(base_27) == 2 and np.allclose(base_27, 0.5),
                "une fois repondérés, les deux groupes ont le même taux de base (0,5).",
                "une fois repondérés, les deux groupes doivent avoir le même taux de base que l'ensemble des données.")

with checked("B6.27"):
    rng_27 = np.random.default_rng(270)
    s_27 = rng_27.random(400)
    gg_27 = np.where(rng_27.random(400) < 0.4, "a", "b")
    yy_27 = (rng_27.random(400) < np.where(gg_27 == "a", 0.3, 0.5)).astype(int)
    thr_27 = tpr_thresholds_27(s_27.copy(), yy_27.copy(), gg_27.copy(), 0.75)
    if returned("B6.27", "tpr_thresholds_27", thr_27):
        if not isinstance(thr_27, dict) or sorted(thr_27) != ["a", "b"]:
            print(f"❌ Ex B6.27 : tpr_thresholds_27 doit rendre un dictionnaire {{groupe: seuil}}, reçu {type(thr_27).__name__}.")
            raise StopCheck
        want_t_27 = {k: np.quantile(s_27[(gg_27 == k) & (yy_27 == 1)], 0.25) for k in ("a", "b")}
        if all(np.isclose(thr_27[k], np.quantile(s_27[(gg_27 == k) & (yy_27 == 1)], 0.75)) for k in "ab"):
            print("❌ Ex B6.27 : c'est le quantile target_tpr : pour garder 75 % des positifs, le seuil laisse 25 % des scores "
                  "des positifs au-dessous de lui.")
        elif all(np.isclose(thr_27[k], np.quantile(s_27[gg_27 == k], 0.25)) for k in "ab"):
            print("❌ Ex B6.27 : le quantile est pris sur tous les scores du groupe : le TPR ne regarde que ses positifs.")
        else:
            ok_27["thresholds"] = same("B6.27", "tpr_thresholds_27 (TPR visé 0,75)", [thr_27["a"], thr_27["b"]],
                                       [want_t_27["a"], want_t_27["b"]], rtol=1e-9, atol=1e-12)
    d_27 = decide_27(np.array([0.2, 0.5, 0.7, 0.5, 0.9]), np.array(["a", "a", "a", "b", "b"]), {"a": 0.5, "b": 0.6})
    if returned("B6.27", "decide_27", d_27):
        ok_27["decide"] = same("B6.27", "decide_27 (un seuil égal au score décide 1)", np.asarray(d_27).astype(int),
                               [0, 1, 1, 0, 1])

In [ ]:
# Five strategies, measured on EVAL_27: accuracy, the gaps by sex (with their interval, your gap_ci_24 of B6.24) and
# the gaps by age band, re-audited after each correction
if len(ok_27) == 3:
    start_27 = time.perf_counter()
    nosex_27 = globals().get("NOSEX_26")                                  # the model of B6.26, if it ran
    if nosex_27 is None:
        nosex_27 = adult_pipeline(drop=("sex",)).fit(XA_TRAIN, YA_TRAIN)
    noproxy_27 = adult_pipeline(drop=("sex", "relationship", "marital-status")).fit(XA_TRAIN, YA_TRAIN)
    w_27 = reweigh_27(XA_TRAIN["sex"].to_numpy(), YA_TRAIN)
    reweighed_27 = adult_pipeline().fit(XA_TRAIN, YA_TRAIN, model__sample_weight=w_27)
    target_27 = float(PRED_VAL[CAL_27][YA_VAL[CAL_27] == 1].mean())        # the TPR of the reference on CAL_27
    thr_27 = tpr_thresholds_27(P_VAL[CAL_27], YA_VAL[CAL_27], SEX_VAL[CAL_27], target_27)
    DECISIONS_27 = {
        "référence (seuil 0,5)": PRED_VAL,
        "sans sex": (nosex_27.predict_proba(XA_VAL)[:, 1] >= 0.5).astype(int),
        "sans sex, relationship, marital-status": (noproxy_27.predict_proba(XA_VAL)[:, 1] >= 0.5).astype(int),
        "repondération (sexe × label)": (reweighed_27.predict_proba(XA_VAL)[:, 1] >= 0.5).astype(int),
        "seuils par sexe (même TPR)": np.asarray(decide_27(P_VAL, SEX_VAL, thr_27)).astype(int),
    }
    print("seuils par sexe, choisis sur CAL_27 pour un TPR de", fr(target_27, 3), ":",
          ", ".join(f"{k} {fr(v, 3)}" for k, v in thr_27.items()))
    y_e_27, s_e_27, a_e_27 = YA_VAL[EVAL_27], SEX_VAL[EVAL_27], AGE_VAL[EVAL_27]
    use_ci_27 = len(globals().get("ok_24", {})) == 3
    if not use_ci_27:
        print("⏳ Ex B6.27 : les intervalles attendent ta gap_ci_24 (B6.24) : seuls les écarts sont affichés.")
    TABLE_27 = {}
    for name_27, pred_27 in DECISIONS_27.items():
        p_e_27 = pred_27[EVAL_27]
        row_27 = {"accuracy": fr(np.mean(p_e_27 == y_e_27), 4)}
        for label_27, groups_27 in (("sexe", s_e_27), ("âge", a_e_27)):
            for kind_27 in ("dp", "eo"):
                if use_ci_27 and label_27 == "sexe":
                    g_27, lo_27, hi_27 = gap_ci_24(y_e_27, p_e_27, groups_27, kind_27, n_boot=200, seed=0)
                    row_27[f"Δ{kind_27.upper()} {label_27}"] = f"{fr(g_27, 3)} [{fr(lo_27, 3)} ; {fr(hi_27, 3)}]"
                else:
                    row_27[f"Δ{kind_27.upper()} {label_27}"] = fr(sk_audit(y_e_27, p_e_27, groups_27)[1][kind_27], 3)
        rates_27 = sk_audit(y_e_27, p_e_27, s_e_27)[0]
        row_27["TPR F / M"] = f"{fr(rates_27['Female']['tpr'], 2)} / {fr(rates_27['Male']['tpr'], 2)}"
        row_27["FPR F / M"] = f"{fr(rates_27['Female']['fpr'], 3)} / {fr(rates_27['Male']['fpr'], 3)}"
        TABLE_27[name_27] = row_27
    print(pd.DataFrame(TABLE_27).T.to_string())
    print(f"ℹ️ {time.perf_counter() - start_27:.0f} s")
else:
    print("⏳ Ex B6.27 : l'expérience attend tes trois fonctions (cellule précédente).")

### Ex B6.28 — Fiche modèle avec sections explicabilité et équité : indicateurs par groupe, limites, usage prévu 🛠️ ★★ ⏱️ 25 min
**Objectif :** documenter un modèle de décision dans une fiche modèle complète, chiffres par groupe et limites compris.  
**Prérequis :** Ex B6.24, B6.27 · ch. 23 (Ex 23.28) · MP3 (fiche modèle) · fiche §206.11 (« La fiche modèle ») · **Parcours :** R, C

Au ch. 23, tu as rédigé la fiche modèle d'un MLP ; au MP3, celle du modèle de revenu. Rédige ici celle du modèle de référence de la partie (`HGB_C`), avec les deux rubriques que la fiche ajoute pour un modèle de décision, **explicabilité** et **équité** (206.11). La cellule suivante affiche ce qu'il faut savoir (`FACTS_28` : le modèle, les données, l'accuracy et la ROC-AUC de validation, l'effectif, le TPR et le FPR de chaque sexe et de chaque tranche d'âge, l'importance par permutation sur la validation) et les onze sections (`REQUIRED_28`) ; ≈ 5 s.

**Contrat** : `CARD_28` est une chaîne Markdown d'au moins 300 mots, avec les onze sections de `REQUIRED_28` comme titres `## …`, dans cet ordre (les accents et la casse des titres ne comptent pas, et un titre peut continuer : « ## Usage prévu et hors périmètre ») ; « Usage prévu » dit aussi les usages **hors** périmètre ; « Métriques » donne le seuil (0,5) ; l'accuracy de validation y est, à 4 décimales ; les sections « Analyses quantitatives » ou « Équité » ont, pour chaque groupe de `FACTS_28["groupes"]`, une ligne (de tableau, par exemple) avec son nom, son effectif, son TPR et son FPR à 2 décimales ; « Explicabilité » nomme une méthode et les données sur lesquelles elle est calculée (la validation, ou le fond d'une valeur SHAP) ; « Équité » nomme le critère retenu (parité démographique, égalité des chances ou equalized odds) et parle des intervalles. Une f-string à triples guillemets (`f` devant trois guillemets) t'évite de recopier les nombres. Sers-toi de ce que tu as mesuré en B6.24, B6.26 et B6.27.

La vérification relit ta fiche, dit ce qui manque, puis l'écrit dans `fiche_modele_adult_b6.md`, à côté de ton notebook (dans `mon_travail`) : elle a sa place dans ton portfolio, avec la data card d'*Adult* (`data/cards/adult.md`).

Dans tes notes : qu'est-ce qu'une vérification de structure ne peut pas juger dans ta fiche ? Qui la lirait (un métier, un auditeur, une autorité), et que chercherait chacun d'abord ?

In [ ]:
# What the fiche must say about the reference model of part C (FACTS_28), and the eleven sections it must have
import re
import tempfile
import unicodedata
from pathlib import Path

rates_sex_28, gaps_sex_28 = sk_audit(YA_VAL, PRED_VAL, SEX_VAL)
rates_age_28, gaps_age_28 = sk_audit(YA_VAL, PRED_VAL, AGE_VAL)
FACTS_28 = {
    "modèle": "HistGradientBoostingClassifier(random_state=0) dans le pipeline du MP3 (encodage ordinal), seuil 0,5",
    "données": f"Adult (recensement américain de 1994, UCI / OpenML, licence CC BY 4.0) : {len(XA_TRAIN)} personnes "
               f"d'entraînement, {len(XA_VAL)} de validation",
    "accuracy (validation)": round(float(np.mean(PRED_VAL == YA_VAL)), 4),
    "ROC-AUC (validation)": round(float(roc_auc_score(YA_VAL, P_VAL)), 4),
    "groupes": {k: {"n": r["n"], "TPR": round(float(r["tpr"]), 2), "FPR": round(float(r["fpr"]), 2)}
                for k, r in {**rates_sex_28, **rates_age_28}.items()},
}
perm_28 = permutation_importance(HGB_C, XA_VAL, YA_VAL, n_repeats=3, random_state=0, scoring="accuracy")
PERM_28 = pd.Series(perm_28.importances_mean, index=XA_VAL.columns).sort_values(ascending=False)
FACTS_28["permutation (validation, baisse d'accuracy, 3 répétitions)"] = {k: round(float(v), 4) for k, v in PERM_28.head(5).items()}
REQUIRED_28 = ["Détails du modèle", "Usage prévu", "Facteurs", "Métriques", "Données d'évaluation",
               "Données d'entraînement", "Analyses quantitatives", "Explicabilité", "Équité", "Considérations éthiques",
               "Mises en garde et recommandations"]


def _plain_28(text):
    """Lower case, without accents nor apostrophes of any kind (to compare section titles)."""
    text = unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9 ]", " ", text.lower()).split()


def sections_28(card):
    """{normalised title: text} of the sections `## ...` of a Markdown text, in order."""
    out, current = {}, None
    for line in str(card).splitlines():
        if line.startswith("## "):
            current = " ".join(_plain_28(line[3:]))
            out[current] = ""
        elif current is not None:
            out[current] += line + "\n"
    return out


def check_card_28(card):
    """The list of what is missing in the fiche (empty: everything is there)."""
    if not isinstance(card, str):
        return ["CARD_28 doit être une chaîne de caractères (du Markdown)."]
    problems, found = [], sections_28(card)
    titles = list(found)
    wanted = []
    for title in REQUIRED_28:                       # a title may go on: "## Usage prévu (et hors périmètre)"
        words = " ".join(_plain_28(title))
        wanted.append(next((t for t in titles if t == words or t.startswith(words + " ")), words))
    missing = [t for t, w in zip(REQUIRED_28, wanted) if w not in found]
    if missing:
        problems.append(f"il manque les sections (titres « ## … ») : {', '.join(missing)}.")
    else:
        order = [titles.index(w) for w in wanted]
        if order != sorted(order):
            problems.append("les onze sections doivent venir dans l'ordre de REQUIRED_28.")
        text = {t: found[w] for t, w in zip(REQUIRED_28, wanted)}
        if "hors" not in _plain_28(text["Usage prévu"]):
            problems.append("« Usage prévu » doit aussi dire les usages hors périmètre (le mot « hors »).")
        if not re.search(r"0[.,]5\b", text["Métriques"]):
            problems.append("« Métriques » doit donner le seuil de décision (0,5).")
        acc = FACTS_28["accuracy (validation)"]
        if not re.search(rf"0[.,]{str(acc)[2:]}", card):
            problems.append(f"la fiche doit donner l'accuracy de validation à 4 décimales ({str(acc).replace('.', ',')}).")
        table = text["Analyses quantitatives"] + text["Équité"]
        for group, facts in FACTS_28["groupes"].items():
            lines = [line for line in table.splitlines() if group in line]
            numbers = {f"{facts[k]:.2f}" for k in ("TPR", "FPR")}
            if not any(str(facts["n"]) in line and all(re.search(v.replace(".", "[.,]"), line) for v in numbers)
                       for line in lines):
                problems.append(f"aucune ligne des sections « Analyses quantitatives » et « Équité » ne donne, pour le "
                                f"groupe {group}, son effectif ({facts['n']}), son TPR et son FPR (2 décimales).")
                break
        words = set(_plain_28(text["Équité"]))
        if not ({"equalized", "odds"} <= words or {"egalite", "chances"} <= words or {"parite", "demographique"} <= words):
            problems.append("« Équité » doit nommer le critère retenu (parité démographique, égalité des chances ou "
                            "equalized odds) et dire pourquoi.")
        if "intervalle" not in " ".join(_plain_28(text["Équité"])):
            problems.append("« Équité » doit parler des intervalles (bootstrap) de ses écarts.")
        if not set(_plain_28(text["Explicabilité"])) & {"shap", "shapley", "permutation", "dependance", "ice"}:
            problems.append("« Explicabilité » doit nommer une méthode (SHAP, permutation, dépendance partielle, ICE…).")
        if not {"fond", "validation"} & set(_plain_28(text["Explicabilité"])):
            problems.append("« Explicabilité » doit dire sur quelles données l'explication est calculée (la validation, "
                            "ou le fond d'une valeur SHAP).")
    if len(re.findall(r"\w+", card)) < 300:
        problems.append(f"la fiche fait {len(re.findall(r'\w+', card))} mots : il en faut au moins 300.")
    return problems


print("ce que la fiche doit dire (FACTS_28) :")
for key_28, value_28 in FACTS_28.items():
    print(f"  {key_28} : {value_28}")
print("sections, dans cet ordre (REQUIRED_28) :", " · ".join(REQUIRED_28))

In [ ]:
_g = FACTS_28["groupes"]
_perm = ", ".join(f"`{k}` {fr(v, 4)}" for k, v in PERM_28.head(4).items())
_rows = "\n".join(f"| {k} | {v['n']} | {v['TPR']:.2f} | {v['FPR']:.2f} |".replace(".", ",") for k, v in _g.items())
CARD_28 = f"""# Fiche modèle : revenu supérieur à 50 000 $ (Adult), gradient boosting audité (workbook, B6)

## Détails du modèle
Gradient boosting `HistGradientBoostingClassifier(random_state=0)` de scikit-learn 1.6, dans le pipeline sans fuite
du MP3 : catégories codées 0, 1, 2… (`OrdinalEncoder`, catégories rares regroupées), colonnes numériques telles
quelles, `fnlwgt` et `education` écartées. Entraîné le 3 novembre 2026 pour le chapitre B6 du workbook ; version 1 ;
code sous licence du dépôt. Seuil de décision : 0,5.

## Usage prévu
Exercice pédagogique : illustrer un audit d'équité et des corrections. Hors périmètre : toute décision sur une
personne réelle (crédit, embauche, logement, aide sociale), et toute population d'aujourd'hui : les données datent de
1994 et ne représentent que des adultes interrogés aux États-Unis.

## Facteurs
Le sexe (deux valeurs dans les données), la tranche d'âge (six tranches du MP3), et, à auditer ensuite, la catégorie
raciale et le pays d'origine, aux effectifs très inégaux. Les performances varient d'abord avec le taux de base de chaque
groupe (validation) : {fr(rates_sex_28["Female"]["positives"] / rates_sex_28["Female"]["n"], 3)} pour les femmes,
{fr(rates_sex_28["Male"]["positives"] / rates_sex_28["Male"]["n"], 3)} pour les hommes,
{fr(rates_age_28["<25"]["positives"] / rates_age_28["<25"]["n"], 3)} pour les moins de 25 ans
({rates_age_28["<25"]["positives"]} positifs sur {rates_age_28["<25"]["n"]}).

## Métriques
Accuracy au seuil 0,5 et ROC-AUC sur la validation ; par groupe : taux de sélection, TPR, FPR et precision ; écarts :
parité démographique (ΔDP) et equalized odds (ΔEO, le plus grand des écarts de TPR et de FPR), avec des intervalles
bootstrap à 95 % (tirages dans chaque groupe).

## Données d'évaluation
La partie validation de `wbx.adult()` : {len(XA_VAL)} personnes, stratifiée sur le label ; le test du MP3
({len(ADULT.X_test)} personnes) reste à part pour le défi B6.29.

## Données d'entraînement
{FACTS_28["données"]} ; taux de base 0,239 ; data card : `data/cards/adult.md` (provenance, valeurs manquantes,
biais connus de la collecte).

## Analyses quantitatives
Validation : accuracy {str(FACTS_28["accuracy (validation)"]).replace(".", ",")}, ROC-AUC
{str(FACTS_28["ROC-AUC (validation)"]).replace(".", ",")}.

| groupe | n | TPR | FPR |
|---|---|---|---|
{_rows}

## Explicabilité
Importance par permutation sur la validation (baisse d'accuracy, 3 répétitions) : {_perm}. Une valeur SHAP
par personne demanderait de dire son fond (la valeur de base et les signes en dépendent, 🔬 B6.16). Limites : des
features corrélées se partagent l'information (`relationship`, `marital-status` et `sex`), et une importance n'est pas
une cause.

## Équité
Critère retenu : l'equalized odds par sexe, parce qu'une erreur de classement fausse les statistiques publiées d'un
groupe dans les deux sens (faux positifs et faux négatifs). Au seuil 0,5, ΔEO par sexe vaut
{fr(gaps_sex_28["eo"], 3)} et ΔDP {fr(gaps_sex_28["dp"], 3)} ; les intervalles bootstrap (B6.24) excluent 0 pour
l'écart de TPR. Corrections essayées (B6.27, moitié d'évaluation de la validation) : retirer `sex` ne réduit pas les
écarts (les autres colonnes prédisent le sexe avec une ROC-AUC de 0,93) ; retirer aussi `relationship` et
`marital-status` coûte 2,5 points d'accuracy et réduit ΔDP, pas ΔEO ; des seuils par sexe choisis pour un même TPR
le dépassent chez les femmes sur d'autres personnes (175 positives par moitié de validation, c'est peu), et utilisent
l'attribut dans la décision (risque juridique, 206.10). Les écarts de sélection par âge suivent surtout les taux de
base, très différents d'une tranche à l'autre.

## Considérations éthiques
Le sexe et l'âge sont des attributs protégés ; les catégories du recensement de 1994 (sexe binaire, catégories
raciales) ne décrivent pas tout le monde. Les labels (revenu déclaré) portent eux-mêmes les inégalités de l'époque :
un modèle qui les reproduit fidèlement n'est pas « neutre ».

## Mises en garde et recommandations
Ne pas déployer. Réauditer après toute correction, sur tous les groupes, avec les effectifs et les intervalles ; les
moins de 25 ans n'ont qu'une poignée de positifs, et leurs taux sont très incertains. Refaire l'audit sur des données
récentes (ACS, `folktables`) avant toute conclusion sur aujourd'hui.
"""


if CARD_28 is ... or len(re.findall(r"\w+", str(CARD_28))) < 10:
    print("⏳ Ex B6.28 : pas encore fait (rédige CARD_28).")
else:
    problems_28 = check_card_28(CARD_28)
    for problem_28 in problems_28:
        print(f"❌ Ex B6.28 : {problem_28}")
    if not problems_28:
        print("✅ Ex B6.28 : les onze sections, les chiffres par groupe, le critère et ses limites sont là.")
        card_path_28 = (Path("fiche_modele_adult_b6.md") if "mon_travail" in str(Path.cwd())   # next to your notebook
                        else Path(tempfile.mkdtemp(prefix="wb_b6_28_")) / "fiche_modele_adult_b6.md")
        card_path_28.write_text(CARD_28, encoding="utf-8")
        print(f"   écrite dans {card_path_28.name}" + (", à côté de ton notebook : garde-la pour ton portfolio."
                                                        if "mon_travail" in str(Path.cwd()) else "."))

### Ex B6.29 — Défi : écart d'equalized odds sous 0,07 sur Adult, en perdant moins de 2 points d'accuracy 🏆 ★★★★ ⏱️ 90 min
**Objectif :** concevoir une méthode qui rapproche les taux d'erreur des deux sexes sur des personnes jamais vues, pour un coût en accuracy borné.  
**Prérequis :** Ex B6.24, B6.27 · tout le chapitre · fiche §206.10 · **Parcours :** C

**Le défi** : écris `method_29(X_train, y_train, X_val, y_val)`, qui ajuste **un modèle neuf** sur ces seules données (des DataFrames des 14 colonnes d'*Adult*, `sex` compris, et des tableaux de labels 0/1) et rend une fonction `decide(X)`, qui rend les décisions 0/1 des personnes du DataFrame `X` (un tableau d'entiers de forme `(len(X),)`), de façon déterministe. La mesure, sur le **test du MP3** (9 769 personnes que ta méthode ne voit jamais) : un ΔEO par sexe sous `GOAL_EO_29` (0,07), et une accuracy inférieure de moins de 2 points (`MAX_LOSS_29`) à celle du gradient boosting de référence **sans correction** (seuil 0,5) sur le même test ; **et de même** sur un découpage de contrôle (`wbx.adult(seed=CONTROL_SEED_29)`, dont le test contient d'autres personnes), pour qu'une méthode réglée sur le test du MP3 ne passe pas par chance.

**Le sexe dans la décision.** Dans ce défi, `decide` **peut** utiliser `sex` : des seuils par sexe sont permis. C'est un choix d'exercice : en entreprise, utiliser l'attribut sensible dans la décision peut être une discrimination directe, même pour corriger (206.10) ; dis dans tes notes ce que tu ferais s'il était interdit.

**Les règles**, contrôlées par `evaluate_29(method_29)` :
- **à froid et sans fuite** : pendant que ta méthode s'ajuste et décide, les variables du notebook qui contiennent des données (un DataFrame, une Series ou un tableau d'au moins 1 000 éléments, un découpage de `wbx`) ou un modèle scikit-learn déjà ajusté, directement ou dans une liste, un tuple ou un dictionnaire, sont masquées, et les chargeurs de `wbx` et de `wb.datasets` neutralisés ; `method_29` ne garde rien de tel dans ses arguments par défaut ; elle ne reçoit que l'entraînement et la validation (leurs lignes numérotées à partir de 0 : l'index d'origine d'*Adult* est retiré), jamais les labels du test ;
- **chaque personne est décidée seule** : sur les 200 premières personnes du test, `decide` appliquée à une ligne à la fois, avant et après le test entier, doit redonner les décisions du test entier, et le test présenté dans un autre ordre, les mêmes décisions dans cet ordre (aucun réglage sur les features du test) ;
- **contrôle des labels mélangés** : ta méthode, ajustée sur des labels d'entraînement et de validation mélangés, ne doit plus rien savoir (une balanced accuracy d'au plus `MAX_BALANCED_29` = 0,56 sur le test ; le hasard : 0,5) ; et aucune accuracy ne dépasse de plus de 2 points celle de la référence sans correction (une correction d'équité ne fait pas mieux qu'elle d'autant) ;
- **révélé une fois** : le premier résultat sur le test du MP3 est gardé pour la session, et c'est lui qui compte. Règle d'honneur : ne règle pas ta méthode sur le test, que tu pourrais lire dans `ADULT` ou dans le fichier d'*Adult* sur le disque.

La cellule suivante donne `evaluate_29`, `GOAL_EO_29`, `MAX_LOSS_29`, `CONTROL_SEED_29`, `MAX_BALANCED_29` et `REFERENCE_29` (le meilleur et le pire ΔEO, et la plus petite et la plus grande baisse d'accuracy de la solution de référence sur 11 découpages) ; elle ajuste les deux gradient boostings sans correction (≈ 10 s). L'évaluation ajuste ta méthode trois fois (≈ 20 s pour la référence). Tu peux reprendre tes fonctions du chapitre, ou faire autrement : d'autres seuils, une repondération, une contrainte pendant l'entraînement, un autre modèle.

Dans tes notes : ta méthode, ce que tu as essayé, ce qui a aidé et ce qui n'a rien changé, tes deux ΔEO et tes deux baisses d'accuracy. Pourquoi viser sur la validation un écart plus petit que l'objectif ? De combien le ΔEO de ta méthode varie-t-il d'un découpage à l'autre, comparé à l'objectif ? Et si `sex` était interdit dans la décision ? La réussite est affichée ; elle n'est pas enregistrée : c'est une règle d'honneur.

In [ ]:
# The challenge: ΔEO by sex below GOAL_EO_29 on the test of the MP3, losing less than MAX_LOSS_29 of accuracy against
# the reference gradient boosting (no correction, threshold 0.5) on the same test; the same on a control split
GOAL_EO_29 = 0.07
MAX_LOSS_29 = 0.02
REFERENCE_29 = {'eo': [0.0119, 0.058], 'loss': [0.0053, 0.0093]}       # the reference solution on 11 splits and seeds: ΔEO (best, worst), drop of accuracy (smallest, largest)
CONTROL_SEED_29 = 1   # the control split: wbx.adult(seed=CONTROL_SEED_29), whose test rows are other people
MAX_BALANCED_29 = 0.56        # balanced accuracy allowed to a method trained on shuffled labels (chance: 0.5)


def _make_evaluate_29():
    """evaluate_29, with the test parts and their labels kept here only (the global variables that hold data, a fitted
    model or a split are replaced by a decoy while your method runs and decides)."""
    import re
    from sklearn.base import BaseEstimator
    from sklearn.utils.validation import check_is_fitted
    from wb import datasets as wbdata

    splits = {"test du MP3": wbx.adult(), "contrôle": wbx.adult(seed=CONTROL_SEED_29)}
    goal, max_loss, max_balanced, reference = GOAL_EO_29, MAX_LOSS_29, MAX_BALANCED_29, dict(REFERENCE_29)
    audit, balanced_accuracy = sk_audit, balanced_accuracy_score     # kept here: your method cannot replace them

    def change(loss):
        """'0,70 point de moins' or '0,20 point de plus': an accuracy above the reference is not a loss."""
        return f"{fr(100 * abs(loss), 2)} point{'s' if abs(loss) >= 0.02 else ''} de {'moins' if loss >= 0 else 'plus'}"

    def baseline(split):
        model = adult_pipeline().fit(split.X_train, split.y_train.to_numpy())
        return float(np.mean((model.predict_proba(split.X_test)[:, 1] >= 0.5) == split.y_test.to_numpy()))

    base = {name: baseline(split) for name, split in splits.items()}

    class Hidden:
        """What a hidden global variable holds during the run: any use of it says why."""

        def __init__(self, name):
            self._name = name

        def _refuse(self, *args, **kwargs):
            raise RuntimeError(f"pendant le défi, {self._name} est masqué : ta méthode ne reçoit que X_train, y_train, "
                               "X_val et y_val, et decide que les features à décider")

        __getattr__ = __getitem__ = __call__ = __iter__ = __len__ = _refuse

    ipython_names = re.compile(r"^(_+|_i+|_i?\d+|_[ido]h|In|Out|exit|quit|get_ipython)$")

    def fitted(v):
        try:
            check_is_fitted(v)
            return True
        except Exception:  # noqa: BLE001 - not fitted, or not an estimator scikit-learn knows
            return False

    def hideable(v, level=0):
        """True for what could hold Adult (a table, an array, a split) or a fitted model, up to three levels down."""
        if isinstance(v, wbx.TabularSplit):
            return True
        if isinstance(v, (pd.DataFrame, pd.Series)):
            return len(v) >= 1000
        if isinstance(v, np.ndarray):
            return v.size >= 1000
        if isinstance(v, BaseEstimator):
            return fitted(v)
        if isinstance(v, dict):
            return level < 3 and any(hideable(x, level + 1) for x in list(v.values())[:10_000])
        if isinstance(v, (list, tuple, set, frozenset)):
            return level < 3 and any(hideable(x, level + 1) for x in list(v)[:10_000])
        return False

    def captured(method):
        kept = list(getattr(method, "__defaults__", None) or ()) + list((getattr(method, "__kwdefaults__", None)
                                                                          or {}).values())
        kept += [cell.cell_contents for cell in (getattr(method, "__closure__", None) or ())]
        return any(hideable(v) for v in kept)

    def refused(*args, **kwargs):
        raise RuntimeError("pendant le défi, ta méthode ne reçoit que ses arguments (ni chargeur de données du workbook, "
                           "ni lecture du disque par wb)")

    def run(method, split, shuffle_seed=None):
        """Fit the method cold on the train and validation parts (labels shuffled for the control), then decide the
        test part, whole and row by row (on 200 rows), every hideable global variable hidden."""
        X_tr, y_tr = split.X_train.reset_index(drop=True), split.y_train.to_numpy().copy()   # row numbers 0, 1, 2…:
        X_va, y_va = split.X_val.reset_index(drop=True), split.y_val.to_numpy().copy()       # the index of Adult says
        if shuffle_seed is not None:                                                         # nothing of the test rows
            rng = np.random.default_rng(shuffle_seed)
            y_tr, y_va = rng.permutation(y_tr), rng.permutation(y_va)
        X_te = split.X_test.reset_index(drop=True)
        order = np.random.default_rng(290).permutation(len(X_te))                # the test in another order
        g = globals()
        hidden = {n: Hidden(n) for n, v in list(g.items()) if not ipython_names.match(n) and hideable(v)}
        hidden.update({n: Hidden(n) for n in ("evaluate_29", "_make_evaluate_29") if n in g})   # and this evaluation
        originals = {n: g[n] for n in hidden}
        saved = {name: getattr(wbx, name) for name in ("adult", "california", "penguins")}
        saved_data = {name: getattr(wbdata, name) for name in ("load_adult",)}
        cache = dict(wbx._TABULAR)
        try:
            g.update(hidden)
            wbx._TABULAR.clear()
            for name in saved:
                setattr(wbx, name, refused)
            for name in saved_data:
                setattr(wbdata, name, refused)
            start = time.perf_counter()
            decide = method(X_tr, y_tr, X_va, y_va)
            seconds = time.perf_counter() - start
            if not callable(decide):
                return None, f"method_29 doit rendre une fonction decide(X) ; reçu {type(decide).__name__}."
            first_rows = np.concatenate([np.asarray(decide(X_te.iloc[[i]].copy())).reshape(-1) for i in range(200)])
            whole = np.asarray(decide(X_te.copy()))
            shuffled_order = np.asarray(decide(X_te.iloc[order].reset_index(drop=True)))
            alone = np.concatenate([np.asarray(decide(X_te.iloc[[i]].copy())).reshape(-1) for i in range(200)])
        finally:
            g.update(originals)
            wbx._TABULAR.update(cache)
            for name, f in saved.items():
                setattr(wbx, name, f)
            for name, f in saved_data.items():
                setattr(wbdata, name, f)
        if whole.shape != (len(X_te),) or not np.isin(whole, (0, 1)).all():
            return None, (f"decide(X) doit rendre un tableau de 0 et de 1 de forme ({len(X_te)},) ; reçu la forme "
                          f"{whole.shape}.")
        if not (np.array_equal(alone.astype(int), whole[:200].astype(int))
                and np.array_equal(first_rows.astype(int), whole[:200].astype(int))):
            return None, ("decide ne décide pas chaque personne seule : sur les 200 premières, décider une ligne à la fois "
                          "(avant et après le test entier) ne donne pas les mêmes décisions que tout le test d'un coup "
                          "(une décision ne doit dépendre que de la personne).")
        if shuffled_order.shape != whole.shape or not np.array_equal(shuffled_order.astype(int), whole[order].astype(int)):
            return None, ("decide ne rend pas les mêmes décisions quand le test arrive dans un autre ordre : une décision "
                          "ne doit dépendre que de la personne, pas de sa place dans le tableau.")
        return {"decisions": whole.astype(int), "seconds": seconds}, ""

    def measure(split, decisions, name):
        y = split.y_test.to_numpy()
        rates, gaps = audit(y, decisions, split.X_test["sex"].to_numpy())
        acc = float(np.mean(decisions == y))
        return {"split": name, "accuracy": acc, "loss": base[name] - acc, "eo": gaps["eo"], "tpr_gap": gaps["tpr"],
                "fpr_gap": gaps["fpr"], "balanced": float(balanced_accuracy(y, decisions))}

    def evaluate_29(method):
        """method(X_train, y_train, X_val, y_val) -> decide(X): the rules of the statement, then the two measures."""
        if captured(method):
            print("❌ Ex B6.29 : method_29 garde, dans ses arguments par défaut ou sa fermeture, des données d'Adult ou un "
                  "modèle déjà entraîné : elle ne doit utiliser que ses arguments.")
            return None
        results = {}
        for name, split in splits.items():
            out, why = run(method, split)
            if out is None:
                print(f"❌ Ex B6.29 ({name}) : {why}")
                return None
            results[name] = measure(split, out["decisions"], name) | {"seconds": out["seconds"]}
        out, why = run(method, splits["test du MP3"], shuffle_seed=29)
        if out is None:
            print(f"❌ Ex B6.29 (labels mélangés) : {why}")
            return None
        shuffled = measure(splits["test du MP3"], out["decisions"], "test du MP3")
        first = globals().get("FIRST_29")
        if first is None:
            globals()["FIRST_29"] = first = dict(results["test du MP3"])
        for name, r in results.items():
            print(f"{name} : accuracy {fr(r['accuracy'], 4)} (référence sans correction {fr(base[name], 4)}, "
                  f"{change(r['loss'])}) ; ΔEO par sexe {fr(r['eo'], 3)} "
                  f"(écart de TPR {fr(r['tpr_gap'], 3)}, de FPR {fr(r['fpr_gap'], 3)}) ; ajustement en {fr(r['seconds'], 1)} s.")
        if first["eo"] != results["test du MP3"]["eo"] or first["accuracy"] != results["test du MP3"]["accuracy"]:
            print(f"ℹ️ le test du MP3 a déjà été révélé dans cette session : ΔEO {fr(first['eo'], 3)}, accuracy "
                  f"{fr(first['accuracy'], 4)} à la première révélation ; c'est elle qui compte (règle d'honneur).")
        rules = [(shuffled["balanced"] <= max_balanced,
                  f"contrôle des labels mélangés : balanced accuracy {fr(shuffled['balanced'], 3)} (au plus "
                  f"{fr(max_balanced, 2)} ; le hasard : 0,5).",
                  f"entraînée sur des labels mélangés, ta méthode décide encore le test avec une balanced accuracy de "
                  f"{fr(shuffled['balanced'], 3)} (au plus {fr(max_balanced, 2)}) : apprend-elle d'une autre source "
                  "que les labels qu'elle reçoit ?"),
                 (all(r["accuracy"] <= base[n] + 0.02 for n, r in results.items()),
                  "aucune accuracy trop belle (au plus 2 points au-dessus de la référence sans correction).",
                  "une accuracy dépasse de plus de 2 points celle de la référence sans correction : une correction "
                  "d'équité ne fait pas mieux qu'elle d'autant ; une fuite ?")]
        for ok, success, failure in rules:
            verdict("B6.29", ok, success, failure)
        goals = {n: r["eo"] < goal and r["loss"] < max_loss for n, r in results.items()}
        ok = all(rule[0] for rule in rules) and all(goals.values())
        verdict("B6.29", ok,
                f"défi réussi : ΔEO {fr(results['test du MP3']['eo'], 3)} et {fr(results['contrôle']['eo'], 3)} (objectif "
                f"sous {fr(goal, 2)}) ; accuracy : {change(results['test du MP3']['loss'])} et "
                f"{change(results['contrôle']['loss'])} (perte permise : moins de {fr(100 * max_loss, 0)} points) ; référence : "
                f"ΔEO de {fr(reference['eo'][0], 3)} à {fr(reference['eo'][1], 3)}.",
                "pas encore : " + " ; ".join(
                    f"{n} : ΔEO {fr(r['eo'], 3)} (sous {fr(goal, 2)}), accuracy {change(r['loss'])} "
                    f"(perte permise : moins de {fr(100 * max_loss, 0)} points)" for n, r in results.items()) + ".")
        return {"results": results, "shuffled": shuffled, "ok": ok}

    return evaluate_29


evaluate_29 = _make_evaluate_29()
print(f"Objectif : ΔEO par sexe sous {fr(GOAL_EO_29, 2)} et moins de {fr(100 * MAX_LOSS_29, 0)} points d'accuracy perdus, sur "
      f"le test du MP3 et sur le découpage de contrôle (seed={CONTROL_SEED_29}) ; référence : ΔEO de "
      f"{fr(REFERENCE_29['eo'][0], 3)} à {fr(REFERENCE_29['eo'][1], 3)}, accuracy en baisse de "
      f"{fr(100 * REFERENCE_29['loss'][0], 2)} à {fr(100 * REFERENCE_29['loss'][1], 2)} point.")

In [ ]:
TARGET_EO_29 = 0.02     # the gap aimed at on the validation part: below the goal, for a margin on new people


def method_29(X_train, y_train, X_val, y_val):
    """Your method: fit a NEW model on these data only (X_*: DataFrames of the 14 columns of Adult, sex included;
    y_*: int arrays of 0/1) and return decide(X), which returns the 0/1 decisions (an int array (len(X),)) of the
    people of the DataFrame X, each person decided alone."""
    model = adult_pipeline().fit(X_train, y_train)                 # the reference recipe, without correction
    scores, sex = model.predict_proba(X_val)[:, 1], X_val["sex"].to_numpy()
    grid = np.round(np.arange(0.05, 0.951, 0.01), 2)
    curves = {}
    for group in ("Female", "Male"):                               # TPR, FPR and right decisions at each threshold
        pos, neg = scores[(sex == group) & (y_val == 1)], scores[(sex == group) & (y_val == 0)]
        tpr = (pos[None, :] >= grid[:, None]).mean(axis=1)
        fpr = (neg[None, :] >= grid[:, None]).mean(axis=1)
        curves[group] = (tpr, fpr, tpr * len(pos) + (1 - fpr) * len(neg))
    (tpr_f, fpr_f, right_f), (tpr_m, fpr_m, right_m) = curves["Female"], curves["Male"]
    eo = np.maximum(np.abs(tpr_f[:, None] - tpr_m[None, :]), np.abs(fpr_f[:, None] - fpr_m[None, :]))
    acc = (right_f[:, None] + right_m[None, :]) / len(y_val)
    if (eo <= TARGET_EO_29).any():                                 # the best accuracy among the fair enough pairs
        i, j = np.unravel_index(np.argmax(np.where(eo <= TARGET_EO_29, acc, -1.0)), acc.shape)
    else:                                                          # else the smallest gap
        i, j = np.unravel_index(np.argmin(eo), eo.shape)
    thresholds = {"Female": float(grid[i]), "Male": float(grid[j])}

    def decide(X):
        t = np.array([thresholds.get(value, 0.5) for value in X["sex"]])
        return (model.predict_proba(X)[:, 1] >= t).astype(int)

    return decide


result_29 = None
with checked("B6.29"):
    if globals().get("method_29") is None:
        raise StopCheck
    start_29 = time.perf_counter()
    result_29 = evaluate_29(method_29)
    print(f"ℹ️ {time.perf_counter() - start_29:.0f} s")

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu calculer à la main les valeurs de Shapley d'un petit jeu, avec un fond, et dire ce que le fond change ?
2. Sais-tu calculer des gradients intégrés et une carte Grad-CAM à la main, et vérifier la complétude ?
3. Sais-tu auditer un classifieur par groupe (taux de sélection, TPR, FPR, precision, écarts) et expliquer pourquoi tous les critères ne peuvent pas tenir ensemble ?
4. Sais-tu dire ce qu'un scoring de crédit implique au regard du RGPD et de l'AI Act ?

**Pour aller plus loin** : l'article de Lundberg et Lee et le livre en ligne de C. Molnar, cités dans la fiche. La suite : le bonus B7, le MLOps.